[Markdown Cell]

# Phase 6 / 07 — B distillation, batched for Kaggle T4 ×2
Attach your trained own Phase 6 A/C/D teacher output directly. No separate teacher-export
notebook is required. Keep the P6_00 prepared data and own B foundation/tokenizer inputs.
This notebook generates all Train teacher targets internally, then trains/resumes B.

Both T4s train with **NCCL DDP and mp.spawn**. FP16, fused AdamW, gradient checkpointing,
pinned transfers and the complete dataset remain enabled. The added executor measures
safe source/target microbatches before DDP starts; target windows are batched **within
the same document**, preserving the saved document/update plan and weighted token loss.

Teacher and student greedy inference pack source chunks across documents and decode
multiple documents with native KV/GRU caches per GPU. A saved beam size above one retains
the original beam search. Neither source/reference tokens nor generation limits are cut.
Only the supervising process prints generation tqdm, aggregating both GPUs. Training
tqdm prints on rank 0, with epoch, step, loss, LR and AMP skips.

All original hashed runtime sources and model state keys remain unchanged. Legacy
optimizer/scaler/scheduler/RNG/cursor states are loaded. Larger batches can change future
dropout draws and floating-point rounding; future updates are not promised bit-identical.
Keep this SAME optimized notebook/settings for subsequent continuation. The shared ten-hour
launch cap and every-500-update/epoch checkpoint cadence remain. Actual speed depends on
document lengths and the attached teacher; no fixed speedup or 100% GPU occupancy is claimed.

[Markdown Cell]

## 1. Environment

In [ ]:
# [Code Cell]
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

requirements = {'tokenizers': 'tokenizers>=0.21,<1', 'numpy': 'numpy>=1.26,<3',
    'ijson': 'ijson>=3.3,<4', 'huggingface_hub': 'huggingface_hub>=0.30,<2',
    'tqdm': 'tqdm>=4.66,<5', 'rouge_score': 'rouge-score>=0.1.2,<1'}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Dependencies ready. Kaggle PyTorch is preserved; use a fresh kernel before Run All.')


[Markdown Cell]

## 2. Paths and settings

In [ ]:
# [Code Cell]
INPUT_ROOT = '/kaggle/input'
EXPERIMENT_ID = 'P6_07_B_Sequence_Distillation'
FAMILY, METHOD = 'b', 'kd'
RESUME_RUN_DIR = None
OUTPUT_ROOT = '/kaggle/working/cs781_phase6/' + EXPERIMENT_ID
PREPARED_DIR = None
PARENT_RUN_DIR = None  # exact foundation export root, if discovery is ambiguous
TOKENIZER_PATH = None
METADATA_PATH = None
TEACHER_RUN_DIR = None  # manually set the completed own Phase 6 A/C/D OUTPUT root
REQUIRE_EXISTING_CHECKPOINT = False
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

# auto: continue existing window cursor; use document grouping for a NEW run.
TRAINING_LAYOUT = 'auto'  # auto/windows/documents
WARM_START_RUN_DIR = None  # optional SAME Phase 6 model/method, actual task weights
GRADIENT_ACCUMULATION_STEPS = 1  # new document runs: 2 documents/update across T4 x2
# For warm-start document grouping, ALSO give EXPERIMENT_ID a new suffix before
# OUTPUT_ROOT is constructed, e.g. P6_04_C_Houlsby_Adapters_FastDocs.

# Example ancestor: '/kaggle/input/notebooks/YOUR_USER/YOUR_P6_A_NOTEBOOK'
# Select the job.json root or an ancestor containing one teacher run; not the .ipynb.
TEACHER_CHECKPOINT = 'auto'  # auto / best / latest; fixed once generation starts
AUTO_TEACHER_MAX_MINUTES = None  # None shares the ten-hour launch cap; a number allows a resumable teacher pause

# Execution-only caps; on-GPU probes choose safe TRAINING microbatches.
# These do NOT change global document batch, epochs, data or decoding length.
T4_SOURCE_BATCH_CAP = 32
T4_TARGET_WINDOW_BATCH_CAP = 4
T4_GENERATION_BATCH = 4  # multiple summaries per GPU; OOM automatically backs off
T4_INFERENCE_CHUNKS = 64  # complete source chunks across documents; OOM backs off
T4_TUNING_SECONDS = 30  # short independent probe on each GPU before DDP hooks


In [ ]:
# [Code Cell]
MAX_EPOCHS = 4
TARGET_TOTAL_EPOCHS = None  # absolute E+1 to extend; do not change MAX_EPOCHS on resume
TRAINING_MINUTES = 600
CHUNK_TOKENS = 256
CHUNK_MICRO_BATCH = 4
MEMORY_SLOTS = 128
TARGET_TOKENS = 256
TARGET_CONTEXT = 128
BACKBONE_LR = 5e-05
MEMORY_LR = 0.0003
GENERATION_MAX_TOKENS = 512
GENERATION_MIN_TOKENS = 32
BEAM_SIZE = 1
MODE = 'train'


[Markdown Cell]

### What is trained
All source tokens are encoded in chunks. Pooled segment features receive learned document positions
and positive salience weights; a bounded memory compresses the full document. This is a learned
representation bottleneck, not lossless reconstruction, and it may lose fine details. A/C/D condition
their original causal blocks on this memory prefix; B uses its original cross-attention decoder.

For every gold summary, all target tokens are covered by exhaustive teacher-forced windows:
$$\mathcal L_{CE}=-\sum_t \log p_\theta(y_t\mid y_{<t},\operatorname{Memory}(x)).$$
Only summary targets receive loss. Source/prefix/context tokens are not prediction targets.
For B distillation, supervised gold windows and own-teacher summary windows use a weighted token
mixture, with teacher weight $\lambda=0.25$ by default. Teacher text is retokenized using B's own
tokenizer, so different teacher/student vocabularies are supported. Gold targets are never removed.

LoRMA learns $(I+\alpha BA/r)W_0$, whereas LoRA learns $W_0+\alpha BA/r$. B learns encoder/decoder soft
prompts. C's Houlsby-style adapters are at both attention and FFN outputs; D's Pfeiffer-style adapters
are only at FFN outputs. These are five distinct techniques; decoding/epoch variations do not add
techniques. The shared source-memory bridge is learned in each experiment.

FP32 master parameters, FP16 AMP, bounded recomputation and native KV/GRU caches target T4 efficiency.
The full training split is used. Epochs, optimizer updates, loss and LR print only on rank 0.
Checkpoints include model, Adam, scaler, scheduler, exact next-step cursor and both workers' RNG states.
They are committed every 500 updates, each epoch and at a timed stop. Resume requires full saved
outputs, not an `.ipynb`, observations archive or prediction ZIP. The own foundation and total model
must be between 50M and 300M unique parameters; the much smaller active PEFT count is also reported.

[Markdown Cell]

### Direct teacher input — skip notebook 06
Attach the COMPLETE saved output of a trained Phase 6 A LoRMA/LoRA, C Houlsby, or D Pfeiffer
summarizer. Set TEACHER_RUN_DIR to the directory containing job.json, checkpoints/, tokenizer/
and parent_source/. It can be a Kaggle Notebook Output or an uploaded dataset with that structure.
The .ipynb alone has no model weights. Phase 1 / Phase 5 task outputs cannot replace a trained
summarization teacher. The existing B student foundation remains a separate input.

All Train documents get real model-generated pseudo summaries. Gold summaries are never renamed
as teacher predictions. Tune/confirmation/test references are not used as distillation targets.
Set TEACHER_CHECKPOINT='auto' to reuse the teacher's frozen screened rule when present, otherwise
its saved inference choice. Explicit 'best' and 'latest' are also supported for fresh target generation.

[Markdown Cell]

### Complete importable runtime
Small source cells below materialize ordinary Python modules. No code is fetched from a remote notebook.

In [ ]:
# [Code Cell]
module_parts = {}


[Markdown Cell]

`__init__.py` — Phase 6 version identity.

In [ ]:
# [Code Cell]
module_parts.setdefault('__init__.py', []).append(r'''"""Own-foundation hierarchical abstractive legal summarization."""
VERSION = 'cs781-phase6-summ-v1'
''')


[Markdown Cell]

`adaptation.py` — Multiplicative LoRMA, additive LoRA, dual soft prompts, two-placement Houlsby and FFN-only Pfeiffer adapters.

In [ ]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''"""Five explicit PEFT methods; KD updates the own smaller B foundation."""
import math
import torch
from torch import nn
from torch.nn import functional as F


class LowRank(nn.Module):
    """Additive LoRA or identity-inflated multiplicative LoRMA on a frozen Linear."""
    def __init__(self, base, rank, alpha, multiplicative=False):
        super().__init__()
        self.base, self.multiplicative = base, multiplicative
        base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank, base.out_features if multiplicative else base.in_features))
        self.b = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))

    def forward(self, value):
        hidden = F.linear(value, self.base.weight, None)
        residual = F.linear(F.linear(hidden if self.multiplicative else value, self.a), self.b)
        result = hidden + residual * self.scale
        return result if self.base.bias is None else result + self.base.bias


class Adapter(nn.Module):
    """Zero-initialized bottleneck residual after a frozen sublayer projection."""
    def __init__(self, base, width):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, width)
        self.up = nn.Linear(width, base.out_features)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, value):
        hidden = self.base(value)
        return hidden + self.up(F.gelu(self.down(hidden)))


''')


In [ ]:
# [Code Cell]
module_parts.setdefault('adaptation.py', []).append(r'''def adapt(backbone, cfg):
    backbone.requires_grad_(False)
    if cfg.method == 'kd':
        backbone.requires_grad_(True)
        return ['all own B encoder/decoder weights']
    if cfg.method == 'prompt':
        width = backbone.shared_embedding.embedding_dim
        for name in ('soft_encoder_prompt', 'soft_decoder_prompt'):
            setattr(backbone, name, nn.Parameter(torch.randn(cfg.prompt_tokens, width) * .02))
        return ['soft_encoder_prompt', 'soft_decoder_prompt']
    changed = []
    for name, module in list(backbone.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        replacement = None
        if cfg.method in ('lora', 'lorma') and name.startswith('layers.'):
            replacement = LowRank(module, cfg.rank, cfg.alpha, cfg.method == 'lorma')
        # Houlsby: bottlenecks at BOTH attention and FFN sublayer outputs.
        if cfg.method == 'houlsby' and name.startswith('stages.') and name.endswith(('attention.out', 'ffn.down')):
            replacement = Adapter(module, cfg.adapter_width)
        # Pfeiffer: one bottleneck at the FFN output, no attention-output adapter.
        if cfg.method == 'pfeiffer' and name.startswith('layers.') and name.endswith('ffn.down'):
            replacement = Adapter(module, cfg.adapter_width)
        if replacement is not None:
            parent, _, child = name.rpartition('.')
            setattr(backbone.get_submodule(parent), child, replacement)
            changed.append(name)
    if not changed:
        raise ValueError('No intended adaptation modules were found; inspect actual foundation source')
    return changed
''')


[Markdown Cell]

`compare.py` — Actual file-level scoring, required-six accounting and a separately copied overall winner.

In [ ]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''"""Score actual generated files, choose on tune and verify the required six methods."""
from pathlib import Path
import shutil
import zipfile
from .evaluation import evaluate, read_predictions
from .io_utils import atomic_json, read_json, records, sha256_file
from .preparation import verify_prepared
from .parents import source_hashes
from .state import Checkpoints

REQUIRED = {('a', 'lorma'), ('a', 'lora'), ('b', 'prompt'), ('c', 'houlsby'), ('d', 'pfeiffer'), ('b', 'kd')}


def compare(input_root, prepared_dir, output, task_roots=None, bert=True, device='cuda'):
    prepared, output = Path(prepared_dir), Path(output)
    manifest = verify_prepared(prepared)
    roots = [Path(path) for path in task_roots] if task_roots else sorted({path.parent for path in Path(input_root).rglob('job.json')
        if read_json(path).get('version') == 'cs781-phase6-summ-v1'})
    reference = {split: {row['id']: row['summary'] for row in records(prepared / (split + '.jsonl'))}
                 for split in ('tune', 'confirm')}
    test_ids = [row['id'] for row in records(prepared / 'test.jsonl')]
    choices = []
    rejected = []
    for root in roots:
        try:
            job = read_json(root / 'job.json')
            if job.get('version') != 'cs781-phase6-summ-v1' or job['prepared_identity'] != manifest['identity']:
                raise ValueError('Not a matching Phase 6 run')
            if read_json(root / 'training_summary.json')['status'] != 'complete':
                raise ValueError('Training incomplete')
            if source_hashes(root / 'code/summ') != job['runtime_source_sha256']:
                raise ValueError('Saved summarization source differs from its learned runtime identity')
            if source_hashes(root / 'parent_source' / ('model_' + job['config']['family'])) != job['parent_contract']['source_sha256']:
                raise ValueError('Saved own foundation source changed')
            manager = Checkpoints(root, job['fingerprint'], 0)
            available_checkpoints = {receipt['sha256'] for _, receipt in manager.candidates()}
            reports = [root / name for name in ('final_report.json', 'screen_report.json') if (root / name).is_file()]
            for file in reports:
                receipt = read_json(file)
                if receipt.get('status') != 'submission_verified':
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''                    continue
                # Receipts may contain an old /kaggle/working path. Resolve by the
                # task-relative generation variant instead of trusting old mounts.
                directory = root / 'generation' / Path(receipt['generation_directory']).name
                generation = read_json(directory / 'generation_identity.json')
                if generation['checkpoint_sha256'] not in available_checkpoints or receipt['checkpoint_sha256'] != generation['checkpoint_sha256']:
                    raise ValueError('Generation refers to an older/different checkpoint; re-export the selected learned snapshot')
                target = root / ('screened' if file.name == 'screen_report.json' else '') / 'prediction.json'
                archive = target.parent / Path(receipt['zip_path']).name
                if sha256_file(target) != receipt['prediction_sha256'] or sha256_file(archive) != receipt['zip_sha256']:
                    raise ValueError('Actual submission payload does not match its receipt')
                with zipfile.ZipFile(archive) as zipped:
                    if zipped.testzip() is not None or zipped.namelist() != ['prediction.json'] or zipped.read('prediction.json') != target.read_bytes():
                        raise ValueError('Invalid submission ZIP bytes/root layout')
                if set(read_predictions(target)) != set(test_ids):
                    raise ValueError('Submission is missing original test IDs')
                tune_file = directory / 'tune_prediction.json'
                tune = evaluate(read_predictions(tune_file), reference['tune'], bert, device)
                choices.append({'experiment_id': job['config']['experiment_id'], 'family': job['config']['family'],
                    'method': job['config']['method'], 'root': str(root), 'tune': tune, 'receipt': str(file),
                    'zip': str(archive), 'prediction_sha256': receipt['prediction_sha256'],
                    'generation_directory': str(directory)})
        except (ValueError, KeyError, OSError) as error:
            rejected.append({'root': str(root), 'reason': str(error)})
    if not choices:
        raise FileNotFoundError('Attach complete finished Phase 6 outputs, not .ipynb files or observations alone. Rejections: ' + str(rejected))
    metric = 'bertscore_f1' if bert else 'rougeL_f1'
    selected = {}
    for candidate in choices:
        key = (candidate['family'], candidate['method'])
        if key not in selected or candidate['tune'][metric] > selected[key]['tune'][metric]:
            selected[key] = candidate
    output.mkdir(parents=True, exist_ok=True)
    result_rows = []
    for candidate in selected.values():
        confirm_file = Path(candidate['generation_directory']) / 'confirm_prediction.json'
        candidate['confirmation'] = evaluate(read_predictions(confirm_file), reference['confirm'], bert, device)
        destination = output / Path(candidate['zip']).name
        shutil.copyfile(candidate['zip'], destination)
        candidate['selected_zip'] = str(destination)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('compare.py', []).append(r'''        result_rows.append(candidate)
    winner = max(result_rows, key=lambda row: row['tune'][metric])
    best = output / 'P6_Best_Validated_Submission.zip'
    shutil.copyfile(winner['selected_zip'], best)
    duplicates = {}
    for row in result_rows:
        duplicates.setdefault(row['prediction_sha256'], []).append(row['experiment_id'])
    summary = {'status': 'verified', 'selected_on': 'tune ' + metric, 'winner': winner,
               'best_zip': str(best), 'methods': result_rows, 'required_methods_missing': [list(x) for x in sorted(REQUIRED - set(selected))],
               'course_six_experiments_complete': REQUIRED.issubset(selected), 'rejected_inputs': rejected,
               'duplicate_prediction_groups': [items for items in duplicates.values() if len(items) > 1],
               'private_test_BERTScore': None, 'confirmation_used_for_selection': False,
               'BERTScore_used_only_for_evaluation': True, 'test_labels_used': False}
    atomic_json(output / 'final_comparison.json', summary)
    return summary
''')


[Markdown Cell]

`config.py` — Typed immutable learning choices; transport paths, generation and explicit epoch extensions remain separate.

In [ ]:
# [Code Cell]
module_parts.setdefault('config.py', []).append(r'''"""Explicit, immutable learning choices and portable transport paths."""
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class SummConfig:
    experiment_id: str = 'P6_01_A_LoRMA'
    family: str = 'a'
    method: str = 'lorma'
    run_dir: str = '/kaggle/working/cs781_phase6/P6_01_A_LoRMA'
    input_root: str = '/kaggle/input'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    teacher_run_dir: str | None = None
    parent_checkpoint: str = 'best'
    max_epochs: int = 4
    target_total_epochs: int | None = None
    minimum_epochs: int = 2
    patience: int = 2
    seed: int = 781
    world_size: int = 2
    accumulation_steps: int = 4
    max_run_seconds: int = 36000
    min_free_bytes: int = 1 << 30
    chunk_tokens: int = 256
    chunk_micro_batch: int = 4
    tokens_per_chunk: int = 4
    memory_slots: int = 128
    target_tokens: int = 256
    target_context: int = 128
    rank: int = 16
    alpha: float = 16.
    adapter_width: int = 32
    prompt_tokens: int = 32
    backbone_lr: float = 0.0002
    memory_lr: float = 0.0003
    weight_decay: float = 0.01
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('config.py', []).append(r'''    warmup_fraction: float = 0.05
    min_lr_ratio: float = 0.1
    max_grad_norm: float = 1.
    label_smoothing: float = 0.05
    kd_weight: float = 0.25
    checkpoint_encoder: bool = True
    checkpoint_every_steps: int = 500
    log_every_steps: int = 10
    max_consecutive_overflows: int = 8
    export_train_summaries: bool = False
    inference_checkpoint: str = 'best'
    generation_max_tokens: int = 512
    generation_min_tokens: int = 32
    repetition_penalty: float = 1.1
    no_repeat_ngram: int = 3
    beam_size: int = 1
    length_penalty: float = 0.8

    def validate(self):
        combinations = {'a': ('lorma', 'lora'), 'b': ('prompt', 'kd'),
                        'c': ('houlsby',), 'd': ('pfeiffer',)}
        if self.family not in combinations or self.method not in combinations[self.family]:
            raise ValueError('Unsupported architecture/adaptation combination')
        if self.world_size != 2 or min(self.accumulation_steps, self.max_epochs, self.max_run_seconds) < 1:
            raise ValueError('Use two NCCL workers and positive training budgets')
        if min(self.chunk_tokens, self.chunk_micro_batch, self.target_tokens, self.memory_slots) < 1:
            raise ValueError('Chunk, memory and target windows must be positive')
        if self.target_context < 0 or min(self.tokens_per_chunk, self.rank, self.adapter_width, self.prompt_tokens) < 1:
            raise ValueError('Invalid context, pooling or adaptation dimensions')
        if not 0 <= self.kd_weight < 1 or not 0 <= self.label_smoothing < 1:
            raise ValueError('Invalid loss mixture/smoothing')
        if not 0 <= self.generation_min_tokens < self.generation_max_tokens:
            raise ValueError('Invalid generation length controls')
        if self.beam_size not in (1, 2, 4) or self.checkpoint_every_steps != 500:
            raise ValueError('Beam size must be 1/2/4; required checkpoint cadence is 500 updates')
        if self.inference_checkpoint not in ('best', 'latest'):
            raise ValueError('Select best or latest for inference')
        if self.target_total_epochs is not None and self.target_total_epochs < 1:
            raise ValueError('TARGET_TOTAL_EPOCHS is an absolute positive epoch target')

''')


In [ ]:
# [Code Cell]
module_parts.setdefault('config.py', []).append(r'''    def choices(self):
        value = asdict(self)
        transport = ('run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path',
                     'metadata_path', 'resume_run_dir', 'teacher_run_dir', 'max_run_seconds',
                     'min_free_bytes', 'target_total_epochs', 'export_train_summaries',
                     'generation_max_tokens', 'generation_min_tokens', 'repetition_penalty',
                     'no_repeat_ngram', 'beam_size', 'length_penalty', 'inference_checkpoint')
        return {key: item for key, item in value.items() if key not in transport}
''')


[Markdown Cell]

`data.py` — Lossless memory maps, exhaustive target windows, context masks and deterministic rank plans.

In [ ]:
# [Code Cell]
module_parts.setdefault('data.py', []).append(r'''"""Lossless uint16 token caches and exhaustive summary-window supervision."""
import math
from pathlib import Path
import numpy as np
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .io_utils import atomic_json, identity, read_json, records, sha256_file


def cache_data(prepared, tokenizer, cfg, teacher=None):
    root = Path(cfg.run_dir) / 'data'
    root.mkdir(exist_ok=True)
    signature = identity({'data': sha256_file(Path(prepared) / 'prepared_manifest.json'),
                          'tokenizer': sha256_file(tokenizer), 'tokenizers_version': __import__('tokenizers').__version__,
                          'teacher': teacher and teacher['sha256']})
    manifest_path = root / 'cache_manifest.json'
    if manifest_path.exists():
        saved = read_json(manifest_path)
        if saved['identity'] != signature:
            raise ValueError('Cached tokens belong to another dataset/tokenizer/teacher')
        for name, expected in saved['files'].items():
            if sha256_file(root / name) != expected:
                raise ValueError('Token cache corruption: ' + name)
        saved['directory'] = str(root)
        return saved
    codec = Tokenizer.from_file(str(tokenizer))
    if codec.get_vocab_size() != 32768:
        raise ValueError('Use the selected foundation tokenizer, not an unrelated model tokenizer')
    cases = {}
    files = {}
    for split in ('train', 'tune', 'confirm', 'test'):
        metadata = []
        with (root / (split + '_source.bin')).open('wb') as source, (root / (split + '_target.bin')).open('wb') as target:
            for row in tqdm(records(Path(prepared) / (split + '.jsonl')), desc='Tokenize ' + split, unit='documents'):
                document = codec.encode(row['document'], add_special_tokens=False).ids + [2]
                summary = codec.encode(row['summary'], add_special_tokens=False).ids + [2] if split != 'test' else []
                if not document or any(token >= 32768 for token in document + summary):
                    raise ValueError('Tokenizer IDs outside the own vocabulary')
                entry = {'id': row['id'], 'source_offset': source.tell() // 2, 'source_length': len(document),
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('data.py', []).append(r'''                         'target_offset': target.tell() // 2, 'target_length': len(summary)}
                np.asarray(document, np.uint16).tofile(source)
                np.asarray(summary, np.uint16).tofile(target)
                metadata.append(entry)
        cases[split] = metadata
        for kind in ('source', 'target'):
            name = split + '_' + kind + '.bin'
            files[name] = sha256_file(root / name)
    if teacher:
        expected = {row['id'] for row in cases['train']}
        targets = {row['id']: row['summary'] for row in records(teacher['path'])}
        if set(targets) != expected:
            raise ValueError('Teacher summaries must cover exactly every training ID, never tune/confirm/test')
        with (root / 'teacher_target.bin').open('wb') as stream:
            for row in cases['train']:
                value = codec.encode(targets[row['id']], add_special_tokens=False).ids + [2]
                row['teacher_offset'], row['teacher_length'] = stream.tell() // 2, len(value)
                np.asarray(value, np.uint16).tofile(stream)
        files['teacher_target.bin'] = sha256_file(root / 'teacher_target.bin')
    saved = {'version': 'summ-token-cache-v1', 'identity': signature, 'directory': str(root),
             'cases': cases, 'files': files, 'all_source_and_reference_tokens_retained': True}
    atomic_json(manifest_path, saved)
    return saved


class SummaryWindows:
    """Each source is whole; every gold/pseudo summary token is a target once."""
    def __init__(self, cache, cfg, split='train'):
        self.cache, self.cfg, self.split = cache, cfg, split
        directory = Path(cache['directory'])
        self.source = np.memmap(directory / (split + '_source.bin'), np.uint16, 'r')
        target_path = directory / (split + '_target.bin')
        self.target = np.memmap(target_path, np.uint16, 'r') if target_path.stat().st_size else None
        self.teacher = np.memmap(directory / 'teacher_target.bin', np.uint16, 'r') if cfg.method == 'kd' and split == 'train' else None
        self.specs = []
        for index, row in enumerate(cache['cases'][split]):
            for kind in (('gold', 'teacher') if self.teacher is not None else ('gold',)):
                length = row['teacher_length'] if kind == 'teacher' else row['target_length']
                for start in range(0, length, cfg.target_tokens):
                    self.specs.append((index, kind, start, min(start + cfg.target_tokens, length)))
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('data.py', []).append(r'''
    def __len__(self):
        return len(self.specs)

    def document(self, index):
        row = self.cache['cases'][self.split][index]
        start = row['source_offset']
        return np.asarray(self.source[start:start + row['source_length']], dtype=np.int64)

    def __getitem__(self, index):
        if index < 0:
            return {'source': np.asarray([1, 2]), 'decoder': np.asarray([1]),
                    'labels': np.asarray([-100]), 'weight': 0.}
        case, kind, start, stop = self.specs[index]
        row = self.cache['cases'][self.split][case]
        array = self.teacher if kind == 'teacher' else self.target
        offset = row['teacher_offset'] if kind == 'teacher' else row['target_offset']
        context = max(0, start - self.cfg.target_context)
        previous = np.asarray(array[offset + context:offset + stop - 1], np.int64)
        # BOS represents the boundary of a context window; earlier real tokens follow it.
        decoder = np.concatenate(([1], previous))
        labels = np.concatenate((np.full(start - context, -100), np.asarray(array[offset + start:offset + stop], np.int64)))
        weight = self.cfg.kd_weight if kind == 'teacher' else 1 - self.cfg.kd_weight if self.teacher is not None else 1.
        assert len(decoder) == len(labels)
        return {'source': self.document(case), 'decoder': decoder, 'labels': labels, 'weight': weight}

    def target_weight(self, index):
        if index < 0:
            return 0.
        _, kind, start, stop = self.specs[index]
        weight = self.cfg.kd_weight if kind == 'teacher' else 1 - self.cfg.kd_weight if self.teacher is not None else 1.
        return (stop - start) * weight


def collate_one(row, chunk_tokens, device=None):
    source = row['source']
    chunks = math.ceil(len(source) / chunk_tokens)
    ids = torch.zeros(chunks, chunk_tokens, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    for index, start in enumerate(range(0, len(source), chunk_tokens)):
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('data.py', []).append(r'''        part = torch.as_tensor(source[start:start + chunk_tokens].copy(), dtype=torch.long)
        ids[index, :len(part)] = part
        mask[index, :len(part)] = True
    result = {'source_ids': ids, 'source_mask': mask,
              'decoder_ids': torch.as_tensor(row['decoder'].copy(), dtype=torch.long)[None],
              'labels': torch.as_tensor(row['labels'].copy(), dtype=torch.long)[None]}
    return {key: value.to(device, non_blocking=True) for key, value in result.items()} if device else result


def epoch_order(size, seed, epoch, slots):
    values = np.random.default_rng(seed + epoch).permutation(size).tolist()
    values.extend([-1] * ((-len(values)) % slots))
    return np.asarray(values, dtype=np.int64).reshape(-1, slots)
''')


[Markdown Cell]

`evaluation.py` — Official-compatible ROUGE and evaluation-only BERTScore. Test reference labels are never read.

In [ ]:
# [Code Cell]
module_parts.setdefault('evaluation.py', []).append(r'''"""Official-compatible ROUGE and evaluation-only BERTScore; never solver features."""
from pathlib import Path
import json
from .io_utils import atomic_json, sha256_file, records


def read_predictions(path):
    value = json.loads(Path(path).read_text(encoding='utf-8'))
    if isinstance(value, dict):
        result = value
    elif isinstance(value, list):
        result = {}
        for row in value:
            if not isinstance(row, dict) or not isinstance(row.get('id'), str) or row['id'] in result:
                raise ValueError('Invalid or duplicate prediction ID')
            result[row['id']] = row['summary']
    else:
        raise ValueError('Prediction must be an ID-summary mapping or list')
    if not result or any(not isinstance(k, str) or not k.strip() or not isinstance(v, str) or not v.strip() for k, v in result.items()):
        raise ValueError('Every document requires a nonempty generated summary')
    return result


def lcs(a, b):
    if len(a) > len(b):
        a, b = b, a
    masks = {}
    for index, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << index)
    value = 0
    for token in b:
        x = masks.get(token, 0) | value
        value = x & ~(x - ((value << 1) | 1))
    return value.bit_count()


def evaluate(predictions, references, bert=False, device='cpu'):
    from rouge_score import rouge_scorer, tokenizers
    if set(predictions) != set(references):
        raise ValueError('Evaluation IDs do not match the reference partition')
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('evaluation.py', []).append(r'''    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    tokenizer = tokenizers.DefaultTokenizer(use_stemmer=True)
    r1, rl = [], []
    for identifier in references:
        hypothesis, reference = predictions[identifier].strip(), references[identifier].strip()
        r1.append(scorer.score(reference, hypothesis)['rouge1'].fmeasure)
        actual, expected = tokenizer.tokenize(hypothesis), tokenizer.tokenize(reference)
        denominator = len(actual) + len(expected)
        rl.append(2 * lcs(actual, expected) / denominator if denominator else 1.)
    result = {'documents': len(references), 'rouge1_f1': sum(r1) / len(r1), 'rougeL_f1': sum(rl) / len(rl),
              'bertscore_f1': None, 'BERTScore_used_only_for_evaluation': True}
    if bert:
        import bert_score
        _, _, f1, scorer_hash = bert_score.score([predictions[key].strip() for key in references],
            [references[key].strip() for key in references], lang='en', batch_size=8,
            verbose=False, device=device, return_hash=True)
        result.update(bertscore_f1=float(f1.mean()), bertscore_configuration_hash=scorer_hash)
    return result


def evaluate_file(prepared, prediction, split, output, bert=False, device='cpu'):
    if split not in ('tune', 'confirm'):
        raise ValueError('Only public held-out partitions can be scored locally')
    reference = {row['id']: row['summary'] for row in records(Path(prepared) / (split + '.jsonl'))}
    result = evaluate(read_predictions(prediction), reference, bert, device)
    result.update(split=split, prediction_sha256=sha256_file(prediction), private_test_score=None)
    atomic_json(output, result)
    return result
''')


[Markdown Cell]

`experiments.py` — Importable screening/comparison entrypoints; no notebook-local spawn workers.

In [ ]:
# [Code Cell]
module_parts.setdefault('experiments.py', []).append(r'''"""Importable decoding-screen and final-comparison command-line entrypoints."""
import argparse
from pathlib import Path
from .config import SummConfig
from .io_utils import read_json


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--action', required=True, choices=('screen', 'compare'))
    parser.add_argument('--plan')
    parser.add_argument('--catalog')
    parser.add_argument('--prepared')
    parser.add_argument('--input-root', default='/kaggle/input')
    parser.add_argument('--output')
    parser.add_argument('--variants')
    parser.add_argument('--no-bert', action='store_true')
    parser.add_argument('--best-only', action='store_true')
    args = parser.parse_args()
    if args.action == 'screen':
        from .jobs import prepare_job, hardware
        from .workflow import stage_completed, observations
        from .screen import screen
        plan = read_json(args.plan)
        cfg = SummConfig(**plan['config'])
        job = prepare_job(cfg, read_json(args.catalog))
        hardware()
        stage_completed(job)
        try:
            result = screen(Path(cfg.run_dir) / 'job.json', read_json(args.variants) if args.variants else None,
                            not args.no_bert, ('best',) if args.best_only else ('best', 'latest'))
        finally:
            observations(Path(cfg.run_dir))
    else:
        import torch
        from .compare import compare
        result = compare(args.input_root, args.prepared, args.output, bert=not args.no_bert,
                         device='cuda' if torch.cuda.is_available() else 'cpu')
    print(result, flush=True)

''')


In [ ]:
# [Code Cell]
module_parts.setdefault('experiments.py', []).append(r'''
if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`generation.py` — Native KV/GRU cache decoding, beam/length controls and reference-free repetition prevention.

In [ ]:
# [Code Cell]
module_parts.setdefault('generation.py', []).append(r'''"""Native cached decoding and reference-free repetition controls."""
import math
import torch


def penalize(logits, generated, minimum, repetition, ngram):
    scores = logits.float().clone()
    scores[0, [0, 1, 3, 4, 5]] = -torch.inf
    if len(generated) < minimum:
        scores[0, 2] = -torch.inf
    if generated and repetition != 1:
        indices = torch.tensor(sorted(set(generated)), device=scores.device)
        previous = scores[0, indices]
        scores[0, indices] = torch.where(previous < 0, previous * repetition, previous / repetition)
    if ngram > 0 and len(generated) >= ngram - 1:
        prefix = tuple(generated[-(ngram - 1):]) if ngram > 1 else ()
        blocked = {generated[i + ngram - 1] for i in range(len(generated) - ngram + 1)
                   if tuple(generated[i:i + ngram - 1]) == prefix}
        if blocked:
            scores[0, list(blocked)] = -torch.inf
    if not torch.isfinite(scores).any():
        # Relax a decoding restriction, never replace generation with reference text.
        scores[0, 2] = 0.
    return scores


@torch.inference_mode()
def generate(model, document, codec, cfg, device):
    from .data import collate_one
    batch = collate_one({'source': document, 'decoder': __import__('numpy').asarray([1]),
                         'labels': __import__('numpy').asarray([-100])}, cfg.chunk_tokens, device)
    model.eval()
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
        memory = model.encode_document(batch['source_ids'], batch['source_mask'])
        logits, state = model.prefill(memory, torch.tensor([[1]], device=device))
        capacity = model.base_cfg.max_decoder_tokens if cfg.family == 'b' else model.base_cfg.max_seq_tokens
        prefix = cfg.prompt_tokens if cfg.family == 'b' and cfg.method == 'prompt' else 0 if cfg.family == 'b' else memory.shape[1]
        context_budget = min(cfg.target_tokens + cfg.target_context, capacity - prefix - 1)
        limit = cfg.generation_max_tokens
        resets = 0
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('generation.py', []).append(r'''        if context_budget < 2:
            raise ValueError('No generation space remains after the conditioning prefix')
        beams = [([], 0., state, logits, False)]
        for step in range(limit):
            candidates = []
            for tokens, score, cache, current, ended in beams:
                if ended:
                    candidates.append((tokens, score, cache, current, ended))
                    continue
                controlled = penalize(current, tokens, min(cfg.generation_min_tokens, limit - 1), cfg.repetition_penalty, cfg.no_repeat_ngram)
                logp = torch.log_softmax(controlled, dim=-1)
                values, indices = torch.topk(logp[0], cfg.beam_size)
                for value, token in zip(values.tolist(), indices.tolist()):
                    candidates.append((tokens + [token], score + value, cache, None, token == 2))
            rank = lambda item: item[1] / max(1, len(item[0])) ** cfg.length_penalty
            beams = sorted(candidates, key=rank, reverse=True)[:cfg.beam_size]
            if all(item[4] for item in beams) or step + 1 == limit:
                break
            updated = []
            for tokens, score, cache, current, ended in beams:
                if current is None and not ended:
                    if cache['position'] - prefix >= context_budget:
                        retained_count = max(0, min(cfg.target_context, context_budget - 2))
                        retained = tokens[-retained_count:] if retained_count else []
                        ids = torch.tensor([[1, *retained]], device=device)
                        current, cache = model.prefill(memory, ids)
                        resets += 1
                    else:
                        current, cache = model.step(torch.tensor(tokens[-1], device=device), cache)
                updated.append((tokens, score, cache, current, ended))
            beams = updated
        best = max(beams, key=rank)
    text = codec.decode([token for token in best[0] if token != 2], skip_special_tokens=True).strip()
    if not text:
        raise ValueError('Model generated an empty summary. Inspect validation outputs; no canned summary is substituted')
    return text, {'generated_tokens': len(best[0]), 'ended_with_eos': best[4],
                  'hit_generation_limit': not best[4], 'memory_tokens': memory.shape[1],
                  'context_replays_across_beams': resets,
                  'source_tokens': len(document), 'all_source_tokens_encoded': True}
''')


[Markdown Cell]

`io_utils.py` — Atomic JSON, checksums, disk space and exclusive training ownership.

In [ ]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''
def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)

def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('io_utils.py', []).append(r'''    os.replace(temporary, destination)
''')


[Markdown Cell]

`jobs.py` — Early teacher checks, verified actual model inputs and transport-safe exact continuation.

In [ ]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''"""Checked foundation loading, early teacher validation and transport-safe resume."""
from dataclasses import asdict
from pathlib import Path
import shutil
import torch
from . import VERSION
from .config import SummConfig
from .io_utils import atomic_json, identity, read_json, sha256_file, records
from .parents import verify_parent, tokenizer_inputs, parent_class, source_hashes
from .preparation import prepared_input
from .data import cache_data, SummaryWindows, collate_one
from .state import Checkpoints


def hardware():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select Kaggle T4 x2; this training runtime needs two CUDA devices and NCCL')
    for rank in range(2):
        if 'T4' not in torch.cuda.get_device_name(rank):
            raise RuntimeError('Expected two T4 GPUs; found ' + torch.cuda.get_device_name(rank))
        print('GPU', rank, torch.cuda.get_device_name(rank), flush=True)


def resume_inputs(cfg):
    local = Path(cfg.run_dir) / 'job.json'
    paths = [local] if local.is_file() else [Path(cfg.resume_run_dir) / 'job.json'] if cfg.resume_run_dir else sorted(Path(cfg.input_root).rglob('job.json'))
    choices = []
    for file in paths:
        value = read_json(file)
        if value.get('version') != VERSION or value.get('config', {}).get('experiment_id') != cfg.experiment_id:
            if cfg.resume_run_dir:
                raise ValueError('RESUME_RUN_DIR must be this same Phase 6 method/experiment, not a foundation or another phase')
            continue
        if value['runtime_source_sha256'] != source_hashes(Path(__file__).parent) or source_hashes(file.parent / 'code/summ') != value['runtime_source_sha256']:
            raise ValueError('Exact recovery needs its original Phase 6 runtime. Do not modify saved source files')
        if SummConfig(**value['config']).choices() != cfg.choices():
            raise ValueError('Learning choices changed. Resume with the saved settings or use a new experiment ID')
        if value['torch'] != str(torch.__version__):
            raise ValueError('Exact continuation requires the saved PyTorch build ' + value['torch'])
        choices.append((file.parent, value))
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''    if len({item['fingerprint'] for _, item in choices}) > 1:
        raise ValueError('Different saved runs share this ID. Set RESUME_RUN_DIR explicitly')
    return choices


def teacher_input(cfg, prepared, data_identity, student_parameters=None):
    if cfg.method != 'kd':
        return None
    train_ids = [row['id'] for row in records(prepared / 'train.jsonl')]
    roots = [Path(cfg.teacher_run_dir)] if cfg.teacher_run_dir else [file.parent for file in Path(cfg.input_root).rglob('teacher_export_receipt.json')]
    # A validated saved target snapshot is sufficient for continuation without teacher GPU weights.
    roots = [Path(cfg.run_dir), *([Path(cfg.resume_run_dir)] if cfg.resume_run_dir else []), *roots]
    accepted, diagnostics = [], []
    for root in roots:
        receipt_file = root / 'teacher_export_receipt.json'
        if not receipt_file.exists():
            continue
        try:
            receipt = read_json(receipt_file)
            if receipt.get('version') != 'summ-teacher-v1' or receipt.get('status') != 'complete':
                raise ValueError('Teacher export is incomplete or from another phase')
            if receipt.get('teacher_is_own_phase6_model') is not True:
                raise ValueError('Teacher target export must come from a verified own Phase 6 model')
            if receipt['prepared_identity'] != data_identity or receipt.get('test_labels_used') is not False:
                raise ValueError('Teacher export data identity mismatch')
            file = root / 'train_summaries.jsonl'
            if sha256_file(file) != receipt['sha256']:
                raise ValueError('Teacher summary bytes changed')
            rows = list(records(file))
            if [row['id'] for row in rows] != train_ids or any(not isinstance(r.get('summary'), str) or not r['summary'].strip() for r in rows):
                raise ValueError('Teacher must generate exactly every training document, in official prepared order')
            if student_parameters is not None and receipt['teacher_parameters'] <= student_parameters:
                raise ValueError('Select a larger own Phase 6 A/C/D teacher for B distillation')
            accepted.append((root, receipt, file))
        except (OSError, KeyError, ValueError) as error:
            diagnostics.append({'root': str(root), 'reason': str(error)})
    if len({r['sha256'] for _, r, _ in accepted}) > 1:
        raise ValueError('Several teacher exports differ. Set TEACHER_RUN_DIR to one exact exported teacher folder')
    if not accepted:
        atomic_json(Path(cfg.run_dir) / 'teacher_input_report.json', {'rejected': diagnostics})
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''        raise FileNotFoundError('First run P6_06_Export_Teacher on a COMPLETED own Phase 6 A/C/D summarizer. '
            'Attach its complete output containing teacher_export_receipt.json and train_summaries.jsonl. '
            'A Phase 5 teacher, .ipynb, checkpoint alone or submission ZIP is not a Phase 6 teacher target export.')
    root, receipt, file = accepted[0]
    destination = Path(cfg.run_dir)
    for name in ('teacher_export_receipt.json', 'train_summaries.jsonl'):
        if (root / name).resolve() != (destination / name).resolve():
            shutil.copyfile(root / name, destination / name)
    return {'path': str(destination / 'train_summaries.jsonl'), 'sha256': receipt['sha256'],
            'teacher_checkpoint_sha256': receipt['teacher_checkpoint_sha256'], 'teacher_parameters': receipt['teacher_parameters']}


def prepare_job(cfg, catalog):
    cfg.validate()
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    prepared, manifest = prepared_input(cfg)
    # Fail on missing KD targets BEFORE foundation loading or tokenization.
    teacher = teacher_input(cfg, prepared, manifest['identity'])
    previous = resume_inputs(cfg)
    prior = previous[0] if previous else None
    source = root / 'parent_source' / ('model_' + cfg.family)
    source.mkdir(parents=True, exist_ok=True)
    token_dir = root / 'tokenizer'
    token_dir.mkdir(exist_ok=True)
    if prior:
        saved_root, old = prior
        parent, contract = old['parent'], old['parent_contract']
        for file in (saved_root / 'parent_source' / ('model_' + cfg.family)).glob('*.py'):
            destination = source / file.name
            if file.resolve() != destination.resolve():
                shutil.copyfile(file, destination)
        if source_hashes(source) != contract['source_sha256']:
            raise ValueError('Saved foundation source is incomplete or changed')
        for file in (saved_root / 'tokenizer').iterdir():
            if file.is_file() and file.resolve() != (token_dir / file.name).resolve():
                shutil.copyfile(file, token_dir / file.name)
        if sha256_file(token_dir / 'tokenizer.json') != parent['tokenizer_sha256']:
            raise ValueError('Saved tokenizer differs from the actual selected foundation')
    else:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''        contract = catalog[cfg.family]
        parent_root, parent = verify_parent(cfg, contract)
        contract = parent.get('native_contract', contract)
        tokenizer, metadata, hashes = tokenizer_inputs(cfg, contract)
        parent.update(hashes)
        origin = parent_root / parent['native_contract']['source_subdir'] if parent.get('native_contract') else parent_root / 'code' / ('model_' + cfg.family)
        for file in origin.glob('*.py'):
            shutil.copyfile(file, source / file.name)
        shutil.copyfile(tokenizer, token_dir / 'tokenizer.json')
        shutil.copyfile(metadata, token_dir / ('foundation_manifest.json' if parent.get('native_contract') else 'metadata.json'))
    Config, Model = parent_class(str(source), cfg.family, contract)
    with torch.device('meta'):
        probe = Model(Config(**parent['config']))
    count = sum(parameter.numel() for parameter in probe.parameters())
    assert 50_000_000 < count < 300_000_000
    del probe
    if teacher and teacher['teacher_parameters'] <= count:
        raise ValueError('B student is not smaller than the selected teacher; choose a larger own summarizer')
    cache = cache_data(prepared, token_dir / 'tokenizer.json', cfg, teacher)
    fingerprint = identity({'version': VERSION, 'choices': cfg.choices(), 'data': manifest['identity'],
        'foundation': parent['checkpoint_sha256'], 'tokenizer': parent['tokenizer_sha256'],
        'runtime': source_hashes(Path(__file__).parent), 'cache': cache['identity'], 'torch': str(torch.__version__),
        'teacher': teacher and {key: teacher[key] for key in teacher if key != 'path'}})
    if prior and old['fingerprint'] != fingerprint:
        raise ValueError('Recovery fingerprint changed; do not mix weights/optimizers from another run')
    target = cfg.target_total_epochs or (old['target_epochs'] if prior else cfg.max_epochs)
    job = {'version': VERSION, 'config': asdict(cfg), 'prepared_dir': str(prepared), 'prepared_identity': manifest['identity'],
        'parent': parent, 'parent_contract': contract, 'parent_source_dir': str(source), 'parent_parameter_count': count,
        'tokenizer_path': str(token_dir / 'tokenizer.json'), 'cache': cache, 'fingerprint': fingerprint,
        'runtime_source_sha256': source_hashes(Path(__file__).parent), 'torch': str(torch.__version__),
        'resume_roots': [str(path) for path, _ in previous], 'teacher': teacher, 'target_epochs': target}
    manager = Checkpoints(root, fingerprint, cfg.min_free_bytes, job['resume_roots'])
    latest = manager.latest()
    if prior and not latest:
        raise FileNotFoundError('Saved job exists but full checkpoints are missing; attach the complete output')
    if latest and target < latest[1]['epoch']:
        raise ValueError('Target epoch is below already completed epochs')
    job['resume_checkpoint'] = str(latest[0]) if latest else None
    job['extension_requested'] = bool(latest and cfg.target_total_epochs is not None and target > latest[1]['epoch'])
    atomic_json(root / 'job.json', job)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('jobs.py', []).append(r'''    atomic_json(root / 'task_config.json', asdict(cfg))
    return job


def sanity(job):
    from .model import build_model
    cfg = SummConfig(**job['config'])
    device = torch.device('cuda', 0)
    model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        payload = torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)
        model.load_state_dict(payload['model'], strict=True)
        del payload
    dataset = SummaryWindows(job['cache'], cfg)
    row = dataset[min(range(len(dataset)), key=lambda i: dataset.specs[i][3] - dataset.specs[i][2])]
    model.train()
    with torch.autocast('cuda', dtype=torch.float16):
        loss = model(**collate_one(row, cfg.chunk_tokens, device))
    if not torch.isfinite(loss):
        raise FloatingPointError('Full-size shape/loss sanity failed')
    loss.backward()
    if not any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in model.parameters() if parameter.requires_grad):
        raise FloatingPointError('No finite active gradients')
    atomic_json(Path(cfg.run_dir) / 'model_report.json', {**model.parameter_report, 'finite_loss_and_gradients': True})
    del model, loss, dataset
    torch.cuda.empty_cache()
''')


[Markdown Cell]

`model.py` — All-document salience-weighted memory and the original own-model conditional generation blocks.

In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''"""All-document memory -> original own-model conditional autoregressive decoding."""
from dataclasses import replace
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from .adaptation import adapt
from .parents import parent_class


class HierarchicalSummarizer(nn.Module):
    """source[P,T] -> memory[1,M,D]; summary[1,L] -> completion-only NLL.

    Every source chunk contributes pooled token features. Decoder-only families
    use these learned source embeddings as a causal prefix in their OWN blocks.
    B uses its pretrained encoder/decoder and original cross-attention. No new
    random language decoder, external model, or extractive fallback is substituted.
    """
    def __init__(self, backbone, cfg, base_cfg):
        super().__init__()
        self.backbone, self.cfg, self.base_cfg = backbone, cfg, base_cfg
        width = base_cfg.d_model
        self.memory_projection = nn.Linear(width, width)
        self.memory_norm = nn.LayerNorm(width)
        self.memory_positions = nn.Linear(3, width, bias=False)
        self.memory_salience = nn.Linear(width, 1)
        nn.init.zeros_(self.memory_salience.weight)
        nn.init.zeros_(self.memory_salience.bias)
        nn.init.eye_(self.memory_projection.weight)
        nn.init.zeros_(self.memory_projection.bias)
        self.memory_scale = nn.Parameter(torch.tensor(-2.))

    def embedding(self):
        return self.backbone.shared_embedding if self.cfg.family == 'b' else self.backbone.token_embedding

    def encode_chunk(self, ids, mask):
        if self.cfg.family == 'b':
            if self.cfg.method == 'prompt':
                prefix = self.backbone.soft_encoder_prompt[None].expand(ids.shape[0], -1, -1)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''                hidden = torch.cat((prefix, self.embedding()(ids)), dim=1)
                hidden = self.backbone.dropout(hidden + self.backbone.source_positions(torch.arange(hidden.shape[1], device=ids.device))[None])
                valid = torch.cat((torch.ones(ids.shape[0], prefix.shape[1], dtype=torch.bool, device=ids.device), mask), dim=1)
                for layer in self.backbone.encoder:
                    hidden = layer(hidden, valid[:, None, None, :])
                hidden = self.backbone.encoder_norm(hidden)[:, prefix.shape[1]:]
            else:
                hidden = self.backbone.encode(ids, mask)
        else:
            hidden = self.backbone.hidden_states(ids, mask)
        # Segment boundaries cover [0,length) exactly, including the final tail.
        output = []
        for row in range(ids.shape[0]):
            length = int(mask[row].sum().item())
            value = hidden[row, :length].float().T[None]
            output.append(F.adaptive_avg_pool1d(value, min(length, self.cfg.tokens_per_chunk))[0].T)
        return torch.cat(output)

    def encode_document(self, source_ids, source_mask):
        values = []
        chunks = source_ids.shape[0]
        for first in range(0, chunks, self.cfg.chunk_micro_batch):
            ids, mask = source_ids[first:first + self.cfg.chunk_micro_batch], source_mask[first:first + self.cfg.chunk_micro_batch]
            if self.training and self.cfg.checkpoint_encoder:
                value = checkpoint(self.encode_chunk, ids, mask, use_reentrant=False)
            else:
                value = self.encode_chunk(ids, mask)
            values.append(value)
        memory = torch.cat(values)
        position = torch.linspace(0, 1, memory.shape[0], device=memory.device)
        positions = torch.stack((position, 1 - position, torch.ones_like(position)), dim=-1)
        memory = self.memory_norm(self.memory_projection(memory) + self.memory_positions(positions))
        slots = min(len(memory), self.cfg.memory_slots)
        importance = F.softplus(self.memory_salience(memory)).clamp_min(1e-4)
        numerator = F.adaptive_avg_pool1d((memory * importance).T[None], slots)
        denominator = F.adaptive_avg_pool1d(importance.T[None], slots)
        memory = (numerator / denominator).transpose(1, 2)
        # Keep the prefix in the scale range of learned token embeddings initially.
        return memory * torch.sigmoid(self.memory_scale)

''')


In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''    def decoder_hidden(self, decoder_ids, memory):
        backbone = self.backbone
        if self.cfg.family == 'b':
            hidden = self.embedding()(decoder_ids)
            prefix = backbone.soft_decoder_prompt[None] if self.cfg.method == 'prompt' else None
            if prefix is not None:
                hidden = torch.cat((prefix, hidden), dim=1)
            hidden = backbone.dropout(hidden + backbone.target_positions(torch.arange(hidden.shape[1], device=hidden.device))[None])
            allowed = torch.ones(hidden.shape[1], hidden.shape[1], dtype=torch.bool, device=hidden.device).tril()[None, None]
            cross = torch.ones(1, 1, 1, memory.shape[1], dtype=torch.bool, device=hidden.device)
            for layer in backbone.decoder:
                hidden = checkpoint(layer, hidden, memory, allowed, cross, use_reentrant=False) if self.training and self.cfg.checkpoint_encoder else layer(hidden, memory, allowed, cross)
            hidden = backbone.decoder_norm(hidden)
            return hidden[:, -decoder_ids.shape[1]:]
        hidden = torch.cat((memory.to(self.embedding().weight.dtype), self.embedding()(decoder_ids)), dim=1)
        # None selects native causal attention, including D's sparse routes.
        modules = backbone.stages if self.cfg.family == 'c' else backbone.layers
        for layer in modules:
            hidden = checkpoint(layer, hidden, None, use_reentrant=False) if self.training and self.cfg.checkpoint_encoder else layer(hidden, None)
        return backbone.final_norm(hidden)[:, -decoder_ids.shape[1]:]

    def forward(self, source_ids, source_mask, decoder_ids, labels):
        memory = self.encode_document(source_ids, source_mask)
        hidden = self.decoder_hidden(decoder_ids, memory)
        loss = hidden.sum() * 0.
        weight = self.embedding().weight
        for start in range(0, labels.shape[1], 64):
            actual = labels[:, start:start + 64]
            value = hidden[:, start:start + 64]
            def vocabulary_loss(x, target):
                return F.cross_entropy(F.linear(x.float(), weight.float()).flatten(0, 1), target.flatten(),
                                       reduction='sum', ignore_index=-100, label_smoothing=self.cfg.label_smoothing if self.training else 0.)
            loss = loss + (checkpoint(vocabulary_loss, value, actual, use_reentrant=False)
                           if self.training and self.cfg.checkpoint_encoder else vocabulary_loss(value, actual))
        return loss

    def prefill(self, memory, ids):
        """Unpadded prefix -> next-token logits plus exact native KV/GRU caches."""
        if self.cfg.family == 'b':
            cross = [layer.cross_attention.project_kv(memory) for layer in self.backbone.decoder]
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''            state = {'self': [None] * len(cross), 'cross': cross, 'position': 0}
            if self.cfg.method == 'prompt':
                for embedding in self.backbone.soft_decoder_prompt:
                    _, state = self._b_step(embedding[None, None], state)
            logits = None
            for token in ids[0]:
                logits, state = self._b_step(self.embedding()(token.reshape(1, 1)), state)
            return logits, state
        hidden = torch.cat((memory.to(self.embedding().weight.dtype), self.embedding()(ids)), dim=1)
        positions = torch.arange(hidden.shape[1], device=hidden.device)
        modules = self.backbone.stages if self.cfg.family == 'c' else self.backbone.layers
        values = []
        for layer in modules:
            if self.cfg.family == 'c':
                hidden, kv, recurrent = layer.cached(hidden, positions, None, None)
                values.append((kv, recurrent))
            else:
                hidden, kv = layer.cached(hidden, positions, None)
                values.append(kv)
        logits = F.linear(self.backbone.final_norm(hidden[:, -1]).float(), self.embedding().weight.float())
        return logits, {'values': values, 'position': hidden.shape[1]}

    def _b_step(self, embedded, state):
        position = state['position']
        hidden = embedded + self.backbone.target_positions(torch.tensor([position], device=embedded.device))[None]
        hidden = self.backbone.dropout(hidden)
        values = []
        for index, layer in enumerate(self.backbone.decoder):
            cross = state['cross'][index]
            mask = torch.ones(1, 1, 1, cross[0].shape[2], device=hidden.device, dtype=torch.bool)
            hidden, current = layer.step(hidden, state['self'][index], cross, mask)
            values.append(current)
        logits = F.linear(self.backbone.decoder_norm(hidden[:, -1]).float(), self.embedding().weight.float())
        return logits, {'self': values, 'cross': state['cross'], 'position': position + 1}

    def step(self, token, state):
        hidden = self.embedding()(token.reshape(1, 1))
        if self.cfg.family == 'b':
            return self._b_step(hidden, state)
        positions = torch.tensor([state['position']], device=hidden.device)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''        modules = self.backbone.stages if self.cfg.family == 'c' else self.backbone.layers
        values = []
        for index, layer in enumerate(modules):
            if self.cfg.family == 'c':
                kv, recurrent = state['values'][index]
                hidden, kv, recurrent = layer.cached(hidden, positions, kv, recurrent)
                values.append((kv, recurrent))
            else:
                hidden, kv = layer.cached(hidden, positions, state['values'][index])
                values.append(kv)
        logits = F.linear(self.backbone.final_norm(hidden[:, -1]).float(), self.embedding().weight.float())
        return logits, {'values': values, 'position': state['position'] + 1}


def build_model(job, cfg, device, load_parent=True):
    Config, Model = parent_class(job['parent_source_dir'], cfg.family, job['parent_contract'])
    changes = {'activation_checkpointing': False}
    if cfg.family == 'd':
        changes['sparse_attention_checkpointing'] = False
    base_cfg = replace(Config(**job['parent']['config']), **changes)
    with torch.device('meta'):
        backbone = Model(base_cfg)
    if load_parent:
        path = Path(job['parent']['checkpoint'])
        if (path.stat().st_size != job['parent']['checkpoint_bytes']
                or path.stat().st_mtime_ns != job['parent']['checkpoint_mtime_ns']):
            raise ValueError('Foundation checkpoint changed after verification')
        payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        weights = payload if job['parent'].get('format') == 'native_state_dict' else payload['model']
        backbone.load_state_dict(weights, strict=True, assign=True)
        del payload
    else:
        backbone._apply(lambda tensor: torch.empty_like(tensor, device=device) if tensor.is_meta else tensor.to(device))
    capacity = base_cfg.max_decoder_tokens if cfg.family == 'b' else base_cfg.max_seq_tokens
    source_capacity = base_cfg.max_source_tokens if cfg.family == 'b' else capacity
    prompt = cfg.prompt_tokens if cfg.method == 'prompt' else 0
    if cfg.chunk_tokens + prompt > source_capacity or cfg.target_tokens + cfg.target_context + max(prompt, cfg.memory_slots) + 1 > capacity:
        raise ValueError('Chunk/summary/prefix settings exceed this actual foundation context capacity')
    foundation = sum(p.numel() for p in backbone.parameters())
    assert 50_000_000 < foundation < 300_000_000
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('model.py', []).append(r'''    changed = adapt(backbone, cfg)
    model = HierarchicalSummarizer(backbone, cfg, base_cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 50_000_000 < total < 300_000_000 and trainable > 0
    model.parameter_report = {'foundation_unique_parameters': foundation, 'total_unique_parameters': total,
        'trainable_parameters': trainable, 'adapted_names': changed, 'own_decoder_reused': True,
        'all_document_chunks_encoded': True, 'external_pretrained_solver': False}
    return model
''')


[Markdown Cell]

`native_contracts.py` — Reviewed own-model source profiles; no external pretrained solver.

In [ ]:
# [Code Cell]
module_parts.setdefault('native_contracts.py', []).append(r'''"""Reviewed own-foundation portable structural fingerprints; Python 3.10-3.13."""
APPROVED_NATIVE_SOURCES = [{'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '586cd48bdc420d8fed5db7b074ceba38491b8f091f8f28d702ed74983c000b63',
                      'model.py': '836d92b81250ca5d0e1a84c3df8181c1129e437bfea75c4f8653d2ebd3bf545b'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'bce55e18e1388720b701933df11a420783e56fe826a6780cfedce2551f468910',
                      'model.py': '43d74a2894b9bf74eee67f044a344557c39776a270491cd03a80e0440c44f1f4'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_40h/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '6b6759c5c8375120cd82882ba4a2c48ef7ab4160895379c5ec29144f29551a52',
                      'model.py': '8af20d2199c02557c5f4c7b7f3ddf08a8dd3f0e3941bcb75bcda21761bc6e88e'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_AutoResume_v3/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'b83d3fee697e9eee3861de2d5d0e386a5b07a3397e3b80073aa58c1bdd1b0150',
                      'loss.py': 'd803e39be8f8998383b5627e711dda1a427f95cf358af7a0a9a58519cb38e92c',
                      'model.py': '9a90daad6e2cb3544d94ad14fbaa2df5453fd15250e7a7aa2ee45c51a4cc1025',
                      'model_b.py': '80f26dd23889221032a5444b37457aae0419536b7c498de328dbd675d9f9fb15',
                      'model_c.py': '0e1f50425fdcde72c90745888212fe305e898dcd844c246bab6a669d44316158',
                      'model_d.py': '8e768f3e73f78cf3e69347eae5972956fcfd10005e6849f564fef3be265f5b7f'},
  'package': 'foundation_upgrade',
  'profile': 'Foundation_Upgrade/Models_BCD/runtime/foundation_upgrade',
  'schema': 'bcd-upgraded-foundation-v1'}]
''')


[Markdown Cell]

`native_parents.py` — Actual upgraded 04 weights, tokenizer and source validation without fake legacy receipts.

In [ ]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''"""Load reviewed own foundation exports directly, without fabricating old receipts."""
import importlib
import importlib.util
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file, identity
from .native_contracts import APPROVED_NATIVE_SOURCES
from .source_identity import ALGORITHM, source_fingerprint


def sources(directory: Path) -> dict:
    return {p.name: sha256_file(p) for p in sorted(directory.glob('*.py'))}


def validate_source(directory: Path, family: str, contract: dict) -> None:
    """Check exact saved bytes AND a reviewed core implementation before import."""
    actual = sources(directory)
    if actual != contract['source_sha256']:
        raise ValueError('Native foundation source snapshot changed or is incomplete')
    expected_schema = 'a-upgraded-foundation-v1' if family == 'a' else 'bcd-upgraded-foundation-v1'
    if contract.get('schema') != expected_schema or contract.get('family') != family:
        raise ValueError('Native foundation family/schema mismatch')
    mismatches = []
    for approved in APPROVED_NATIVE_SOURCES:
        if approved['schema'] != expected_schema:
            continue
        try:
            ast_hashes = {name: source_fingerprint((directory / name).read_text(encoding='utf8'))
                          for name in approved['core_ast_sha256']}
            if ast_hashes == approved['core_ast_sha256']:
                return
            mismatches.append({'profile': approved.get('profile', approved['package']),
                               'different_modules': [n for n in ast_hashes if ast_hashes[n] != approved['core_ast_sha256'][n]],
                               'actual_core_identity': ast_hashes})
        except (OSError, SyntaxError, UnicodeError):
            continue
    raise ValueError('Unrecognized native model implementation after portable structural comparison. '
                     f'Algorithm={ALGORITHM}; differences={mismatches}. Preserve source files for review; do not bypass checks.')
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''

def native_class(directory: str, family: str, contract: dict):
    directory = Path(directory)
    validate_source(directory, family, contract)
    package = 'verified_native_' + family + '_' + identity(contract['source_sha256'])[:16]
    if package not in sys.modules:
        spec = importlib.util.spec_from_file_location(package, directory / '__init__.py',
                                                       submodule_search_locations=[str(directory)])
        module = importlib.util.module_from_spec(spec)
        sys.modules[package] = module
        spec.loader.exec_module(module)
    configuration = importlib.import_module(package + '.config')
    implementation = importlib.import_module(package + '.model')
    return configuration.TrainConfig, getattr(implementation, contract['class_name'])


def verify_export(root: Path, cfg) -> tuple[Path, dict, dict]:
    """Verify actual portable weights and tokenizer; no LM binaries or old metadata required."""
    manifest_path = root / 'foundation_manifest.json'
    manifest = read_json(manifest_path)
    schema = manifest.get('schema')
    family = 'a' if schema == 'a-upgraded-foundation-v1' else manifest.get('family')
    if family != cfg.family or schema not in ('a-upgraded-foundation-v1', 'bcd-upgraded-foundation-v1'):
        raise ValueError('Export belongs to another model family or unsupported schema')
    if (manifest.get('status') != 'exported' or manifest.get('training_run_status') != 'complete'
            or manifest.get('full_epoch_completed') is not True or manifest.get('completed_full_epochs', 0) < 1):
        raise ValueError('Finish 03 and run 04 with a complete full epoch before using this foundation')
    if manifest.get('selected_snapshot_has_new_updates') is not True or manifest.get('best_snapshot_step', 0) < 1:
        raise ValueError('This export selected the initial baseline; no upgraded weight updates were selected')
    weights, config, token = root / 'model_weights.pt', root / 'model_config.json', root / 'tokenizer.json'
    if sha256_file(weights) != manifest['model_weights_sha256'] or sha256_file(token) != manifest['tokenizer_sha256']:
        raise ValueError('Native export weights/tokenizer SHA-256 mismatch')
    if manifest.get('model_config_sha256') and sha256_file(config) != manifest['model_config_sha256']:
        raise ValueError('Native export model configuration SHA-256 mismatch')
    values = read_json(config)
    if family != 'a' and values.get('family') != family:
        raise ValueError('Model configuration belongs to another family')
    package = 'a_upgrade' if family == 'a' else 'foundation_upgrade'
    directory = root / 'code' / package
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''    code_hashes = sources(directory)
    if manifest.get('source_sha256') and manifest['source_sha256'] != code_hashes:
        raise ValueError('Source files differ from the native export manifest')
    contract = {'kind': 'native_foundation_export', 'family': family, 'schema': schema,
                'source_sha256': code_hashes, 'class_name': 'LegalCausalDecoder' if family == 'a' else 'build_model',
                'data_fingerprint': identity({'export_manifest': sha256_file(manifest_path),
                                              'tokenizer': manifest['tokenizer_sha256']}),
                'export_root': str(root), 'source_subdir': 'code/' + package}
    Config, Model = native_class(str(directory), family, contract)
    if values.get('vocab_size') != 32768:
        raise ValueError('Unexpected native vocabulary size')
    if family == 'a' and any(values.get(k) != v for k, v in
                             {'d_model': 768, 'n_heads': 12, 'n_layers': 12, 'd_ff': 2048,
                              'rms_epsilon': 1e-6, 'rope_base': 10000.0, 'max_seq_tokens': 2048}.items()):
        raise ValueError('Model A export differs from the reviewed 110M architecture')
    tokenizer = Tokenizer.from_file(str(token))
    if tokenizer.get_vocab_size() != values['vocab_size'] or [tokenizer.token_to_id(t) for t in
            ('<pad>', '<s>', '</s>', '<unk>', '<sep>', '<mask>')] != list(range(6)):
        raise ValueError('Tokenizer vocabulary/special IDs differ')
    if cfg.tokenizer_path and sha256_file(Path(cfg.tokenizer_path)) != manifest['tokenizer_sha256']:
        raise ValueError('TOKENIZER_PATH differs from the selected native foundation tokenizer')
    with torch.device('meta'):
        model = Model(Config(**values))
    state = torch.load(weights, map_location='cpu', weights_only=True, mmap=True)
    if not isinstance(state, dict) or not all(isinstance(t, torch.Tensor) for t in state.values()):
        raise ValueError('model_weights.pt must be the raw 04 state dictionary. Do not rename a 03 best.pt recovery payload.')
    for name, tensor in state.items():
        if tensor.is_floating_point() and (tensor.dtype != torch.float32 or
                not all(bool(torch.isfinite(chunk).all()) for chunk in tensor.reshape(-1).split(1_000_000))):
            raise ValueError('Nonfinite/non-FP32 native master weight: ' + name)
    model.load_state_dict(state, strict=True, assign=True)
    count = sum(p.numel() for p in model.parameters())
    if not 50_000_000 < count < 300_000_000 or count != manifest['unique_parameters']:
        raise ValueError('Native model parameter count differs from manifest/boundary')
    del model, state
    result = {'family': family, 'config': values, 'format': 'native_state_dict',
              'completed_epochs': manifest['completed_full_epochs'], 'planned_epochs': values['epochs'],
              'checkpoint': str(weights), 'checkpoint_sha256': manifest['model_weights_sha256'],
              'checkpoint_bytes': weights.stat().st_size, 'checkpoint_mtime_ns': weights.stat().st_mtime_ns,
              'checkpoint_epoch': manifest['best_snapshot_epoch'], 'checkpoint_step': manifest['best_snapshot_step'],
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('native_parents.py', []).append(r'''              'selection': '04 exported development-best snapshot', 'data_fingerprint': contract['data_fingerprint'],
              'source_sha256': code_hashes, 'tokenizer_sha256': manifest['tokenizer_sha256'],
              'metadata_sha256': sha256_file(manifest_path), 'tokenizer_receipt_name': 'foundation_manifest.json',
              'source_checkpoint_sha256': manifest['source_checkpoint_sha256'],
              'foundation_manifest_sha256': sha256_file(manifest_path), 'native_contract': contract,
              'original_fingerprint': None, 'original_torch': None, 'checkpoint_warnings': []}
    return root, result, contract


def select_export(cfg):
    """Prefer a verified native export, otherwise let the original verified loader run."""
    base = Path(cfg.parent_run_dir) if cfg.parent_run_dir else Path(cfg.input_root)
    paths = [base / 'foundation_manifest.json'] if (base / 'foundation_manifest.json').is_file() else sorted(base.rglob('foundation_manifest.json'))
    accepted, diagnostics, relevant = [], [], False
    for path in paths:
        value = read_json(path)
        family = 'a' if value.get('schema') == 'a-upgraded-foundation-v1' else value.get('family')
        if family != cfg.family:
            continue
        relevant = True
        row = {'run_dir': str(path.parent), 'format': 'native_foundation_export', 'accepted': False}
        try:
            root, parent, contract = verify_export(path.parent, cfg)
            row.update(accepted=True, checkpoint_sha256=parent['checkpoint_sha256'])
            accepted.append((root, parent))
        except (OSError, ValueError, KeyError, TypeError, RuntimeError, SyntaxError, AssertionError) as error:
            row['reason'] = f'{type(error).__name__}: {error}'
        diagnostics.append(row)
    if not relevant:
        return None
    atomic_json(Path(cfg.run_dir) / 'parent_input_report.json', {'native_candidates': diagnostics,
                'source_identity_algorithm': ALGORITHM, 'loader_python': sys.version.split()[0]})
    print({'native_parent_candidates': diagnostics}, flush=True)
    distinct = {(p['checkpoint_sha256'], identity(p['source_sha256']), p['tokenizer_sha256']): (r, p) for r, p in accepted}
    if len(distinct) != 1:
        raise ValueError(f'Found {len(distinct)} distinct valid upgraded {cfg.family.upper()} exports. '
                         'Attach the COMPLETE 04 output with actual model_weights.pt and code, '
                         'or set PARENT_RUN_DIR to its exact foundation folder. No fallback to old weights was attempted. '
                         'See parent_input_report.json.')
    return next(iter(distinct.values()))
''')


[Markdown Cell]

`parents.py` — Verified own original or upgraded Phase 1 foundations, not downstream task heads.

In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    from .native_parents import select_export
    native = select_export(cfg)
    if native is not None:
        return native
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json. '
                         'For upgraded models attach the complete 04 foundation export with model_weights.pt, '
                         'model_config.json, tokenizer.json, foundation_manifest.json and code; a 03 recovery run is not that export.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting a new LSI experiment.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    if contract.get('kind') == 'native_foundation_export':
        root = Path(contract['export_root'])
        manifest = read_json(root / 'foundation_manifest.json')
        token, receipt = root / 'tokenizer.json', root / 'foundation_manifest.json'
        if sha256_file(token) != manifest['tokenizer_sha256']:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''            raise ValueError('Native tokenizer changed after foundation verification')
        return token, receipt, {'tokenizer_sha256': manifest['tokenizer_sha256'],
                                'metadata_sha256': sha256_file(receipt),
                                'tokenizer_receipt_name': 'foundation_manifest.json'}
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    if contract.get('kind') == 'native_foundation_export':
        from .native_parents import native_class
        return native_class(directory, family, contract)
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('parents.py', []).append(r'''    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

`predict.py` — Two GPU inference replicas, per-document recovery, actual checkpoint reload and strict ZIP validation.

In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''"""Two independent GPU inference replicas, resumable per-document generations."""
from dataclasses import asdict, replace
import json
import os
from pathlib import Path
import shutil
import zipfile
import torch
import torch.multiprocessing as mp
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import SummConfig
from .data import SummaryWindows
from .evaluation import evaluate, read_predictions
from .generation import generate
from .io_utils import atomic_json, identity, read_json, records, sha256_file
from .model import build_model
from .state import Checkpoints, seed_everything


def load_best(job_path, device, selection='best'):
    job = read_json(job_path)
    cfg = SummConfig(**job['config'])
    manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    candidate = manager.best() if selection == 'best' else manager.latest()
    if candidate is None:
        raise FileNotFoundError('No validation-selected trained checkpoint exists yet')
    model = build_model(job, cfg, device, load_parent=False)
    payload = torch.load(candidate[0], map_location='cpu', weights_only=True, mmap=True)
    model.load_state_dict(payload['model'], strict=True)
    del payload
    model.eval()
    return model, job, cfg, candidate[1]


def generation_choices(cfg):
    return {key: asdict(cfg)[key] for key in ('generation_max_tokens', 'generation_min_tokens',
            'repetition_penalty', 'no_repeat_ngram', 'beam_size', 'length_penalty')}


''')


In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''def partial_rows(path):
    """Recover complete JSONL records; discard only an interrupted last write."""
    result, offset = {}, 0
    if not path.exists():
        return result
    with path.open('rb') as stream:
        while True:
            line = stream.readline()
            if not line:
                break
            if not line.endswith(b'\n'):
                break
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                if stream.read(1):
                    raise ValueError('Corrupt generation record before the file tail')
                break
            if row['id'] in result or not isinstance(row.get('summary'), str) or not row['summary'].strip():
                raise ValueError('Invalid or duplicate saved generated record')
            result[row['id']] = row
            offset = stream.tell()
    if path.stat().st_size != offset:
        with path.open('r+b') as stream:
            stream.truncate(offset)
    return result


def inference_worker(rank, job_path, splits, choices, variant, selection):
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    model, job, cfg, best = load_best(job_path, device, selection)
    cfg = replace(cfg, **choices, inference_checkpoint=selection)
    cfg.validate()
    model.cfg = cfg
    seed_everything(cfg.seed + rank)
    codec = Tokenizer.from_file(job['tokenizer_path'])
    output = Path(cfg.run_dir) / 'generation' / variant
    output.mkdir(parents=True, exist_ok=True)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''    for split in splits:
        dataset = SummaryWindows(job['cache'], cfg, split)
        rows = job['cache']['cases'][split]
        path = output / f'{split}_rank{rank}.jsonl'
        existing = partial_rows(path)
        expected = {row['id'] for index, row in enumerate(rows) if index % 2 == rank}
        if not set(existing).issubset(expected):
            raise ValueError('Saved generation shard belongs to a different split/rank')
        progress = tqdm(total=len(expected), initial=len(existing), disable=rank != 0,
                        desc=cfg.experiment_id + ' generate ' + split, unit='documents')
        with path.open('a', encoding='utf-8', newline='\n') as stream:
            for index in range(rank, len(rows), 2):
                identifier = rows[index]['id']
                if identifier in existing:
                    continue
                summary, details = generate(model, dataset.document(index), codec, cfg, device)
                stream.write(json.dumps({'id': identifier, 'summary': summary, 'details': details}, ensure_ascii=False) + '\n')
                stream.flush()
                os.fsync(stream.fileno())
                progress.update()
        progress.close()


def parallel_generate(job_path, splits, overrides=None):
    job = read_json(job_path)
    cfg = replace(SummConfig(**job['config']), **(overrides or {}))
    cfg.validate()
    manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    best = manager.best() if cfg.inference_checkpoint == 'best' else manager.latest()
    if not best:
        raise FileNotFoundError('Finish at least one complete training epoch first')
    variant = identity({'checkpoint': best[1]['sha256'], 'data': job['prepared_identity'], 'decode': generation_choices(cfg)})[:20]
    directory = Path(cfg.run_dir) / 'generation' / variant
    directory.mkdir(parents=True, exist_ok=True)
    signature = {'checkpoint_sha256': best[1]['sha256'], 'prepared_identity': job['prepared_identity'],
                 'generation': generation_choices(cfg)}
    signature_file = directory / 'generation_identity.json'
    if signature_file.exists() and read_json(signature_file) != signature:
        raise ValueError('Saved generation signature changed')
    atomic_json(signature_file, signature)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''    mp.spawn(inference_worker, args=(str(job_path), tuple(splits), generation_choices(cfg), variant, cfg.inference_checkpoint), nprocs=2, join=True)
    results = {}
    for split in splits:
        values = {}
        for rank in range(2):
            rows = partial_rows(directory / f'{split}_rank{rank}.jsonl')
            if set(values) & set(rows):
                raise ValueError('Inference replicas generated duplicate case IDs')
            values.update(rows)
        expected = job['cache']['cases'][split]
        if set(values) != {row['id'] for row in expected}:
            raise ValueError('Incomplete inference: not every original document has a summary')
        path = directory / (split + '_prediction.json')
        atomic_json(path, [{'id': row['id'], 'summary': values[row['id']]['summary']} for row in expected])
        read_predictions(path)
        results[split] = path
    return directory, results


def package(prediction_path, output, name, test_ids):
    predicted = read_predictions(prediction_path)
    if set(predicted) != set(test_ids) or len(test_ids) != len(set(test_ids)):
        raise ValueError('Submission IDs differ from original test documents')
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    target = output / 'prediction.json'
    if Path(prediction_path).resolve() != target.resolve():
        shutil.copyfile(prediction_path, target)
    archive_path = output / ('submission_' + name + '.zip')
    temporary = archive_path.with_suffix('.zip.partial')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(target, 'prediction.json')
    temporary.replace(archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None or archive.namelist() != ['prediction.json'] or archive.read('prediction.json') != target.read_bytes():
            raise ValueError('ZIP layout/content verification failed')
    return {'zip_path': str(archive_path), 'zip_sha256': sha256_file(archive_path), 'prediction_sha256': sha256_file(target)}


def export(job_path, teacher_only=False):
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''    job = read_json(job_path)
    cfg = SummConfig(**job['config'])
    root = Path(cfg.run_dir)
    summary = read_json(root / 'training_summary.json')
    if summary['status'] != 'complete':
        raise ValueError('Paused/incomplete training cannot produce a completed submission')
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    best = manager.best() if cfg.inference_checkpoint == 'best' else manager.latest()
    if teacher_only:
        if cfg.family == 'b':
            raise ValueError('Export a larger completed A/C/D Phase 6 teacher for B')
        directory, generated = parallel_generate(job_path, ('train',))
        target = root / 'train_summaries.jsonl'
        values = read_predictions(generated['train'])
        with target.with_suffix('.jsonl.partial').open('w', encoding='utf-8', newline='\n') as stream:
            for row in job['cache']['cases']['train']:
                stream.write(json.dumps({'id': row['id'], 'summary': values[row['id']]}, ensure_ascii=False) + '\n')
        target.with_suffix('.jsonl.partial').replace(target)
        receipt = {'version': 'summ-teacher-v1', 'status': 'complete', 'prepared_identity': job['prepared_identity'],
                   'teacher_experiment': cfg.experiment_id, 'teacher_checkpoint_sha256': best[1]['sha256'],
                   'teacher_parameters': job['parent_parameter_count'], 'sha256': sha256_file(target),
                   'training_cases': len(values), 'teacher_runtime_source_sha256': job['runtime_source_sha256'],
                   'generation': generation_choices(cfg), 'test_labels_used': False, 'teacher_is_own_phase6_model': True}
        atomic_json(root / 'teacher_export_receipt.json', receipt)
        return receipt
    directory, generated = parallel_generate(job_path, ('tune', 'confirm', 'test'))
    scores = {}
    for split in ('tune', 'confirm'):
        references = {row['id']: row['summary'] for row in records(Path(job['prepared_dir']) / (split + '.jsonl'))}
        scores[split] = evaluate(read_predictions(generated[split]), references)
    model_report = read_json(root / 'model_report.json')
    result = {'version': 'summ-output-v1', 'status': 'submission_verified', 'experiment_id': cfg.experiment_id,
              'family': cfg.family, 'method': cfg.method, 'prepared_identity': job['prepared_identity'],
              'fingerprint': job['fingerprint'], 'checkpoint_sha256': best[1]['sha256'], 'scores': scores,
              'parameter_report': model_report, 'generation': generation_choices(cfg),
              'generation_directory': str(directory), 'checkpoint_selection': cfg.inference_checkpoint,
              'public_BERTScore_not_computed_yet': True, 'private_test_score': None}
    result.update(package(generated['test'], root, cfg.experiment_id, [row['id'] for row in job['cache']['cases']['test']]))
    atomic_json(root / 'submission_receipt.json', result)
    atomic_json(root / 'final_report.json', result)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('predict.py', []).append(r'''    return result
''')


[Markdown Cell]

`preparation.py` — SQLite streaming, duplicate-document grouping, held-out split and artifact verification.

In [ ]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''"""Disk-backed document audit and leakage-aware training/tune/confirmation split."""
import hashlib
import json
from pathlib import Path
import sqlite3
import unicodedata
import zipfile
import shutil
import re

from .io_utils import atomic_json, identity, read_json, sha256_file, records
from .schema import stream_cases, FILES, download_official


def document_group(text):
    return hashlib.sha256(' '.join(unicodedata.normalize('NFKC', text).casefold().split()).encode()).hexdigest()


def verify_prepared(root):
    root = Path(root)
    manifest = read_json(root / 'prepared_manifest.json')
    if (manifest.get('version') != 'summ-data-v1' or manifest.get('status') != 'complete'
            or identity({k: v for k, v in manifest.items() if k != 'identity'}) != manifest['identity']):
        raise ValueError('Attach the completed Phase 6 notebook 00 output, not another phase')
    for name, description in manifest['artifacts'].items():
        if Path(name).name != name or (root / name).stat().st_size != description['bytes'] or sha256_file(root / name) != description['sha256']:
            raise ValueError('Prepared artifact missing or changed: ' + name)
    return manifest


def prepared_input(cfg):
    paths = [Path(cfg.prepared_dir)] if cfg.prepared_dir else [p.parent for p in Path(cfg.input_root).rglob('prepared_manifest.json')
        if read_json(p).get('version') == 'summ-data-v1']
    archives = [path for path in paths if path.is_file() and path.suffix == '.zip']
    paths = [path for path in paths if path not in archives]
    if not paths and not archives:
        archives = sorted(Path(cfg.input_root).rglob('P6_Prepared_Data.zip'))
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            manifests = [name for name in archive.namelist() if name.endswith('prepared_manifest.json')]
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''            if len(manifests) != 1:
                raise ValueError('Prepared ZIP must contain one prepared_manifest.json')
            source = manifests[0]
            manifest = json.loads(archive.read(source))
            if (manifest.get('version') != 'summ-data-v1' or manifest.get('status') != 'complete'
                    or not re.fullmatch('[0-9a-f]{64}', str(manifest.get('identity', '')))
                    or identity({k: v for k, v in manifest.items() if k != 'identity'}) != manifest['identity']):
                raise ValueError('Invalid prepared ZIP identity/version')
            prefix = source[:-len('prepared_manifest.json')]
            allowed = {'train.jsonl', 'tune.jsonl', 'confirm.jsonl', 'test.jsonl', 'data_audit.json'}
            if set(manifest['artifacts']) != allowed or sum(v['bytes'] for v in manifest['artifacts'].values()) > 2 << 30:
                raise ValueError('Unexpected prepared ZIP contents or size')
            if any(value['bytes'] < 0 or archive.getinfo(prefix + name).file_size != value['bytes']
                   for name, value in manifest['artifacts'].items()):
                raise ValueError('Prepared ZIP size does not match its data manifest')
            destination = Path(cfg.run_dir) / 'prepared_input' / manifest['identity'][:16]
            destination.mkdir(parents=True, exist_ok=True)
            for name in [*allowed, 'prepared_manifest.json']:
                target = destination / name
                temporary = target.with_name(target.name + '.partial')
                with archive.open(prefix + name) as incoming, temporary.open('wb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing, 1 << 20)
                temporary.replace(target)
            paths.append(destination)
    if not paths:
        raise FileNotFoundError('Attach the complete P6_00 output: prepared_manifest.json and train/tune/confirm/test.jsonl')
    verified = [(path, verify_prepared(path)) for path in paths]
    if len({manifest['identity'] for _, manifest in verified}) != 1:
        raise ValueError('Several Phase 6 datasets differ. Set PREPARED_DIR to one exact prepared directory')
    return verified[0]


def prepare(paths, output, provenance, seed=781, heldout_fraction=.10, source_layouts=None, field_map=None):
    """Every public source row is retained in exactly one text-group partition."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    choices = {'provenance': provenance, 'raw_sha256': {key: sha256_file(path) for key, path in paths.items()},
               'seed': seed, 'heldout_fraction': heldout_fraction, 'source_layouts': source_layouts or {}, 'field_map': field_map or {}}
    existing = output / 'prepared_manifest.json'
    if existing.exists():
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''        manifest = verify_prepared(output)
        if manifest['choices'] != choices:
            raise ValueError('Preparation choices changed; use a new output directory')
        return manifest
    database = sqlite3.connect(output / 'audit.sqlite3')
    database.execute('DROP TABLE IF EXISTS examples')
    database.execute('DROP TABLE IF EXISTS tests')
    database.execute('CREATE TABLE examples (rowid_internal INTEGER PRIMARY KEY, source_id TEXT, document TEXT, summary TEXT, group_id TEXT)')
    database.execute('CREATE TABLE tests (id TEXT PRIMARY KEY, document TEXT, group_id TEXT, views INTEGER)')
    try:
        for number, row in enumerate(stream_cases(paths['train'], 'train', (source_layouts or {}).get('train'), (field_map or {}).get('train'))):
            database.execute('INSERT INTO examples VALUES (?,?,?,?,?)', (number, row['id'], row['document'], row['summary'], document_group(row['document'])))
        database.commit()
        test_views = 0
        for row in stream_cases(paths['test'], 'test', (source_layouts or {}).get('test'), (field_map or {}).get('test')):
            test_views += 1
            previous = database.execute('SELECT document, views FROM tests WHERE id=?', (row['id'],)).fetchone()
            text, views = (row['document'], 1) if previous is None else (previous[0] + '\n\n' + row['document'], previous[1] + 1)
            database.execute('INSERT OR REPLACE INTO tests VALUES (?,?,?,?)', (row['id'], text, document_group(text), views))
        database.commit()
        groups = [row[0] for row in database.execute('SELECT DISTINCT group_id FROM examples')]
        if len(groups) < 10 or not 0 < heldout_fraction < .4:
            raise ValueError('Need at least ten independent document groups and a heldout fraction in (0,.4)')
        groups.sort(key=lambda group: identity({'seed': seed, 'group': group}))
        per_partition = max(1, round(len(groups) * heldout_fraction / 2))
        tune, confirm = set(groups[:per_partition]), set(groups[per_partition:2 * per_partition])
        counts = dict(train=0, tune=0, confirm=0, test=0)
        lengths = []
        streams = {split: (output / (split + '.jsonl.partial')).open('w', encoding='utf-8', newline='\n') for split in counts}
        try:
            for number, identifier, text, summary, group in database.execute('SELECT * FROM examples ORDER BY rowid_internal'):
                split = 'tune' if group in tune else 'confirm' if group in confirm else 'train'
                row = {'id': f'train_{number:08d}', 'source_id': identifier, 'document': text, 'summary': summary, 'group': group}
                streams[split].write(json.dumps(row, ensure_ascii=False) + '\n')
                counts[split] += 1
                lengths.append(len(summary.split()))
            for identifier, text, group, views in database.execute('SELECT * FROM tests ORDER BY rowid'):
                streams['test'].write(json.dumps({'id': identifier, 'document': text, 'group': group, 'source_views': views}, ensure_ascii=False) + '\n')
                counts['test'] += 1
        finally:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('preparation.py', []).append(r'''            for stream in streams.values():
                stream.close()
        if not all(counts.values()):
            raise ValueError('Each partition must contain documents')
        for split in counts:
            (output / (split + '.jsonl.partial')).replace(output / (split + '.jsonl'))
        overlaps = database.execute('SELECT COUNT(*) FROM tests WHERE group_id IN (SELECT group_id FROM examples)').fetchone()[0]
        audit = {'counts': counts, 'all_public_source_rows_retained': sum(counts[s] for s in ('train', 'tune', 'confirm')),
                 'independent_document_groups': len(groups), 'test_source_views': test_views,
                 'public_train_test_exact_text_overlap': overlaps, 'near_duplicates_checked': False,
                 'test_annotation_values_used': False, 'split_unit': 'normalized document SHA256 group',
                 'summary_words': {'min': min(lengths), 'median': sorted(lengths)[len(lengths) // 2], 'max': max(lengths)},
                 'reference_free_test_projection': True}
        atomic_json(output / 'data_audit.json', audit)
        artifacts = {name: {'bytes': (output / name).stat().st_size, 'sha256': sha256_file(output / name)}
                     for name in [*(s + '.jsonl' for s in counts), 'data_audit.json']}
        manifest = {'version': 'summ-data-v1', 'status': 'complete', 'choices': choices, 'counts': counts, 'artifacts': artifacts}
        manifest['identity'] = identity(manifest)
        atomic_json(existing, manifest)
        return manifest
    finally:
        database.close()
''')


[Markdown Cell]

`schema.py` — Bounded array/wrapper/ID-map JSON parsing and test input allowlisting.

In [ ]:
# [Code Cell]
module_parts.setdefault('schema.py', []).append(r'''"""Bounded official JSON parsing; no training/test annotation confusion."""
from pathlib import Path
import json
from .io_utils import atomic_json, identity, read_json, sha256_file, records
FILES={'train':'Phase_6_Summ/Train/train.json','test':'Phase_6_Summ/Test/test.json'}
IDS=('id','doc_id','document_id','case_id')
TEXTS=('document','text','judgment','judgement','case_text','content','paragraphs','sentences')
SUMMARIES=('summary','reference_summary','reference','abstract','headnote','gold','target')
CONTAINERS=('data','payload','cases','documents','docs','records','examples','items','dataset','train','test')

def normalized_record(value, split, outer_id=None, fields=None, annotation_audit=None):
    fields=fields or {}
    if isinstance(value,str) and outer_id is not None:value={'id':outer_id,'document':value}
    if not isinstance(value,dict):raise ValueError('Each document must be an object')
    get=lambda key,aliases,required=False:value[fields[key]] if key in fields else field(value,aliases,required)
    identifier=get('id',IDS)
    if identifier is None:identifier=outer_id
    if isinstance(identifier,int) and not isinstance(identifier,bool):identifier=str(identifier)
    if identifier is not None and (not isinstance(identifier,str) or not identifier.strip()):raise ValueError('Invalid document ID')
    if split=='test' and identifier is None:raise ValueError('Test document needs its original ID; set FIELD_MAP')
    text=flatten_text(get('text',TEXTS,True))
    if not text.strip():raise ValueError('Empty document: '+str(identifier))
    row={'id':identifier,'document':text}
    if split!='test':
        summary=flatten_text(get('summary',SUMMARIES,True))
        if not summary.strip():raise ValueError('Empty training summary: '+str(identifier))
        row['summary']=summary
    return row


def field(row: dict, names: tuple, required=False):
    found = [(key, row[key]) for key in names if key in row]
    if not found:
        if required:
            raise ValueError('Missing field: expected one of ' + ', '.join(names))
        return None
    if any(value != found[0][1] for _, value in found[1:]):
        raise ValueError('Contradictory redundant fields: ' + ', '.join(k for k, _ in found))
    return found[0][1]

''')


In [ ]:
# [Code Cell]
module_parts.setdefault('schema.py', []).append(r'''def flatten_text(value) -> str:
    """Preserve all original text/paragraph order; never shorten a case or split labels."""
    if isinstance(value, str):
        return value
    if isinstance(value, dict) and 'text' in value:
        return flatten_text(value['text'])
    if isinstance(value, list):
        return '\n\n'.join(flatten_text(part) for part in value)
    raise ValueError('Case text must be a string or ordered string/paragraph lists; set field overrides for a different schema')

def layout(path: Path, prefix='') -> tuple[str, str, dict]:
    """Inspect immediate fields with bounded metadata; named wrappers may be nested."""
    import ijson
    kind = None
    fields = {}
    with Path(path).open('rb') as stream:
        for current, event, value in ijson.parse(stream):
            if current == prefix and event in ('start_array', 'start_map') and kind is None:
                kind = event
                if kind == 'start_array':
                    return 'items', (prefix + '.item').lstrip('.'), {'kind': 'array', 'prefix': prefix}
            if current == prefix and event == 'map_key' and (len(fields) < 32 or value in CONTAINERS):
                fields[value] = None
            for key in tuple(fields):
                if fields[key] is None and current == (prefix + '.' + key).lstrip('.') and event not in ('map_key', 'end_map', 'end_array'):
                    fields[key] = event
    if kind != 'start_map':
        raise ValueError('Expected JSON array or object at prefix ' + prefix)
    containers = [key for key in CONTAINERS if fields.get(key) in ('start_array', 'start_map')]
    if any(fields.get(key) in ('string', 'number') for key in IDS) and any(key in fields for key in TEXTS):
        return 'single', prefix, {'kind': 'record', 'fields': fields, 'prefix': prefix}
    if len(containers) > 1:
        raise ValueError('Multiple document containers; set SOURCE_LAYOUTS explicitly: ' + repr(containers))
    if containers:
        return layout(path, (prefix + '.' + containers[0]).lstrip('.'))
    if any(key in fields for key in TEXTS):
        raise ValueError('Column-oriented/unknown JSON detected. Set SOURCE_LAYOUTS/FIELD_MAP explicitly; use record arrays or ID-record maps.')
    return 'mapping', prefix, {'kind': 'ID_mapping', 'fields': fields, 'prefix': prefix}

def stream_cases(path: Path, split: str, specification=None, fields=None, annotation_audit=None):
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('schema.py', []).append(r'''    """At most one case object/text in memory; supported layouts never load the full array."""
    import ijson
    specification = specification or {}
    if specification:
        mode, prefix = specification['mode'], specification.get('prefix', '')
    else:
        mode, prefix, _ = layout(path)
    if mode not in ('items', 'mapping', 'single'):
        raise ValueError('SOURCE_LAYOUTS mode must be items, mapping or single')
    with Path(path).open('rb') as stream:
        if mode == 'mapping':
            for key, value in ijson.kvitems(stream, prefix):
                yield normalized_record(value, split, key, fields, annotation_audit)
        else:
            for value in ijson.items(stream, prefix):
                yield normalized_record(value, split, fields=fields, annotation_audit=annotation_audit)

def download_official(repository, revision, destination, token, path_map=None):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        commit = api.dataset_info(repository, revision=revision).sha
        available = set(api.list_repo_files(repository, repo_type='dataset', revision=commit))
    except Exception as error:
        raise RuntimeError('Enable Kaggle Internet and HF_TOKEN or HF Secret; accept the official dataset access terms') from error
    selected = {split:(path_map or {}).get(split,name) for split,name in FILES.items()}
    if set(selected.values()) - available:
        raise ValueError('Official Phase 6 paths changed. Inspect the repository and set HF_PATH_MAP explicitly.')
    paths = {split:Path(hf_hub_download(repository, name, repo_type='dataset', revision=commit,
             token=token, local_dir=str(destination))) for split,name in selected.items()}
    return paths, {'repository':repository, 'commit':commit, 'paths':selected}
''')


[Markdown Cell]

`screen.py` — Tune-only decoding selection, frozen controls and confirmation reporting.

In [ ]:
# [Code Cell]
module_parts.setdefault('screen.py', []).append(r'''"""Choose decoding on public tune only; confirm once and preserve six techniques."""
from pathlib import Path
import shutil
import torch
from .config import SummConfig
from .evaluation import evaluate, read_predictions
from .io_utils import atomic_json, read_json, records, sha256_file
from .predict import parallel_generate, package
from .state import Checkpoints

DEFAULT_VARIANTS = [
    {'generation_max_tokens': 256, 'beam_size': 1, 'repetition_penalty': 1.1, 'no_repeat_ngram': 3, 'length_penalty': .8},
    {'generation_max_tokens': 256, 'beam_size': 2, 'repetition_penalty': 1.1, 'no_repeat_ngram': 3, 'length_penalty': 1.},
    {'generation_max_tokens': 512, 'beam_size': 2, 'repetition_penalty': 1.05, 'no_repeat_ngram': 4, 'length_penalty': 1.},
]


def screen(job_path, variants=None, bert=True, selections=('best', 'latest')):
    job = read_json(job_path)
    cfg = SummConfig(**job['config'])
    root, prepared = Path(cfg.run_dir), Path(job['prepared_dir'])
    references = {row['id']: row['summary'] for row in records(prepared / 'tune.jsonl')}
    trials = []
    seen = set()
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    for selection in selections:
        checkpoint = manager.best() if selection == 'best' else manager.latest()
        if checkpoint[1]['sha256'] in seen:
            continue
        seen.add(checkpoint[1]['sha256'])
        for candidate in variants or DEFAULT_VARIANTS:
            directory, values = parallel_generate(job_path, ('tune',), dict(candidate, inference_checkpoint=selection))
            scores = evaluate(read_predictions(values['tune']), references, bert, 'cuda' if torch.cuda.is_available() else 'cpu')
            trials.append({'generation': candidate, 'tune': scores, 'directory': str(directory),
                           'checkpoint_selection': selection, 'checkpoint_sha256': checkpoint[1]['sha256'],
                           'prediction_sha256': sha256_file(values['tune'])})
            atomic_json(root / 'decoding_trials.json', {'status': 'in_progress', 'trials': trials})
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    key = 'bertscore_f1' if bert else 'rougeL_f1'
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('screen.py', []).append(r'''    winner = max(trials, key=lambda item: (item['tune'][key], item['tune']['rougeL_f1']))
    # Lock the winner BEFORE computing confirmation/test outputs.
    atomic_json(root / 'frozen_decoding_rule.json', {'selected_on': 'tune ' + key, **winner})
    directory, outputs = parallel_generate(job_path, ('confirm', 'test'), dict(winner['generation'], inference_checkpoint=winner['checkpoint_selection']))
    confirmation = {row['id']: row['summary'] for row in records(prepared / 'confirm.jsonl')}
    score = evaluate(read_predictions(outputs['confirm']), confirmation, bert, 'cuda' if torch.cuda.is_available() else 'cpu')
    result = {'version': 'summ-screen-v1', 'status': 'submission_verified', 'experiment_id': cfg.experiment_id,
              'family': cfg.family, 'method': cfg.method, 'selected_on': 'tune ' + key,
              'tune': winner['tune'], 'confirmation': score, 'generation': winner['generation'],
              'private_test_score': None, 'counts_as_new_adaptation_method': False,
              'generation_directory': str(directory), 'checkpoint_selection': winner['checkpoint_selection'], 'trials': trials}
    result['checkpoint_sha256'] = read_json(directory / 'generation_identity.json')['checkpoint_sha256']
    result.update(package(outputs['test'], root / 'screened', cfg.experiment_id + '_screened',
                          [row['id'] for row in job['cache']['cases']['test']]))
    atomic_json(root / 'screen_report.json', result)
    return result
''')


[Markdown Cell]

`source_identity.py` — Portable structural source identities across supported Python versions.

In [ ]:
# [Code Cell]
module_parts.setdefault('source_identity.py', []).append(r'''"""Portable structural source identity, independent of ast.dump display changes."""
import ast
import hashlib
import json

ALGORITHM = 'python-canonical-ast-v1'


def canonical_tree(value):
    """Keep meaningful node fields; omit absent/default optional fields consistently.

    Python 3.12 added empty type_params; Python 3.13 changed ast.dump's default
    omission of empty lists. We never hash that debugging representation. Real
    nonempty fields, including generic type parameters, remain in the identity.
    Source locations, whitespace and comments do not affect model semantics.
    """
    if isinstance(value, ast.AST):
        fields = {}
        for name, item in ast.iter_fields(value):
            if item is None or (isinstance(item, list) and not item):
                continue
            fields[name] = canonical_tree(item)
        return {'node': type(value).__name__, 'fields': fields}
    if isinstance(value, list):
        return [canonical_tree(item) for item in value]
    if isinstance(value, bytes):
        return {'constant_type': 'bytes', 'hex': value.hex()}
    if value is Ellipsis:
        return {'constant_type': 'ellipsis'}
    if isinstance(value, complex):
        return {'constant_type': 'complex', 'real': value.real, 'imag': value.imag}
    return value


def source_fingerprint(source: str) -> str:
    """Hash a deterministic JSON representation, not Python-version-specific text."""
    payload = {'algorithm': ALGORITHM, 'tree': canonical_tree(ast.parse(source))}
    serialized = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(serialized.encode('ascii')).hexdigest()
''')


[Markdown Cell]

`state.py` — Full optimizer/scaler/scheduler/cursor/per-rank-RNG checkpoints in atomic rotating slots.

In [ ]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible summarization checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_score'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('state.py', []).append(r'''            raise ValueError('Incomplete/incompatible summarization checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        estimate=sum(tensor.numel()*tensor.element_size() for tensor in payload['model'].values())*3+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_score':payload['cursor']['best_score']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
            atomic_json(self.directory/'best.pt.json',receipt)
        return path,receipt
''')


[Markdown Cell]

`train.py` — Two NCCL spawn workers, FP16 AMP, global target-token normalization, complete epochs and collective stops.

In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''"""Two T4 NCCL workers; exact full-state cursor, token-normalized FP16 training."""
import argparse
from contextlib import nullcontext
from datetime import timedelta
import math
import os
from pathlib import Path
import signal
import socket
import time

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from tqdm.auto import tqdm
from .config import SummConfig
from .data import SummaryWindows, collate_one, epoch_order
from .io_utils import atomic_json, read_json, run_lock
from .model import build_model
from .state import Checkpoints, WarmupCosine, seed_everything, capture_rng, restore_rng


def initial_cursor():
    return {'epoch': 0, 'next_batch': 0, 'global_step': 0, 'optimizer_steps': 0,
            'best_score': -1e30, 'complete': False, 'bad_epochs': 0, 'history': [],
            'loss_sum': 0., 'weight_sum': 0., 'windows': 0, 'overflows': 0, 'consecutive_overflows': 0}


def save_all(rank, manager, model, optimizer, scaler, scheduler, cursor, cfg, device, best=False, paused=False):
    states = [None, None]
    dist.all_gather_object(states, capture_rng(device))
    failure = [None]
    if rank == 0:
        try:
            actual = model.module if hasattr(model, 'module') else model
            payload = {'version': 1, 'fingerprint': manager.fingerprint, 'config': cfg.__dict__,
                       'model': actual.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                       'scheduler': scheduler.state_dict(), 'epoch': cursor['epoch'], 'step': cursor['global_step'],
                       'cursor': cursor, 'rng_by_rank': states}
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''            manager.commit(payload, best=best)
            atomic_json(Path(cfg.run_dir) / 'training_summary.json',
                        {'status': 'complete' if cursor['complete'] else 'paused_recoverable' if paused else 'training', **cursor})
        except Exception as error:
            failure[0] = str(error)
    dist.broadcast_object_list(failure, src=0)
    if failure[0]:
        raise RuntimeError('Checkpoint commit failed: ' + failure[0])
    dist.barrier()


def optimizer_for(model, cfg):
    backbone, memory = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (backbone if name.startswith('backbone.') else memory).append(parameter)
    groups = [{'params': backbone, 'lr': cfg.backbone_lr}, {'params': memory, 'lr': cfg.memory_lr}]
    groups = [group for group in groups if group['params']]
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay, fused=torch.cuda.is_available())


@torch.no_grad()
def validation(rank, model, job, cfg, device):
    dataset = SummaryWindows(job['cache'], cfg, 'tune')
    model.eval()
    values = torch.zeros(2, device=device, dtype=torch.float64)
    for index in range(rank, len(dataset), cfg.world_size):
        row = dataset[index]
        with torch.autocast('cuda', dtype=torch.float16):
            loss = model(**collate_one(row, cfg.chunk_tokens, device))
        values[0] += loss.double()
        values[1] += len(row['labels'][row['labels'] != -100])
    dist.all_reduce(values)
    total, count = values.cpu().tolist()
    if count <= 0 or not math.isfinite(total):
        raise ValueError('Invalid complete tune NLL evaluation')
    return {'tune_nll': total / count, 'tune_target_tokens': int(count)}


def worker(rank, job, port, event):
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''    cfg = SummConfig(**job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank,
                            world_size=cfg.world_size, timeout=timedelta(hours=2))
    try:
        seed_everything(cfg.seed)
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest = manager.latest()
        payload = manager.load(latest) if latest else None
        model = build_model(job, cfg, device, load_parent=payload is None)
        if payload:
            model.load_state_dict(payload['model'], strict=True)
        ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, find_unused_parameters=False, gradient_as_bucket_view=True)
        optimizer = optimizer_for(model, cfg)
        dataset = SummaryWindows(job['cache'], cfg)
        slots = cfg.world_size * cfg.accumulation_steps
        steps = math.ceil(len(dataset) / slots)
        scheduler = WarmupCosine(optimizer, max(1, steps * cfg.max_epochs), cfg.warmup_fraction, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        cursor = initial_cursor()
        if payload:
            optimizer.load_state_dict(payload['optimizer'])
            scaler.load_state_dict(payload['scaler'])
            for name in ('total_steps', 'warmup_steps', 'minimum', 'base_lrs'):
                setattr(scheduler, name, payload['scheduler'][name])
            scheduler.load_state_dict(payload['scheduler'])
            cursor = payload['cursor']
            restore_rng(payload['rng_by_rank'][rank], device)
            del payload
            if cursor['global_step'] != cursor['epoch'] * steps + cursor['next_batch']:
                raise ValueError('Saved cursor differs from the exhaustive summary-window plan')
        else:
            seed_everything(cfg.seed + rank)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
        if cursor['complete'] and job['extension_requested']:
            cursor.update(complete=False, bad_epochs=0)
        if cursor['complete']:
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''            return
        started = time.monotonic()
        for epoch in range(cursor['epoch'], job['target_epochs']):
            ddp.train()
            order = epoch_order(len(dataset), cfg.seed, epoch, slots)
            progress = tqdm(total=steps, initial=cursor['next_batch'], disable=rank != 0,
                desc=f'{cfg.experiment_id} epoch {epoch + 1}/{job["target_epochs"]}')
            for step in range(cursor['next_batch'], steps):
                stop = torch.tensor(int(event.is_set() or time.monotonic() - started >= cfg.max_run_seconds), device=device)
                dist.all_reduce(stop, op=dist.ReduceOp.MAX)
                if stop.item():
                    progress.close()
                    save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device, paused=True)
                    return
                indices = order[step].tolist()
                denominator = sum(dataset.target_weight(index) for index in indices)
                if denominator <= 0:
                    raise ValueError('Empty global target update')
                optimizer.zero_grad(set_to_none=True)
                stats = torch.zeros(2, device=device, dtype=torch.float64)
                for micro in range(cfg.accumulation_steps):
                    index = indices[micro * cfg.world_size + rank]
                    row = dataset[index]
                    context = ddp.no_sync() if micro + 1 < cfg.accumulation_steps else nullcontext()
                    with context:
                        with torch.autocast('cuda', dtype=torch.float16):
                            loss = ddp(**collate_one(row, cfg.chunk_tokens, device)) * row['weight']
                        scaler.scale(loss * cfg.world_size / denominator).backward()
                    stats[0] += loss.detach().double()
                    stats[1] += int(index >= 0)
                dist.all_reduce(stats)
                loss_sum, windows = stats.cpu().tolist()
                if not math.isfinite(loss_sum):
                    raise FloatingPointError('Nonfinite objective; recover the last committed state')
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous_scale:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''                    scheduler.advance()
                    cursor['optimizer_steps'] += 1
                    cursor['consecutive_overflows'] = 0
                else:
                    cursor['overflows'] += 1
                    cursor['consecutive_overflows'] += 1
                if cursor['consecutive_overflows'] > cfg.max_consecutive_overflows:
                    raise FloatingPointError('Repeated AMP overflow; inspect numeric/checkpoint reports')
                cursor.update(global_step=cursor['global_step'] + 1, next_batch=step + 1,
                              loss_sum=cursor['loss_sum'] + loss_sum, weight_sum=cursor['weight_sum'] + denominator,
                              windows=cursor['windows'] + int(windows))
                optimizer.zero_grad(set_to_none=True)
                if rank == 0:
                    progress.update()
                    if cursor['global_step'] % cfg.log_every_steps == 0 or step + 1 == steps:
                        progress.set_postfix(loss=cursor['loss_sum'] / cursor['weight_sum'], lr=optimizer.param_groups[0]['lr'],
                                             step=cursor['global_step'], amp_skips=cursor['overflows'])
                if cursor['global_step'] % 500 == 0:
                    save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
            progress.close()
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device)
            evaluated = validation(rank, ddp.module, job, cfg, device)
            if cursor['windows'] != len(dataset):
                raise ValueError('Epoch did not visit every supervised gold/pseudo target window exactly once')
            score = -evaluated['tune_nll']
            improved = score > cursor['best_score']
            cursor['best_score'] = max(cursor['best_score'], score)
            cursor['bad_epochs'] = 0 if improved else cursor['bad_epochs'] + 1
            cursor['history'].append({'epoch': epoch + 1, 'train_nll': cursor['loss_sum'] / cursor['weight_sum'],
                                     'all_supervised_windows': cursor['windows'], 'amp_skips': cursor['overflows'], **evaluated})
            cursor.update(epoch=epoch + 1, next_batch=0, loss_sum=0., weight_sum=0., windows=0)
            cursor['complete'] = cursor['epoch'] >= job['target_epochs'] or (cursor['epoch'] >= cfg.minimum_epochs and cursor['bad_epochs'] >= cfg.patience)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, device, best=improved)
            if rank == 0:
                print({'epoch': cursor['epoch'], 'tune_nll': evaluated['tune_nll'], 'complete': cursor['complete']}, flush=True)
            if cursor['complete']:
                return
    finally:
        dist.destroy_process_group()

''')


In [ ]:
# [Code Cell]
module_parts.setdefault('train.py', []).append(r'''
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    job = read_json(args.job)
    from .jobs import hardware
    hardware()
    cfg = SummConfig(**job['config'])
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    with run_lock(Path(cfg.run_dir)):
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest, best = manager.latest(), manager.best()
        if best:
            manager.adopt(best, best=True)
        if latest:
            manager.adopt(latest)
            print('Resume exact saved step', latest[1]['step'], 'epoch', latest[1]['epoch'], flush=True)
        context = mp.get_context('spawn')
        event = context.Event()
        signal.signal(signal.SIGINT, lambda *_: event.set())
        signal.signal(signal.SIGTERM, lambda *_: event.set())
        with socket.socket() as sock:
            sock.bind(('127.0.0.1', 0))
            port = sock.getsockname()[1]
        mp.spawn(worker, args=(job, port, event), nprocs=2, join=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`workflow.py` — Single importable preparation/training/export workflow with explicit paused/incomplete status.

In [ ]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''"""Preflight -> full-state train/resume -> actual generated summary ZIP."""
import argparse
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import traceback
import zipfile
from .config import SummConfig
from .io_utils import atomic_json, read_json
from .jobs import prepare_job, hardware, sanity
from .predict import export
from .state import Checkpoints


def run_training(command):
    process = subprocess.Popen(command)
    old = {}
    def stop(signum, frame):
        if process.poll() is None:
            process.send_signal(signum)
    try:
        for signum in (signal.SIGINT, signal.SIGTERM):
            old[signum] = signal.signal(signum, stop)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    finally:
        for signum, handler in old.items():
            signal.signal(signum, handler)


def observations(root):
    target = root / 'observations.zip'
    with zipfile.ZipFile(target.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as archive:
        for file in root.rglob('*'):
            if file.is_file() and file.suffix in ('.json', '.txt', '.log') and file.stat().st_size < 8 << 20:
                archive.write(file, file.relative_to(root).as_posix())
    target.with_suffix('.zip.partial').replace(target)
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''

def stage_completed(job):
    cfg = SummConfig(**job['config'])
    root = Path(cfg.run_dir)
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    latest, best = manager.latest(), manager.best()
    if not latest or not best:
        raise FileNotFoundError('A completed task needs both a latest full-state checkpoint and a learned best checkpoint')
    payload = manager.load(latest)
    if not payload['cursor']['complete']:
        raise ValueError('An export cannot bypass paused or unfinished training')
    manager.adopt(best, best=True)
    manager.adopt(latest)
    atomic_json(root / 'training_summary.json', {'status': 'complete', **payload['cursor']})
    del payload
    for previous in job['resume_roots']:
        origin = Path(previous)
        if origin.resolve() == root.resolve():
            continue
        for name in ('model_report.json', 'final_report.json', 'submission_receipt.json', 'prediction.json',
                     'screen_report.json', 'decoding_trials.json', 'frozen_decoding_rule.json'):
            if not (root / name).is_file() and (origin / name).is_file():
                shutil.copyfile(origin / name, root / name)
        for archive in origin.glob('submission_*.zip'):
            if not (root / archive.name).exists():
                shutil.copyfile(archive, root / archive.name)
        # Keep completed inference records so an interrupted export does not restart.
        if (origin / 'generation').is_dir() and not (root / 'generation').exists():
            shutil.copytree(origin / 'generation', root / 'generation')
        if (origin / 'screened').is_dir() and not (root / 'screened').exists():
            shutil.copytree(origin / 'screened', root / 'screened')


def execute(plan_path, catalog_path):
    plan = read_json(plan_path)
    cfg = SummConfig(**plan['config'])
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    try:
''')


In [ ]:
# [Code Cell]
module_parts.setdefault('workflow.py', []).append(r'''        job = prepare_job(cfg, read_json(catalog_path))
        hardware()
        if plan['mode'] == 'train':
            sanity(job)
            run_training([sys.executable, '-u', '-m', 'summ.train', '--job', str(root / 'job.json')])
            if read_json(root / 'training_summary.json')['status'] != 'complete':
                result = {'version': 'summ-output-v1', 'status': 'paused_recoverable',
                          'next_action': 'Save the complete output and resume this same notebook. No final submission is claimed.'}
                atomic_json(root / 'final_report.json', result)
                return result
        elif plan['mode'] in ('export', 'teacher'):
            stage_completed(job)
        else:
            raise ValueError('Mode must be train, export or teacher')
        return export(root / 'job.json', teacher_only=plan['mode'] == 'teacher')
    except Exception as error:
        atomic_json(root / 'failure_report.json', {'status': 'failed', 'type': type(error).__name__,
                    'message': str(error), 'traceback': traceback.format_exc(),
                    'action': 'Keep full checkpoints. Read the input/teacher/failure report; no silent restart or fake submission is produced.'})
        raise
    finally:
        observations(root)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    args = parser.parse_args()
    print(execute(args.plan, args.catalog), flush=True)


if __name__ == '__main__':
    main()
''')


In [ ]:
# [Code Cell]
EXPECTED_SOURCE_SHA256 = {'__init__.py': 'c058812345f3788951feab67dd3a5d92996eedb96390900a013170238004049f', 'adaptation.py': 'd2c048dc8e024e49a90f2abf83bcbbdd8c9a2fbf7678d2b8eacf3308b844d718', 'compare.py': 'cb7b8304a1836007355455fc61d17cbd937e03576d7ae9be35e091acb9d95121', 'config.py': 'd9c8b3b2ad89f25e8755b8c6228e05cbcbce01fb3365da851ed845e489775dce', 'data.py': 'bbfb5a9cb0eb56a0a2a73d5870d3850200f41e16f49571094f97752f33ba514d', 'evaluation.py': '3a38aa9eab4c9372a1e3638c1fae5d7ffc84d39d9277b5bf5f40e3acc65a1649', 'experiments.py': '2e068ad8dde6fbd6ed0e8883dd2c5daa80b2c72a0d2044939b9c523f0d773950', 'generation.py': '7ddb2b6e10af8e911bf5b515f01da8cd1fcbd2e391a37a2805d75ab53185814e', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'jobs.py': 'e771c0de302c9a1c6c892f6077c534871262e4872befa719039882e95d568d80', 'model.py': 'dcb3c551618754e2196bedb3271335e29ffb4332779d568c4cd1ea81f148ce6f', 'native_contracts.py': 'f76cae1b17467620d65bcdd85d0aba6d15b5bf8cf7f8e4a8b30580b7951e368c', 'native_parents.py': '1c77968d9f566a65e4508d3cb590fe08fba058c7490de2e13baf3165095170ca', 'parents.py': '48ea34b6c454d31337d9b38c08556816b14af3ad48cd02430bedc51be0f83087', 'predict.py': 'c417b8690f4aa4325d03c74aaecc2e17eb2417460aabbd443da7880385c6cc2e', 'preparation.py': 'c17f209343cbbe0bca5cf093a602f0e6e66361518e891f3561f7f19825bf7d51', 'schema.py': 'edca4639fc4f0afe29bf63b3990c7ebdc9da9898ead67f347066261393fe1b95', 'screen.py': 'f643d5e00ba7fcf5c12f1a44fdb72b016f9aea41055ca95d2ccfc11fa1752aa6', 'source_identity.py': 'bff746c41ac050d03b1b02aee9d6e709c334697ad9ff6e69af4edacd0923690c', 'state.py': '7d2d65b3686fb26d27efae65b4b478fc39c59c7e311ebe2a87da7ac5c261223d', 'train.py': '94d1c3eb0c2a307daaffa5c9b261545b024fc7550f5b72d081b55c8f7c264ec2', 'workflow.py': '4d3d108abe6e2c006d8e47ea9fac909131d414a97e8c83711f3ab3410ca5d577'}
import hashlib
import importlib
code_directory = Path(OUTPUT_ROOT) / 'code'
package_directory = code_directory / 'summ'
package_directory.mkdir(parents=True, exist_ok=True)
for name, parts in module_parts.items():
    source = ''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED_SOURCE_SHA256[name], name
    compile(source, name, 'exec')
    target = package_directory / name
    if (Path(OUTPUT_ROOT) / 'job.json').is_file() and target.is_file() and target.read_bytes() != source.encode():
        raise ValueError('Saved runtime changed. Resume with its exact notebook; do not overwrite learned-source identity.')
    target.write_bytes(source.encode())
for name in list(sys.modules):
    if name == 'summ' or name.startswith('summ.'):
        del sys.modules[name]
sys.path.insert(0, str(code_directory))
importlib.invalidate_caches()
print('Verified Phase 6 modules:', len(EXPECTED_SOURCE_SHA256))


[Markdown Cell]

### Numeric checkpoint metadata compatibility
This helper is outside the hashed summarizer. Known numeric NumPy metadata types are supported with `weights_only=True`, including in every spawned child.

In [ ]:
# [Code Cell]
COMPAT_PARTS = {}


In [ ]:
# [Code Cell]
COMPAT_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''"""Read existing scalar-metadata checkpoints without changing the LSI runtime.

Only NumPy's numeric scalar constructor, dtype constructor and a fixed list of
numeric dtype classes are registered. Tensor/state loading stays weights_only.
No checkpoint is rewritten, and no globals are inferred from checkpoint text.
"""
from importlib import import_module
import io
import os

import numpy as np
import torch

VERSION = 'cs781-numpy-scalar-resume-v1'
DTYPE_NAMES = (
    'bool', 'int8', 'int16', 'int32', 'int64',
    'uint8', 'uint16', 'uint32', 'uint64',
    'float16', 'float32', 'float64',
)


def register_numpy_scalars() -> list[str]:
    """Allow the known numeric metadata types in this interpreter, idempotently."""
    namespace = 'numpy._core.multiarray' if int(np.__version__.split('.')[0]) >= 2 else 'numpy.core.multiarray'
    scalar_constructor = import_module(namespace).scalar
    accepted = [scalar_constructor, np.dtype, *[type(np.dtype(name)) for name in DTYPE_NAMES]]
    existing = torch.serialization.get_safe_globals()
    torch.serialization.add_safe_globals([value for value in accepted if value not in existing])
    os.environ['CS781_NUMPY_RESUME_PID'] = str(os.getpid())
    return [value.__module__ + '.' + value.__name__ for value in accepted]


def self_test() -> dict:
    """Exercise scalar loading on CPU without advancing any training RNG."""
    metadata = {name: np.dtype(name).type(1) for name in DTYPE_NAMES}
    stream = io.BytesIO()
    torch.save({'metadata': metadata, 'weights': torch.tensor([1., 2.])}, stream)
    stream.seek(0)
    loaded = torch.load(stream, map_location='cpu', weights_only=True)
    for name, original in metadata.items():
''')


In [ ]:
# [Code Cell]
COMPAT_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''        assert type(loaded['metadata'][name]) is type(original), name
        assert loaded['metadata'][name] == original, name
    assert torch.equal(loaded['weights'], torch.tensor([1., 2.]))
    return {'status': 'passed', 'compatibility_version': VERSION,
            'torch': str(torch.__version__), 'numpy': np.__version__,
            'weights_only': True, 'numeric_dtype_types': len(DTYPE_NAMES)}
''')


In [ ]:
# [Code Cell]
COMPAT_PARTS.setdefault('sitecustomize.py', []).append(r'''"""Run-local startup hook inherited by subprocesses and mp.spawn children.

Keep a pre-existing environment startup hook. This directory is inserted only
in the launch environment of this notebook, never installed into site-packages.
"""
import importlib.machinery
import importlib.util
import os
from pathlib import Path
import sys
import traceback


def _original_startup_hook() -> None:
    own_directory = Path(__file__).resolve().parent
    other_paths = [entry for entry in sys.path
                   if Path(entry or os.getcwd()).resolve() != own_directory]
    spec = importlib.machinery.PathFinder.find_spec('sitecustomize', other_paths)
    if spec is None or spec.loader is None:
        return
    if spec.origin and Path(spec.origin).resolve() == Path(__file__).resolve():
        return
    original = importlib.util.module_from_spec(spec)
    current = sys.modules.get('sitecustomize')
    sys.modules['sitecustomize'] = original
    try:
        spec.loader.exec_module(original)
    finally:
        if current is not None:
            sys.modules['sitecustomize'] = current
        else:
            sys.modules.pop('sitecustomize', None)


try:
    _original_startup_hook()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        from cs781_numpy_resume import register_numpy_scalars
        register_numpy_scalars()
except Exception:
''')


In [ ]:
# [Code Cell]
COMPAT_PARTS.setdefault('sitecustomize.py', []).append(r'''    traceback.print_exc()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        sys.stderr.write('Checkpoint compatibility startup failed; training was not started.\n')
        sys.stderr.flush()
        os._exit(78)
''')


In [ ]:
# [Code Cell]
COMPAT_SHA256 = {'cs781_numpy_resume.py': '5e82ff896f53b50f6bbe320503da0d0ef794368b99900bb4b61c3f835d82d17a', 'sitecustomize.py': '08ca4d65105de3acc37d18d6f3c24580927a6ab076c2814c2287774081750172'}
compat_directory = code_directory / 'checkpoint_compat'
compat_directory.mkdir(exist_ok=True)
for name, parts in COMPAT_PARTS.items():
    source = ''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest() == COMPAT_SHA256[name]
    (compat_directory / name).write_bytes(source.encode())
sys.path.insert(0, str(compat_directory))
import cs781_numpy_resume
cs781_numpy_resume.register_numpy_scalars()
print(cs781_numpy_resume.self_test())


[Markdown Cell]

### Optimized execution — every source and reference token is retained
`auto` keeps an existing run's optimizer, scheduler, scaler, RNG state and next-window cursor.
For a new experiment, all summary windows of a document share one differentiable source encoding.
Document grouping changes update boundaries and source-dropout sharing; it requires a NEW run.
An explicit same-method warm start reuses task weights and starts a new optimizer/epoch schedule.
No old partial epoch is reported as completed. Kernel rounding and dropout draws can differ from
the original implementation, so continuation preserves state/cursor rather than bitwise trajectories.

D uses the same sparse attention routes through a bounded masked SDPA path for lengths <=1024;
long sequences and token-by-token cached inference keep the original sparse implementation.
Vocabulary projections exclude context-only positions. Source/decoder recomputation bounds memory.
FP32 GRUs, FP16 AMP, T4 x2/NCCL, all five PEFT techniques and checkpoint cadence are retained.

In [ ]:
# [Code Cell]
FAST_PARTS = {}


[Markdown Cell]

`p6_fast/__init__.py` — checked execution optimization.

In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('__init__.py', []).append(r'''"""Versioned execution optimizations; foundation and summarizer weights stay unchanged."""
VERSION = 'p6-fast-v1'
''')


[Markdown Cell]

`p6_fast/control.py` — checked execution optimization.

In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''"""Audited execution identities, legacy-cursor continuation and explicit warm starts."""
import os
from pathlib import Path
import sys
import time
import torch
import torch.distributed as dist
from summ.io_utils import atomic_json, identity, read_json, sha256_file
from summ.parents import source_hashes
from summ.state import Checkpoints
from . import VERSION

PROFILE_FILE = 'execution_optimization.json'


def source_identity():
    return source_hashes(Path(__file__).parent)


def bind(plan_path, options_path):
    """Resolve auto layout before execution. Never regroup an existing cursor."""
    plan, options = read_json(plan_path), read_json(options_path)
    cfg = plan['config']
    root = Path(cfg['run_dir'])
    local = root / 'job.json'
    explicit = cfg.get('resume_run_dir')
    paths = [local] if local.is_file() else [Path(explicit) / 'job.json'] if explicit else sorted(Path(cfg['input_root']).rglob('job.json'))
    prior = [(file.parent, read_json(file)) for file in paths if file.is_file()]
    prior = [(folder, job) for folder, job in prior if job.get('version') == 'cs781-phase6-summ-v1'
             and job.get('config', {}).get('experiment_id') == cfg['experiment_id']]
    profiles = [read_json(folder / PROFILE_FILE) for folder, _ in prior if (folder / PROFILE_FILE).is_file()]
    if (root / PROFILE_FILE).is_file():
        profiles.append(read_json(root / PROFILE_FILE))
    if len({item['signature'] for item in profiles}) > 1:
        raise ValueError('Conflicting optimization profiles. Attach one exact saved run.')
    saved = profiles[0] if profiles else None
    requested = options.get('training_layout', 'auto')
    layout = saved['layout'] if saved else 'windows' if prior else 'documents'
    if requested != 'auto' and requested != layout:
        if prior or saved:
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''            raise ValueError('Cannot change the saved update plan. For document grouping use a NEW experiment ID and explicit WARM_START_RUN_DIR.')
        layout = requested
    if layout not in ('windows', 'documents'):
        raise ValueError('TRAINING_LAYOUT must be auto/windows/documents')
    warm = None
    if saved:
        if saved['source_sha256'] != source_identity() or saved['version'] != VERSION:
            raise ValueError('Optimization source changed. Resume using its exact optimized notebook.')
        warm = saved.get('warm_start')
    elif options.get('warm_start_run_dir'):
        if prior:
            raise ValueError('Warm start and exact continuation are different. Use a NEW experiment ID for a warm start.')
        origin = Path(options['warm_start_run_dir'])
        old = read_json(origin / 'job.json')
        if old['version'] != 'cs781-phase6-summ-v1':
            raise ValueError('Only this own Phase 6 summarizer can supply a warm start')
        if source_hashes(origin / 'code/summ') != old['runtime_source_sha256'] or old['runtime_source_sha256'] != source_hashes(Path(__import__('summ').__file__).parent):
            raise ValueError('Warm-start source snapshot differs from the reviewed own Phase 6 runtime')
        if (old['config']['family'], old['config']['method']) != (cfg['family'], cfg['method']):
            raise ValueError('Warm start must use the SAME model and PEFT method')
        manager = Checkpoints(origin, old['fingerprint'], 0)
        choice = manager.latest()  # preserve newest learned task weights, including paused runs
        if choice is None:
            raise FileNotFoundError('Warm start needs actual .pt weights and intact .pt.json receipts')
        manager.load(choice)  # validate full cursor, optimizer schedule and receipts
        warm = {'run_dir': str(origin), 'checkpoint': str(choice[0]), 'sha256': choice[1]['sha256'],
                'fingerprint': old['fingerprint'], 'source_epoch': choice[1]['epoch'], 'source_step': choice[1]['step']}
    body = {'version': VERSION, 'layout': layout, 'source_sha256': source_identity(),
            'dense_attention_max_tokens': 1024, 'warm_start': warm,
            'source_and_reference_tokens_dropped': 0,
            'dropout_and_kernel_rounding_may_differ': True,
            'old_cursor_order_preserved': layout == 'windows'}
    signature = identity(body)
    if saved and saved['signature'] != signature:
        raise ValueError('Saved execution identity is inconsistent')
    profile = {**body, 'signature': signature}
    atomic_json(root / PROFILE_FILE, profile)
    os.environ['CS781_P6_FAST_PROFILE'] = str(root / PROFILE_FILE)
    print('Optimization layout:', layout, '| legacy optimizer/cursor retained:' , bool(prior and layout == 'windows'), flush=True)
    if warm:
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''        print('Explicit task-weight warm start:', warm['checkpoint'], '| NEW optimizer and epoch plan.', flush=True)
    return profile


def validate_warm_start(job, profile):
    warm = profile.get('warm_start')
    if not warm or job['resume_checkpoint']:
        return
    old = read_json(Path(warm['run_dir']) / 'job.json')
    if (old['parent']['checkpoint_sha256'] != job['parent']['checkpoint_sha256'] or
            old['parent']['tokenizer_sha256'] != job['parent']['tokenizer_sha256'] or
            old['prepared_identity'] != job['prepared_identity'] or
            sha256_file(warm['checkpoint']) != warm['sha256']):
        raise ValueError('Warm-start foundation/tokenizer/data/checkpoint differs')
    dimensions = ('family', 'method', 'rank', 'alpha', 'adapter_width', 'prompt_tokens',
                  'memory_slots', 'tokens_per_chunk', 'chunk_tokens', 'target_tokens', 'target_context', 'kd_weight')
    if any(old['config'][key] != job['config'][key] for key in dimensions):
        raise ValueError('Warm-start architecture or target formulation differs')
    job['warm_start'] = warm
    atomic_json(Path(job['config']['run_dir']) / 'warm_start_receipt.json',
        {**warm, 'optimizer_reset': True, 'prior_partial_epoch_counted_as_complete': False,
         'new_training_covers_all_documents_and_target_tokens': True})
    atomic_json(Path(job['config']['run_dir']) / 'job.json', job)


def install(profile):
    """Install the same checked execution code in parent and mp.spawn children."""
    if profile['source_sha256'] != source_identity():
        raise ValueError('Optimized source changed before this worker started')
    from . import kernels
    from .data import SummaryDocuments, collate
    from summ import model, train, jobs, workflow
    if getattr(model, '_p6_fast_installed', False):
        return
    model._p6_fast_installed = True
    original_build = model.build_model
    def build(job, cfg, device, load_parent=True):
        actual = original_build(job, cfg, device, load_parent)
        if cfg.family == 'd':
            kernels.enable_sparse_fast_path(actual.backbone, profile['dense_attention_max_tokens'])
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''        if load_parent and job.get('warm_start'):
            warm = job['warm_start']
            if sha256_file(warm['checkpoint']) != warm['sha256']:
                raise ValueError('Warm-start weights changed')
            payload = torch.load(warm['checkpoint'], map_location='cpu', weights_only=True, mmap=True)
            actual.load_state_dict(payload['model'], strict=True)
        return actual
    model.build_model = train.build_model = build
    # These functions are methods at execution time; parameter registration and
    # the original source snapshot stay byte-identical for checkpoint compatibility.
    model.HierarchicalSummarizer.encode_document = kernels.encode_document
    model.HierarchicalSummarizer.decoder_hidden = kernels.decoder_hidden
    model.HierarchicalSummarizer.forward = kernels.forward
    train.collate_one = jobs.collate_one = collate
    if profile['layout'] == 'documents':
        train.SummaryWindows = jobs.SummaryWindows = SummaryDocuments

    original_prepare = jobs.prepare_job
    if profile['layout'] == 'documents':
        # A document cursor must also be rejected by the ORIGINAL notebooks,
        # whose source has no knowledge of this executor or its extra metadata.
        original_identity = jobs.identity
        jobs.identity = lambda value: identity({'base': original_identity(value),
                                                'execution_optimization': profile['signature']})
    def prepare(cfg, catalog):
        job = original_prepare(cfg, catalog)
        validate_warm_start(job, profile)
        dataset = train.SummaryWindows(job['cache'], cfg)
        cache = job['cache']
        source_tokens = sum(row['source_length'] for row in cache['cases']['train'])
        gold_tokens = sum(row['target_length'] for row in cache['cases']['train'])
        from summ.data import SummaryWindows
        windows = SummaryWindows(cache, cfg)
        repeated = sum(cache['cases']['train'][case]['source_length'] for case, _, _, _ in windows.specs)
        atomic_json(Path(cfg.run_dir) / 'optimization_report.json', {
            'profile': profile['signature'], 'layout': profile['layout'], 'training_units': len(dataset),
            'documents': len(cache['cases']['train']), 'target_windows': len(windows),
            'source_tokens_per_document_pass': source_tokens, 'gold_target_tokens_per_epoch': gold_tokens,
            'teacher_target_tokens_per_epoch': sum(row.get('teacher_length', 0) for row in cache['cases']['train']),
            'legacy_source_token_visits_per_epoch': repeated,
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''            'optimized_source_token_visits_per_epoch': source_tokens if profile['layout'] == 'documents' else repeated,
            'source_work_reduction_ratio': repeated / max(1, source_tokens) if profile['layout'] == 'documents' else 1.,
            'expected_wall_clock_speedup': None, 'all_reference_tokens_retained': True,
            'optimizer_plan_changed': profile['layout'] == 'documents'})
        return job
    jobs.prepare_job = workflow.prepare_job = prepare

    @torch.no_grad()
    def validation(rank, actual, job, cfg, device):
        dataset = train.SummaryWindows(job['cache'], cfg, 'tune')
        actual.eval()
        values = torch.zeros(2, device=device, dtype=torch.float64)
        for index in range(rank, len(dataset), cfg.world_size):
            row = dataset[index]
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                loss = actual(**collate(row, cfg.chunk_tokens, device))
            values[0] += loss.double()
            arrays = row['labels'] if isinstance(row['labels'], list) else [row['labels']]
            values[1] += sum(int((array != -100).sum()) for array in arrays)
        dist.all_reduce(values)
        total, count = values.cpu().tolist()
        if count <= 0 or not __import__('math').isfinite(total):
            raise ValueError('Invalid complete tune NLL')
        return {'tune_nll': total / count, 'tune_target_tokens': int(count)}
    train.validation = validation

    original_training = workflow.run_training
    def run_training(command):
        command = [sys.executable, '-u', '-m', 'p6_fast.launch', 'train', *command[4:]]
        return original_training(command)
    workflow.run_training = run_training

    original_commit, original_load = Checkpoints.commit, Checkpoints.load
    started = time.monotonic()
    def commit(manager, payload, best=False):
        payload['execution_optimization'] = profile
        payload['training_unit'] = 'document' if profile['layout'] == 'documents' else 'target_window'
        if profile['layout'] == 'documents':
            report_path = manager.directory.parent / 'optimization_report.json'
            report = read_json(report_path) if report_path.is_file() else None
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('control.py', []).append(r'''            for row in payload['cursor']['history']:
                if 'all_supervised_windows' in row:
                    row['all_supervised_documents'] = row.pop('all_supervised_windows')
                    row['all_supervised_target_windows'] = report['target_windows'] if report else None
        pair = original_commit(manager, payload, best)
        atomic_json(manager.directory.parent / 'execution_session.json', {
            'elapsed_seconds_at_commit': time.monotonic() - started, 'global_step': payload['step'],
            'epoch': payload['epoch'], 'next_batch': payload['cursor']['next_batch'], 'layout': profile['layout'],
            'peak_gpu_allocated_bytes_rank0': torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
            'model_optimizer_scaler_scheduler_cursor_and_rank_rng_saved': True})
        return pair
    def load(manager, candidate):
        payload = original_load(manager, candidate)
        previous = payload.get('execution_optimization')
        if previous and previous['signature'] != profile['signature']:
            raise ValueError('Checkpoint uses another execution/update plan. Resume its exact optimized notebook.')
        if not previous and profile['layout'] != 'windows':
            raise ValueError('Legacy window checkpoint cannot resume a document cursor. Use an explicit NEW warm-start run.')
        return payload
    Checkpoints.commit, Checkpoints.load = commit, load
    # Modules imported before installation hold function aliases.
    from summ import predict
    predict.build_model = build

''')


[Markdown Cell]

`p6_fast/data.py` — checked execution optimization.

In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('data.py', []).append(r'''"""Document batches retain every original target window, including KD targets."""
import math
import numpy as np
import torch
from summ.data import SummaryWindows


class SummaryDocuments(SummaryWindows):
    """One unit per document, with all gold/pseudo teacher-forced windows.

    The source is encoded once inside one differentiable forward. Changing from
    window units to document units changes the optimizer update plan; it is only
    allowed for a new run, never by interpreting an old mid-epoch cursor as docs.
    """
    def __init__(self, cache, cfg, split='train'):
        super().__init__(cache, cfg, split)
        self.window_specs = self.specs
        self.groups = [[] for _ in cache['cases'][split]]
        for spec in self.window_specs:
            self.groups[spec[0]].append(spec)
        if any(not group for group in self.groups):
            raise ValueError('Every supervised document must have real target tokens')
        self.specs = [(i, 'document', 0, sum(stop - start for _, _, start, stop in group))
                      for i, group in enumerate(self.groups)]

    def __getitem__(self, index):
        if index < 0:
            return {'source': np.asarray([1, 2]), 'decoder': [np.asarray([1])],
                    'labels': [np.asarray([-100])], 'loss_weights': [0.], 'weight': 1.}
        decoders, labels, weights = [], [], []
        for case, kind, start, stop in self.groups[index]:
            row = self.cache['cases'][self.split][case]
            array = self.teacher if kind == 'teacher' else self.target
            offset = row['teacher_offset'] if kind == 'teacher' else row['target_offset']
            context = max(0, start - self.cfg.target_context)
            decoders.append(np.concatenate(([1], np.asarray(array[offset + context:offset + stop - 1], np.int64))))
            labels.append(np.concatenate((np.full(start - context, -100), np.asarray(array[offset + start:offset + stop], np.int64))))
            weights.append(self.cfg.kd_weight if kind == 'teacher' else
                           1 - self.cfg.kd_weight if self.teacher is not None else 1.)
        return {'source': self.document(index), 'decoder': decoders, 'labels': labels,
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('data.py', []).append(r'''                'loss_weights': weights, 'weight': 1.}

    def target_weight(self, index):
        if index < 0:
            return 0.
        return sum((stop - start) * (self.cfg.kd_weight if kind == 'teacher' else
                   1 - self.cfg.kd_weight if self.teacher is not None else 1.)
                   for _, kind, start, stop in self.groups[index])


def collate(row, chunk_tokens, device=None):
    """CPU contiguous padding -> optional pinned, nonblocking CUDA transfers."""
    source = torch.from_numpy(np.asarray(row['source'], dtype=np.int64).copy())
    length = len(source)
    chunks = math.ceil(length / chunk_tokens)
    ids = torch.zeros(chunks * chunk_tokens, dtype=torch.long)
    ids[:length] = source
    mask = torch.arange(chunks * chunk_tokens) < length
    many = isinstance(row['decoder'], list)
    decoders = row['decoder'] if many else [row['decoder']]
    labels = row['labels'] if many else [row['labels']]
    def move(value):
        if device is not None and torch.device(device).type == 'cuda':
            return value.pin_memory().to(device, non_blocking=True)
        return value.to(device) if device is not None else value
    result = {'source_ids': move(ids.reshape(chunks, chunk_tokens)),
              'source_mask': move(mask.reshape(chunks, chunk_tokens)),
              'decoder_ids': [move(torch.from_numpy(np.asarray(value, np.int64).copy())[None]) for value in decoders],
              'labels': [move(torch.from_numpy(np.asarray(value, np.int64).copy())[None]) for value in labels]}
    if many:
        result['loss_weights'] = row['loss_weights']
    else:
        result['decoder_ids'], result['labels'] = result['decoder_ids'][0], result['labels'][0]
    return result
''')


[Markdown Cell]

`p6_fast/kernels.py` — checked execution optimization.

In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('kernels.py', []).append(r'''"""Exact sparse routes with bounded dense SDPA, batched pooling and shared sources."""
import importlib
import math
import types
import torch
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


def enable_sparse_fast_path(backbone, limit=1024):
    """Replace execution only; preserve parameters, state keys and legal key sets.

    At short summarization lengths, a [T,T] boolean mask is smaller than gathered
    [B,H,Q,K,Dh] K/V copies. Long sequences and cached single-token queries retain
    the original sparse implementation. Duplicate routes are collapsed exactly
    as the foundation's sparse_key_indices already requires.
    """
    count = 0
    for attention in backbone.modules():
        if type(attention).__name__ != 'SparseCausalSelfAttention' or hasattr(attention, '_fast_original_attend'):
            continue
        route = importlib.import_module(type(attention).__module__).sparse_key_indices
        attention._fast_original_attend = attention.attend
        attention._fast_masks = {}
        def attend(self, query, kv, positions, key_valid=None, route_function=route):
            length = kv[0].shape[2]
            if length > limit or query.shape[2] != length or len(positions) != length:
                return self._fast_original_attend(query, kv, positions, key_valid)
            signature = (query.device, length)
            mask = self._fast_masks.get(signature)
            if mask is None:
                # Full-length calls in this foundation always have positions 0..T-1.
                indices, valid = route_function(torch.arange(length, device=query.device), length,
                    self.local_window, self.dilation, self.dilated_links, self.global_stride)
                counts = torch.zeros(length, length, dtype=torch.int32, device=query.device)
                counts.scatter_add_(1, indices, valid.to(torch.int32))
                mask = (counts > 0)[None, None]
                if len(self._fast_masks) >= 2:
                    self._fast_masks.clear()
                self._fast_masks[signature] = mask
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('kernels.py', []).append(r'''            allowed = mask if key_valid is None else mask & key_valid[:, None, None, :]
            result = F.scaled_dot_product_attention(query, kv[0], kv[1], attn_mask=allowed,
                is_causal=False, dropout_p=self.dropout if self.training else 0.)
            return self.out(result.transpose(1, 2).contiguous().flatten(2))
        attention.attend = types.MethodType(attend, attention)
        count += 1
    return count


def encode_chunk(self, ids, mask, lengths):
    """Pool whole source chunks in a batch, with a separately pooled final tail."""
    full = all(length == ids.shape[1] for length in lengths)
    if self.cfg.family == 'b':
        if self.cfg.method == 'prompt':
            prefix = self.backbone.soft_encoder_prompt[None].expand(ids.shape[0], -1, -1)
            hidden = torch.cat((prefix, self.embedding()(ids)), dim=1)
            hidden = self.backbone.dropout(hidden + self.backbone.source_positions(torch.arange(hidden.shape[1], device=ids.device))[None])
            valid = torch.cat((torch.ones(ids.shape[0], prefix.shape[1], dtype=torch.bool, device=ids.device), mask), dim=1)
            for layer in self.backbone.encoder:
                hidden = layer(hidden, valid[:, None, None, :])
            hidden = self.backbone.encoder_norm(hidden)[:, prefix.shape[1]:]
        else:
            hidden = self.backbone.encode(ids, mask)
    else:
        # All-true source chunks need no pad mask or repeated CUDA mask assertions.
        hidden = self.backbone.hidden_states(ids, None if full else mask)
    output = []
    first = 0
    while first < len(lengths):
        length, stop = lengths[first], first + 1
        while stop < len(lengths) and lengths[stop] == length:
            stop += 1
        pooled = F.adaptive_avg_pool1d(hidden[first:stop, :length].float().transpose(1, 2),
                                      min(length, self.cfg.tokens_per_chunk))
        output.append(pooled.transpose(1, 2).reshape(-1, hidden.shape[-1]))
        first = stop
    return torch.cat(output)


def encode_document(self, source_ids, source_mask):
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('kernels.py', []).append(r'''    """One CPU length transfer per document; checkpoint every source microbatch."""
    lengths = source_mask.sum(-1).cpu().tolist()
    if any(length < 1 for length in lengths):
        raise ValueError('Empty source chunk')
    values = []
    for first in range(0, len(lengths), self.cfg.chunk_micro_batch):
        stop = first + self.cfg.chunk_micro_batch
        args = (source_ids[first:stop], source_mask[first:stop], tuple(lengths[first:stop]))
        values.append(checkpoint(encode_chunk_bound(self), *args, use_reentrant=False)
                      if self.training and self.cfg.checkpoint_encoder else encode_chunk(self, *args))
    memory = torch.cat(values)
    position = torch.linspace(0, 1, memory.shape[0], device=memory.device)
    positions = torch.stack((position, 1 - position, torch.ones_like(position)), dim=-1)
    memory = self.memory_norm(self.memory_projection(memory) + self.memory_positions(positions))
    importance = F.softplus(self.memory_salience(memory)).clamp_min(1e-4)
    slots = min(len(memory), self.cfg.memory_slots)
    numerator = F.adaptive_avg_pool1d((memory * importance).T[None], slots)
    denominator = F.adaptive_avg_pool1d(importance.T[None], slots)
    return (numerator / denominator).transpose(1, 2) * torch.sigmoid(self.memory_scale)


def encode_chunk_bound(model):
    return lambda ids, mask, lengths: encode_chunk(model, ids, mask, lengths)


def decoder_hidden(self, decoder_ids, memory):
    """No nested layer checkpoints: the enclosing target-window loss is checkpointed."""
    backbone = self.backbone
    if self.cfg.family == 'b':
        hidden = self.embedding()(decoder_ids)
        prefix = backbone.soft_decoder_prompt[None] if self.cfg.method == 'prompt' else None
        if prefix is not None:
            hidden = torch.cat((prefix, hidden), dim=1)
        hidden = backbone.dropout(hidden + backbone.target_positions(torch.arange(hidden.shape[1], device=hidden.device))[None])
        allowed = torch.ones(hidden.shape[1], hidden.shape[1], dtype=torch.bool, device=hidden.device).tril()[None, None]
        cross = torch.ones(1, 1, 1, memory.shape[1], dtype=torch.bool, device=hidden.device)
        for layer in backbone.decoder:
            hidden = layer(hidden, memory, allowed, cross)
        return backbone.decoder_norm(hidden)[:, -decoder_ids.shape[1]:]
    hidden = torch.cat((memory.to(self.embedding().weight.dtype), self.embedding()(decoder_ids)), dim=1)
''')


In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('kernels.py', []).append(r'''    for layer in (backbone.stages if self.cfg.family == 'c' else backbone.layers):
        hidden = layer(hidden, None)
    return backbone.final_norm(hidden)[:, -decoder_ids.shape[1]:]


def window_loss(self, decoder_ids, labels, memory):
    hidden = decoder_hidden(self, decoder_ids, memory)
    # Context participates in attention, but needs no expensive vocabulary logits.
    selected = labels != -100
    values, targets = hidden[selected], labels[selected]
    loss = hidden.sum() * 0.
    weight = self.embedding().weight
    for first in range(0, len(targets), 256):
        logits = F.linear(values[first:first + 256].float(), weight.float())
        loss = loss + F.cross_entropy(logits.float(), targets[first:first + 256], reduction='sum',
            label_smoothing=self.cfg.label_smoothing if self.training else 0.)
    return loss


def forward(self, source_ids, source_mask, decoder_ids, labels, loss_weights=None):
    """All target windows share one differentiable full-document representation."""
    memory = encode_document(self, source_ids, source_mask)
    decoders = decoder_ids if isinstance(decoder_ids, (list, tuple)) else [decoder_ids]
    targets = labels if isinstance(labels, (list, tuple)) else [labels]
    weights = loss_weights if loss_weights is not None else [1.] * len(decoders)
    if not (len(decoders) == len(targets) == len(weights)) or not decoders:
        raise ValueError('Mismatched or empty target-window lists')
    loss = memory.sum() * 0.
    for ids, actual, scale in zip(decoders, targets, weights):
        # One decoder graph at a time is reconstructed in backward. The source
        # graph is shared, so its VJP executes once after all window contributions.
        function = lambda ids, actual, memory: window_loss(self, ids, actual, memory)
        part = checkpoint(function, ids, actual, memory, use_reentrant=False) if self.training and self.cfg.checkpoint_encoder else function(ids, actual, memory)
        loss = loss + part * scale
    return loss
''')


[Markdown Cell]

`p6_fast/launch.py` — checked execution optimization.

In [ ]:
# [Code Cell]
FAST_PARTS.setdefault('launch.py', []).append(r'''"""Importable launcher: identical installation in NCCL mp.spawn children."""
import argparse
import os
from pathlib import Path
import sys
from summ.io_utils import read_json
from .control import bind, install


def inherited_install():
    path = os.environ.get('CS781_P6_FAST_PROFILE')
    if path:
        install(read_json(path))


inherited_install()  # also executed as __mp_main__ by torch.multiprocessing.spawn


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('workflow', 'train', 'experiments'))
    parser.add_argument('--optimization')
    args, remainder = parser.parse_known_args()
    if args.action != 'train':
        if '--plan' not in remainder or not args.optimization:
            parser.error('--plan and --optimization are required')
        plan = remainder[remainder.index('--plan') + 1]
        profile = bind(plan, args.optimization)
        install(profile)
    elif not os.environ.get('CS781_P6_FAST_PROFILE'):
        parser.error('Training must be launched by the checked optimized workflow')
    from summ import workflow, train, experiments
    sys.argv = ['p6_fast.' + args.action, *remainder]
    {'workflow': workflow.main, 'train': train.main, 'experiments': experiments.main}[args.action]()


if __name__ == '__main__':
    main()
''')


In [ ]:
# [Code Cell]
FAST_SOURCE_SHA256 = {'__init__.py': 'c7610b889dff2da020816b80de7c8eea7507c4b0c66fdc917eed2e25f9225760', 'control.py': '8da45f4ebb2f34fe2817155eac6db3ae7a61308867f34fa9097d89c97485dae6', 'data.py': '6c96fb5f3eea1c3f0246567aa997919bcbed0c7666096d930d1a76b46c9db31a', 'kernels.py': '1f90309eae1e3de1935ddcf04dabfa6a3a5c9d8f00f9197d251d768770f532fa', 'launch.py': '1dc0c058c6d77876fcaf7bad8ca78aeb68ec0b414f06f2e2f34089f961a9e8cb'}
fast_directory = code_directory / 'p6_fast'
fast_directory.mkdir(exist_ok=True)
for name, parts in FAST_PARTS.items():
    text = ''.join(parts)
    assert hashlib.sha256(text.encode()).hexdigest() == FAST_SOURCE_SHA256[name], name
    compile(text, name, 'exec')
    file = fast_directory / name
    if (Path(OUTPUT_ROOT) / 'execution_optimization.json').is_file() and file.is_file() and file.read_bytes() != text.encode():
        raise ValueError('Saved optimized source changed. Use its exact optimized notebook.')
    file.write_bytes(text.encode())
optimization_path = Path(OUTPUT_ROOT) / 'optimization_options.json'
optimization_path.write_text(json.dumps({'training_layout': TRAINING_LAYOUT,
    'warm_start_run_dir': WARM_START_RUN_DIR}, indent=2), encoding='utf-8')
print('Verified optimized executor. Original summarizer and foundation sources are preserved.')


[Markdown Cell]

### Importable automatic teacher orchestration
The added package is outside the hashed summarizer and optimized executor. Their source snapshots
remain byte-identical for full student recovery. Teacher inference runs in a separate process,
so its execution profile cannot overwrite the student's document/window optimizer plan.

In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS = {}


[Markdown Cell]

`p6_auto_teacher/__init__.py`

In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('__init__.py', []).append(r'''"""Direct own Phase-6 teacher input for the unchanged optimized B KD notebook."""
VERSION = 'p6-inline-teacher-v1'
''')


[Markdown Cell]

`p6_auto_teacher/generate.py`

In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('generate.py', []).append(r'''"""Teacher inference entry point; optimized kernels also install in spawn workers."""
import argparse
import json
import os
from summ.io_utils import read_json

if os.environ.get('CS781_P6_FAST_PROFILE'):
    from p6_fast.control import install
    install(read_json(os.environ['CS781_P6_FAST_PROFILE']))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    from summ.jobs import hardware
    from summ.predict import export
    hardware()
    print(json.dumps(export(args.job, teacher_only=True), indent=2), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`p6_auto_teacher/launch.py`

In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('launch.py', []).append(r'''"""One launch: own completed teacher -> all Train targets -> original optimized KD."""
import argparse
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
from summ.config import SummConfig
from summ.io_utils import atomic_json, read_json, publish_copy
from summ.preparation import prepared_input
from .teacher import reuse_targets, find_teacher, stage_teacher, valid_export, student_foundation_count


def child(command, deadline, profile=None):
    """Forward one stop request; allow durable generation/checkpoint cleanup."""
    env = os.environ.copy()
    env.pop('CS781_P6_FAST_PROFILE', None)
    if profile:
        env['CS781_P6_FAST_PROFILE'] = str(profile)
    process = subprocess.Popen(command, env=env)
    handlers, interrupted = {}, [False]
    def request_stop(signum, frame):
        if not interrupted[0] and process.poll() is None:
            interrupted[0] = True
            process.send_signal(signal.SIGINT)
    try:
        for signum in (signal.SIGINT, signal.SIGTERM):
            handlers[signum] = signal.signal(signum, request_stop)
        while process.poll() is None:
            if time.time() >= deadline:
                request_stop(signal.SIGINT, None)
            time.sleep(.25)
        if interrupted[0]:
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('launch.py', []).append(r'''            return False
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, command)
        return True
    finally:
        for signum, handler in handlers.items():
            signal.signal(signum, handler)


def paused(root, stage):
    report = {'version': 'summ-output-v1', 'status': 'paused_recoverable', 'stage': stage,
        'next_action': 'Save COMPLETE output, attach it to this SAME notebook and resume. '
        'If B job.json does not yet exist, point TEACHER_RUN_DIR to the saved auto_teacher/ job root '
        '(or keep the original teacher attached), and leave RESUME_RUN_DIR unset. '
        'After B job.json/checkpoints exist, set RESUME_RUN_DIR to the saved B run root. '
        'Committed teacher document rows / student optimizer steps are retained.'}
    atomic_json(Path(root) / 'final_report.json', report)
    atomic_json(Path(root) / 'auto_teacher_status.json', report)
    return report


def execute(plan_path, catalog_path, optimization_path, options_path):
    plan, options = read_json(plan_path), read_json(options_path)
    cfg = SummConfig(**plan['config'])
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    deadline = time.time() + cfg.max_run_seconds
    stage = 'teacher_preflight'
    try:
        if plan['mode'] == 'train' and cfg.method == 'kd':
            prepared, manifest = prepared_input(cfg)
            receipt = reuse_targets(cfg, prepared, manifest['identity'],
                include_attached_exports=options.get('teacher_checkpoint', 'auto') == 'auto')
            if receipt is None:
                origin = find_teacher(cfg)
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('launch.py', []).append(r'''                student_parameters = student_foundation_count(cfg, read_json(catalog_path))
                original_job = read_json(origin / 'job.json')
                if original_job['parent_parameter_count'] <= student_parameters:
                    raise ValueError('The selected teacher is not larger than your B student. Choose a larger completed own A/C/D summarizer.')
                print('Direct teacher input:', origin, flush=True)
                print('Stage trained weights for inference only; no teacher training/export notebook is required.', flush=True)
                stage = 'teacher_staging'
                job = stage_teacher(origin, root / 'auto_teacher', prepared, manifest,
                                    options.get('teacher_checkpoint', 'auto'))
                # Record explicitly that pseudo-targets come from generation,
                # not gold reference summaries renamed as teacher predictions.
                atomic_json(root / 'auto_teacher_status.json', {'status': 'generating_train_targets',
                    'teacher_source': str(origin), 'training_documents': len(job['cache']['cases']['train']),
                    'teacher_parameters': job['parent_parameter_count'], 'teacher_training_performed': False,
                    'test_labels_used': False})
                profile = root / 'auto_teacher/execution_optimization.json'
                use_profile = None
                if profile.is_file():
                    from p6_fast.control import source_identity
                    if read_json(profile).get('source_sha256') == source_identity():
                        use_profile = profile
                    else:
                        print('Teacher executor version differs; using the verified original inference implementation.', flush=True)
                teacher_deadline = deadline
                if options.get('teacher_max_minutes') is not None:
                    minutes = float(options['teacher_max_minutes'])
                    if minutes <= 0:
                        raise ValueError('AUTO_TEACHER_MAX_MINUTES must be positive or None')
                    teacher_deadline = min(deadline, time.time() + minutes * 60)
                stage = 'teacher_generation'
                print('Generate every Train document on both GPUs; completed document journals resume automatically.', flush=True)
                success = child([sys.executable, '-u', '-m', 'p6_auto_teacher.generate',
                    '--job', str(root / 'auto_teacher/job.json')], teacher_deadline, use_profile)
                if not success:
                    return paused(root, stage)
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('launch.py', []).append(r'''                receipt = valid_export(root / 'auto_teacher', prepared, manifest['identity'])
                for name in ('teacher_export_receipt.json', 'train_summaries.jsonl'):
                    publish_copy(root / 'auto_teacher' / name, root / name)
            else:
                print('Reuse the fixed verified teacher-target snapshot; teacher inference is skipped. '
                    'Checkpoint selection applies only to a new target snapshot.', flush=True)
            # Transport-only setting: the target checksum still defines the
            # original student loss/cache/recovery fingerprint.
            plan['config']['teacher_run_dir'] = str(root)
            atomic_json(root / 'auto_teacher_status.json', {'status': 'teacher_targets_ready',
                'training_cases': receipt['training_cases'], 'teacher_sha256': receipt['sha256'],
                'teacher_checkpoint_sha256': receipt['teacher_checkpoint_sha256'],
                'teacher_training_performed': False, 'test_labels_used': False})
        remaining = int(deadline - time.time())
        if remaining < 1:
            return paused(root, 'before_student_training')
        plan['config']['max_run_seconds'] = remaining
        atomic_json(plan_path, plan)
        stage = 'student_training_or_export'
        print('Launch the original optimized student with its unchanged model and full-state recovery.', flush=True)
        success = child([sys.executable, '-u', '-m', 'p6_fast.launch', 'workflow',
            '--optimization', str(optimization_path), '--plan', str(plan_path), '--catalog', str(catalog_path)], deadline)
        if not success:
            return paused(root, stage)
        return read_json(root / 'final_report.json')
    except KeyboardInterrupt:
        return paused(root, stage)
    except Exception as error:
        import traceback
        atomic_json(root / 'auto_teacher_failure_report.json', {'status': 'failed', 'stage': stage,
            'type': type(error).__name__, 'message': str(error), 'traceback': traceback.format_exc(),
            'next_action': 'Keep complete outputs. Correct the teacher input or reported incompatibility; no fake teacher targets are produced.'})
        raise


''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('launch.py', []).append(r'''def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('workflow',))
    parser.add_argument('--optimization', required=True)
    parser.add_argument('--auto-teacher', required=True)
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    args = parser.parse_args()
    from summ.io_utils import run_lock
    root = Path(read_json(args.plan)['config']['run_dir'])
    with run_lock(root / 'auto_teacher_ownership'):
        result = execute(args.plan, args.catalog, args.optimization, args.auto_teacher)
    print(json.dumps(result, indent=2), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

`p6_auto_teacher/teacher.py`

In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''"""Verify/stage a completed own summarizer and its resumable training-only targets."""
from dataclasses import replace
from pathlib import Path
import shutil
import torch
from summ import VERSION
from summ.config import SummConfig
from summ.io_utils import atomic_json, identity, read_json, records, sha256_file, publish_copy
from summ.parents import source_hashes, parent_class
from summ.data import cache_data
from summ.state import Checkpoints


def student_origins(cfg):
    """Rebase automatically discovered KD outputs after Kaggle changes input mounts."""
    if cfg.resume_run_dir:
        return [Path(cfg.resume_run_dir)]
    roots, fingerprints = [], set()
    for file in Path(cfg.input_root).rglob('job.json'):
        value = read_json(file)
        if value.get('version') == VERSION and value.get('config', {}).get('experiment_id') == cfg.experiment_id:
            roots.append(file.parent)
            fingerprints.add(value['fingerprint'])
    if len(fingerprints) > 1:
        raise ValueError('Conflicting saved KD versions. Set RESUME_RUN_DIR to the exact desired run.')
    return sorted(set(roots))


def valid_export(root, prepared, prepared_identity):
    """Accept exactly all Train IDs and verified nonempty generated summaries."""
    root = Path(root)
    receipt = read_json(root / 'teacher_export_receipt.json')
    path = root / 'train_summaries.jsonl'
    if (receipt.get('version') != 'summ-teacher-v1' or receipt.get('status') != 'complete'
            or receipt.get('teacher_is_own_phase6_model') is not True
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''            or receipt.get('test_labels_used') is not False
            or receipt.get('prepared_identity') != prepared_identity
            or receipt.get('sha256') != sha256_file(path)):
        raise ValueError('Teacher targets are incomplete, corrupted or from another Phase 6 split')
    expected = [row['id'] for row in records(Path(prepared) / 'train.jsonl')]
    actual = list(records(path))
    if ([row['id'] for row in actual] != expected
            or any(not isinstance(row.get('summary'), str) or not row['summary'].strip() for row in actual)):
        raise ValueError('Teacher targets must cover every original Train ID, in order, with a real generated summary')
    if receipt.get('training_cases') != len(expected) or not isinstance(receipt.get('teacher_parameters'), int):
        raise ValueError('Teacher target count/parameter receipt is missing or inconsistent')
    return receipt


def reuse_targets(cfg, prepared, prepared_identity, include_attached_exports=True):
    """Prefer the already fixed target snapshot when continuing the same KD run."""
    root = Path(cfg.run_dir)
    paths = [root]
    paths += student_origins(cfg)
    if cfg.teacher_run_dir and include_attached_exports:
        paths.append(Path(cfg.teacher_run_dir))
    elif not cfg.teacher_run_dir and include_attached_exports:
        paths += [p.parent for p in Path(cfg.input_root).rglob('teacher_export_receipt.json')]
    accepted, rejected = [], []
    for path in dict.fromkeys(paths):
        if not (path / 'teacher_export_receipt.json').is_file():
            continue
        try:
            receipt = valid_export(path, prepared, prepared_identity)
            accepted.append((path, receipt))
        except (OSError, KeyError, ValueError) as error:
            rejected.append({'root': str(path), 'reason': str(error)})
    if len({receipt['sha256'] for _, receipt in accepted}) > 1:
        raise ValueError('Several teacher target snapshots differ. Keep the original KD snapshot or select one teacher explicitly.')
    if accepted:
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''        origin, receipt = accepted[0]
        for name in ('train_summaries.jsonl', 'teacher_export_receipt.json'):
            publish_copy(origin / name, root / name)
        return receipt
    atomic_json(root / 'auto_teacher_target_search.json', {'rejected': rejected})
    return None


def find_teacher(cfg):
    """Explicit run root/ancestor, or one unambiguous completed A/C/D output."""
    old_students = student_origins(cfg)
    snapshots = [Path(cfg.run_dir), *old_students]
    saved_teacher_pointer = bool(cfg.teacher_run_dir and (
        Path(cfg.teacher_run_dir).resolve() == Path(cfg.run_dir).resolve()
        or any(read_json(old / 'job.json').get('config', {}).get('teacher_run_dir') == cfg.teacher_run_dir
               for old in old_students if (old / 'job.json').is_file())))
    if cfg.teacher_run_dir and not saved_teacher_pointer:
        supplied = Path(cfg.teacher_run_dir)
        if supplied.is_file():
            raise ValueError('TEACHER_RUN_DIR must be the saved Kaggle OUTPUT directory, not an .ipynb. '
                'Add the trained Phase 6 notebook through Add Input and point to its folder containing job.json/checkpoints/.')
        if not supplied.is_dir():
            raise FileNotFoundError('Teacher output directory does not exist: ' + str(supplied))
        paths = [supplied / 'job.json'] if (supplied / 'job.json').is_file() else list(supplied.rglob('job.json'))
    else:
        staged = [p / 'auto_teacher/job.json' for p in snapshots if (p / 'auto_teacher/job.json').is_file()]
        if staged:
            paths = staged[:1]
        else:
            paths = list(Path(cfg.input_root).rglob('job.json'))
    candidates = []
    for file in paths:
        job = read_json(file)
        if (job.get('version') == VERSION and job.get('config', {}).get('family') in ('a', 'c', 'd')
                and job['config'].get('method') != 'kd'):
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''            candidates.append(file.parent)
    if len(candidates) != 1:
        atomic_json(Path(cfg.run_dir) / 'auto_teacher_candidates.json', {'candidates': [str(p) for p in candidates]})
        raise FileNotFoundError('Attach one COMPLETED own Phase 6 A/C/D summarization output, or set TEACHER_RUN_DIR '
            'to its exact job.json root. A notebook alone, Phase 1/5 output, or same-size B is not a larger trained summarization teacher.')
    return candidates[0]


def foundation_count(job, source):
    Config, Model = parent_class(str(source), job['config']['family'], job['parent_contract'])
    with torch.device('meta'):
        model = Model(Config(**job['parent']['config']))
    count = sum(value.numel() for value in model.parameters())
    if not 50_000_000 < count < 300_000_000 or count != job['parent_parameter_count']:
        raise ValueError('Teacher foundation parameter count differs from its verified own-model contract')
    return count


def student_foundation_count(cfg, catalog):
    """Check teacher > student BEFORE costly teacher generation; never load GPU weights."""
    local = Path(cfg.run_dir) / 'job.json'
    prior = student_origins(cfg)
    saved = local if local.is_file() else prior[0] / 'job.json' if prior else None
    if saved and saved.is_file():
        job = read_json(saved)
        if job.get('version') == VERSION and job['config']['experiment_id'] == cfg.experiment_id:
            return job['parent_parameter_count']
    from summ.parents import verify_parent
    contract = catalog[cfg.family]
    parent_root, parent = verify_parent(cfg, contract)
    contract = parent.get('native_contract', contract)
    source = (parent_root / parent['native_contract']['source_subdir'] if parent.get('native_contract')
              else parent_root / 'code' / ('model_' + cfg.family))
    Config, Model = parent_class(str(source), cfg.family, contract)
    with torch.device('meta'):
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''        model = Model(Config(**parent['config']))
    count = sum(value.numel() for value in model.parameters())
    if not 50_000_000 < count < 300_000_000:
        raise ValueError('Student foundation must be within the original 50M–300M boundary')
    return count


def verify_fingerprint(job, profile=None):
    body = {'version': VERSION, 'choices': SummConfig(**job['config']).choices(),
        'data': job['prepared_identity'], 'foundation': job['parent']['checkpoint_sha256'],
        'tokenizer': job['parent']['tokenizer_sha256'], 'runtime': job['runtime_source_sha256'],
        'cache': job['cache']['identity'], 'torch': job['torch'],
        'teacher': job.get('teacher') and {k: v for k, v in job['teacher'].items() if k != 'path'}}
    expected = identity(body)
    if profile:
        if identity({k: v for k, v in profile.items() if k != 'signature'}) != profile['signature']:
            raise ValueError('Teacher execution profile signature is inconsistent')
        if profile['layout'] == 'documents':
            expected = identity({'base': expected, 'execution_optimization': profile['signature']})
    if expected != job['fingerprint']:
        raise ValueError('Teacher learning/source/checkpoint fingerprint is inconsistent')


def stage_teacher(origin, destination, prepared, manifest, checkpoint='auto'):
    """Inference-only relocation; never resume/retrain the teacher's optimizer.

    Its full latest state proves training completion; the chosen best/latest
    trained state supplies inference. Original learning fingerprints remain intact.
    """
    origin, root = Path(origin), Path(destination)
    root.mkdir(parents=True, exist_ok=True)
    origin_job = origin / 'teacher_origin_job.json'
    old = read_json(origin_job if origin_job.is_file() else origin / 'job.json')
    cfg = SummConfig(**old['config'])
    if old.get('version') != VERSION or cfg.family not in ('a', 'c', 'd'):
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''        raise ValueError('Choose a larger completed own Phase 6 A/C/D summarizer')
    if old['prepared_identity'] != manifest['identity']:
        raise ValueError('Teacher and student prepared splits differ. Attach the matching P6_00 output.')
    from summ import __file__ as runtime_file
    expected_sources = source_hashes(Path(runtime_file).parent)
    if old['runtime_source_sha256'] != expected_sources or source_hashes(origin / 'code/summ') != expected_sources:
        raise ValueError('Teacher summarizer source differs from this supported Phase 6 runtime')
    parent_source = origin / 'parent_source' / ('model_' + cfg.family)
    if source_hashes(parent_source) != old['parent_contract']['source_sha256']:
        raise ValueError('Teacher foundation source is incomplete or changed')
    tokenizer = origin / 'tokenizer/tokenizer.json'
    if sha256_file(tokenizer) != old['parent']['tokenizer_sha256']:
        raise ValueError('Teacher tokenizer does not match its own trained weights')
    profile_path = origin / 'execution_optimization.json'
    profile = read_json(profile_path) if profile_path.is_file() else None
    verify_fingerprint(old, profile)
    manager = Checkpoints(origin, old['fingerprint'], 0)
    latest, best = manager.latest(), manager.best()
    if latest is None or best is None:
        raise FileNotFoundError('Teacher needs real full checkpoint .pt files and matching .pt.json receipts. An .ipynb/observations ZIP is insufficient.')
    payload = manager.load(latest)
    if not payload['cursor']['complete']:
        raise ValueError('Teacher training is paused/incomplete. Finish it or attach a completed A/C/D teacher.')
    if SummConfig(**payload['config']).choices() != cfg.choices():
        raise ValueError('Teacher checkpoint and saved job learning choices differ')
    completed_cursor = payload['cursor']
    del payload
    count = foundation_count(old, parent_source)
    overrides, selection = {}, cfg.inference_checkpoint
    screened = origin / 'screen_report.json'
    if checkpoint == 'auto' and screened.is_file():
        screen = read_json(screened)
        if screen.get('status') == 'submission_verified':
            overrides = screen['generation']
            selection = screen.get('checkpoint_selection', 'best')
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''    elif checkpoint in ('best', 'latest'):
        selection = checkpoint
    elif checkpoint != 'auto':
        raise ValueError('TEACHER_CHECKPOINT must be auto, best or latest')
    chosen = best if selection == 'best' else latest
    binding = {'teacher_checkpoint_sha256': chosen[1]['sha256'], 'teacher_fingerprint': old['fingerprint'],
        'prepared_identity': manifest['identity'], 'tokenizer_sha256': old['parent']['tokenizer_sha256'],
        'selection': selection, 'generation': {**{k: getattr(cfg, k) for k in (
            'generation_max_tokens', 'generation_min_tokens', 'repetition_penalty', 'no_repeat_ngram', 'beam_size', 'length_penalty')}, **overrides}}
    binding_path = root / 'auto_teacher_binding.json'
    if binding_path.is_file() and read_json(binding_path) != binding:
        raise ValueError('Teacher/data/decoding changed during target generation. Resume the same teacher snapshot.')
    for directory in ('code/summ', 'tokenizer', 'parent_source/model_' + cfg.family):
        source, target = origin / directory, root / directory
        target.mkdir(parents=True, exist_ok=True)
        for file in source.glob('*'):
            if file.is_file() and file.resolve() != (target / file.name).resolve():
                shutil.copyfile(file, target / file.name)
    if (origin / 'screen_report.json').is_file():
        publish_copy(origin / 'screen_report.json', root / 'screen_report.json')
    if profile:
        publish_copy(profile_path, root / 'execution_optimization.json')
    target_manager = Checkpoints(root, old['fingerprint'], cfg.min_free_bytes)
    target_manager.adopt(best, best=True)
    target_manager.adopt(latest)
    # Durable per-document generation journals are rebased along with the job.
    if origin.resolve() != root.resolve() and (origin / 'generation').is_dir() and not (root / 'generation').exists():
        shutil.copytree(origin / 'generation', root / 'generation')
    staged_cfg = replace(cfg, run_dir=str(root), input_root=str(Path(prepared).parent), prepared_dir=str(prepared),
        resume_run_dir=str(origin), inference_checkpoint=selection, **overrides)
    cache = cache_data(prepared, root / 'tokenizer/tokenizer.json', staged_cfg)
    job = {**old, 'config': staged_cfg.__dict__, 'prepared_dir': str(prepared),
        'parent_source_dir': str(root / 'parent_source' / ('model_' + cfg.family)),
        'tokenizer_path': str(root / 'tokenizer/tokenizer.json'), 'cache': cache, 'resume_roots': [],
        'teacher_inference_only_staging': True, 'original_teacher_torch': old['torch']}
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_PARTS.setdefault('teacher.py', []).append(r'''    # Cache tokenization can use the installed tokenizer version for inference;
    # this staged job is never passed off as an optimizer-resumable teacher run.
    atomic_json(root / 'job.json', job)
    atomic_json(root / 'teacher_origin_job.json', old)
    atomic_json(root / 'training_summary.json', {'status': 'complete', **completed_cursor})
    atomic_json(root / 'auto_teacher_binding.json', binding)
    atomic_json(root / 'teacher_staging_report.json', {'status': 'ready_for_generation',
        'teacher_parameters': count, 'teacher_training_or_optimizer_updates_performed': False,
        'selected_checkpoint_sha256': chosen[1]['sha256'], 'source_snapshot_verified': True,
        'current_torch': str(torch.__version__), 'original_teacher_torch': old['torch']})
    return job
''')


In [ ]:
# [Code Cell]
AUTO_TEACHER_SOURCE_SHA256 = {'__init__.py': 'b324fcca507a19b9d42647779081353f9b353a67c40ca500d7f37d7ac2cac973', 'generate.py': 'ccdf5d24be4d84782a21cac189900f0bdcbffadeaf84f57fc616f47cef21c4d2', 'launch.py': 'e26977b95a85ccfbc2ea57f0730ee86c833a89e306a26678c13245956dd6d741', 'teacher.py': 'e9ce329c50744cb5615522cade8a8852097430e60092d1bf0dbd5e80da0e84b3'}
automatic_directory = code_directory / 'p6_auto_teacher'
automatic_directory.mkdir(exist_ok=True)
for name, pieces in AUTO_TEACHER_PARTS.items():
    source = ''.join(pieces)
    assert hashlib.sha256(source.encode()).hexdigest() == AUTO_TEACHER_SOURCE_SHA256[name], name
    compile(source, name, 'exec')
    (automatic_directory / name).write_bytes(source.encode())
auto_teacher_options_path = Path(OUTPUT_ROOT) / 'auto_teacher_options.json'
atomic_options = {'teacher_checkpoint': TEACHER_CHECKPOINT, 'teacher_max_minutes': AUTO_TEACHER_MAX_MINUTES}
auto_teacher_options_path.write_text(json.dumps(atomic_options, indent=2), encoding='utf-8')
print('Automatic teacher modules verified; separate export notebook is unnecessary.')


[Markdown Cell]

### T4 execution modules
These importable modules install in the supervisor, both NCCL workers and both inference
workers. Original checkpoint/data/model identities are kept; the extra execution profile
is recorded separately in t4_speed_profile.json and new full-state checkpoints.

For a new student the existing auto layout uses one full document per GPU per update.
For an old student the saved window/document layout and accumulation remain in force.
The notebook never treats a completed epoch as an unfinished one or restarts an optimizer
silently. Generated Train targets are reused once their full checksum/ID receipt verifies.

In [ ]:
# [Code Cell]
T4_PARTS = {}


[Markdown Cell]

`p6_t4/__init__.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('__init__.py', []).append(r'''"""Execution-only T4 x2 batching for the direct-teacher Phase 6 KD notebook."""
VERSION = 'p6-t4-batching-v1'
''')


[Markdown Cell]

`p6_t4/control.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''"""Execution-only installation; unchanged model keys, objective and update cursor."""
import os
from pathlib import Path
import sys
import torch
import torch.distributed as dist
from tqdm.auto import tqdm
from summ import workflow as workflow_module
from summ.io_utils import atomic_json, identity, read_json
from summ.parents import source_hashes
from . import VERSION

# Capture the original subprocess runner BEFORE p6_fast installs its launcher.
BASE_RUN_TRAINING = workflow_module.run_training
PROFILE_FILE = 't4_speed_profile.json'


def bind(plan_path, options_path):
    plan, options = read_json(plan_path), read_json(options_path)
    for key, upper in (('source_batch', 64), ('window_batch', 8),
                       ('generation_batch', 16), ('inference_chunks', 128)):
        value = options.get(key)
        if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= upper:
            raise ValueError(key + ' must be an integer from 1 to ' + str(upper))
    if not 5 <= options.get('tuning_seconds', 0) <= 120:
        raise ValueError('tuning_seconds must be between 5 and 120')
    body = {'version': VERSION, 'source_sha256': source_hashes(Path(__file__).parent),
            'options': options, 'changes_model_or_global_update_plan': False,
            'source_or_reference_tokens_dropped': 0,
            'batching_dropout_and_kernel_rounding_may_differ': True}
    result = {**body, 'signature': identity(body)}
    root = Path(plan['config']['run_dir'])
    origins = [root]
    if plan['config'].get('resume_run_dir'):
        origins.append(Path(plan['config']['resume_run_dir']))
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''    for origin in origins:
        saved = origin / PROFILE_FILE
        if saved.is_file() and read_json(saved)['signature'] != result['signature']:
            raise ValueError('Saved T4 execution settings differ. Keep the same optimized notebook/settings to continue this run.')
    atomic_json(root / PROFILE_FILE, result)
    os.environ['CS781_P6_T4_PROFILE'] = str(root / PROFILE_FILE)
    return result


def profile():
    path = os.environ.get('CS781_P6_T4_PROFILE')
    if not path:
        raise RuntimeError('Launch through the notebook so both GPU workers inherit the verified T4 settings')
    result = read_json(path)
    if result['version'] != VERSION or result['source_sha256'] != source_hashes(Path(__file__).parent):
        raise ValueError('T4 execution source differs from its saved profile')
    body = {key: value for key, value in result.items() if key != 'signature'}
    if result['signature'] != identity(body):
        raise ValueError('T4 speed profile is inconsistent')
    return result


def install():
    from . import kernels, inference
    from summ import model, train, jobs, predict, workflow
    from summ.state import Checkpoints
    if getattr(model, '_p6_t4_installed', False):
        return
    settings = profile()
    model._p6_t4_installed = True
    original_build, original_ddp = model.build_model, train.DDP

    def build(job, cfg, device, load_parent=True):
        actual = original_build(job, cfg, device, load_parent)
        if cfg.family == 'd':
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''            from p6_fast.kernels import enable_sparse_fast_path
            enable_sparse_fast_path(actual.backbone, 1024)
        # Inference fallback starts conservatively; packed inference has its own
        # larger microbatch limit with automatic OOM backoff.
        actual._p6_t4 = {'source_batch': min(cfg.chunk_micro_batch, settings['options']['source_batch']),
                         'window_batch': 1}
        return actual
    model.build_model = train.build_model = predict.build_model = build
    model.HierarchicalSummarizer.encode_document = kernels.encode_document
    model.HierarchicalSummarizer.forward = kernels.forward
    train.collate_one = jobs.collate_one = kernels.collate
    # p6_fast.validation closes over its own collate. forward accepts both its
    # existing arguments and the optional CPU-known source lengths.

    def ddp(actual, *args, **kwargs):
        report = kernels.tune(actual, settings['options'])
        rank = torch.distributed.get_rank()
        atomic_json(Path(actual.cfg.run_dir) / f't4_probe_gpu{rank}.json', report)
        if rank == 0:
            print('Measured source/target microbatches:', report['selected'], flush=True)
        return original_ddp(actual, *args, **kwargs)
    train.DDP = ddp

    def run_training(command):
        return BASE_RUN_TRAINING([sys.executable, '-u', '-m', 'p6_t4.launch', 'train', *command[4:]])
    workflow.run_training = run_training
    predict.parallel_generate = inference.parallel_generate

    @torch.no_grad()
    def validation(rank, actual, job, cfg, device):
        dataset = train.SummaryWindows(job['cache'], cfg, 'tune')
        actual.eval()
        totals = torch.zeros(2, device=device, dtype=torch.float64)
        for index in tqdm(range(rank, len(dataset), cfg.world_size), disable=rank != 0,
                          desc='Tune NLL — GPU 0 share', unit='documents'):
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''            row = dataset[index]
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                loss = actual(**kernels.collate(row, cfg.chunk_tokens, device))
            totals[0] += loss.double()
            arrays = row['labels'] if isinstance(row['labels'], list) else [row['labels']]
            totals[1] += sum(int((array != -100).sum()) for array in arrays)
        dist.all_reduce(totals)
        loss, tokens = totals.cpu().tolist()
        if tokens <= 0 or not __import__('math').isfinite(loss):
            raise ValueError('Invalid complete tune NLL')
        return {'tune_nll': loss / tokens, 'tune_target_tokens': int(tokens)}
    train.validation = validation

    original_commit, original_load = Checkpoints.commit, Checkpoints.load
    def commit(manager, payload, best=False):
        payload['t4_execution'] = settings
        return original_commit(manager, payload, best)
    def load(manager, candidate):
        payload = original_load(manager, candidate)
        previous = payload.get('t4_execution')
        if previous and previous['signature'] != settings['signature']:
            raise ValueError('Checkpoint used another T4 executor. Continue with its exact saved notebook/settings.')
        return payload
    Checkpoints.commit, Checkpoints.load = commit, load


def redirect_children():
    from p6_auto_teacher import launch
    original = launch.child
    def child(command, deadline, profile=None):
        role = None
        if len(command) > 3 and command[3] == 'p6_auto_teacher.generate':
            command = [*command[:3], 'p6_t4.launch', 'teacher', *command[4:]]
            role = 'teacher'
        elif len(command) > 4 and command[3:5] == ['p6_fast.launch', 'workflow']:
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('control.py', []).append(r'''            command = [*command[:3], 'p6_t4.launch', 'student', *command[5:]]
            role = 'student'
        previous = os.environ.get('CS781_P6_T4_ROLE')
        if role:
            os.environ['CS781_P6_T4_ROLE'] = role
        try:
            return original(command, deadline, profile)
        finally:
            if previous is None:
                os.environ.pop('CS781_P6_T4_ROLE', None)
            else:
                os.environ['CS781_P6_T4_ROLE'] = previous
    launch.child = child
''')


[Markdown Cell]

`p6_t4/decoding.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''"""Packed complete-source encoding and native batched greedy KV/GRU decoding."""
import numpy as np
import torch
from torch.nn import functional as F
from p6_fast.kernels import encode_chunk
from summ.generation import penalize
from .kernels import pool_memory


def penalize_many(logits, histories, minimum, repetition, ngram):
    """Same per-case controls without a GPU-to-CPU finite-check per case/token."""
    scores = logits.float().clone()
    scores[:, [0, 1, 3, 4, 5]] = -torch.inf
    too_short = [row for row, tokens in enumerate(histories) if len(tokens) < minimum]
    if too_short:
        scores[too_short, 2] = -torch.inf
    repeated_rows, repeated_tokens, blocked_rows, blocked_tokens = [], [], [], []
    for row, tokens in enumerate(histories):
        if repetition != 1:
            values = sorted(set(tokens))
            repeated_rows.extend([row] * len(values))
            repeated_tokens.extend(values)
        if ngram > 0 and len(tokens) >= ngram - 1:
            prefix = tuple(tokens[-(ngram - 1):]) if ngram > 1 else ()
            blocked = sorted({tokens[i + ngram - 1] for i in range(len(tokens) - ngram + 1)
                              if tuple(tokens[i:i + ngram - 1]) == prefix})
            blocked_rows.extend([row] * len(blocked))
            blocked_tokens.extend(blocked)
    if repeated_tokens:
        rows, columns = torch.tensor((repeated_rows, repeated_tokens), device=scores.device)
        previous = scores[rows, columns]
        scores[rows, columns] = torch.where(previous < 0, previous * repetition, previous / repetition)
    if blocked_tokens:
        rows, columns = torch.tensor((blocked_rows, blocked_tokens), device=scores.device)
        scores[rows, columns] = -torch.inf
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''    # Tensor-only fallback exactly matches the original per-case EOS relaxation.
    scores[:, 2] = torch.where(torch.isfinite(scores).any(-1), scores[:, 2], torch.zeros_like(scores[:, 2]))
    return scores


def memory_length(source_length, cfg):
    full, tail = divmod(source_length, cfg.chunk_tokens)
    return min(cfg.memory_slots, full * min(cfg.chunk_tokens, cfg.tokens_per_chunk) + min(tail, cfg.tokens_per_chunk))


@torch.inference_mode()
def encode_many(model, documents, cfg, device, batch_size, stop=None):
    """Pack chunks across documents; preserve every tail and per-document pooling."""
    parts, tags = [], []
    for index, document in enumerate(documents):
        for first in range(0, len(document), cfg.chunk_tokens):
            parts.append(document[first:first + cfg.chunk_tokens])
            tags.append(index)
    values = [[] for _ in documents]
    first = 0
    while first < len(parts):
        if stop is not None and stop.is_set():
            return None
        amount = min(batch_size, len(parts) - first)
        try:
            hidden = packed_chunks(model, parts[first:first + amount], cfg, device)
        except torch.cuda.OutOfMemoryError:
            if amount == 1:
                raise
            batch_size = max(1, amount // 2)
            torch.cuda.empty_cache()
            continue
        offset = 0
        for document, part in zip(tags[first:first + amount], parts[first:first + amount]):
            length = min(len(part), cfg.tokens_per_chunk)
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''            values[document].append(hidden[offset:offset + length])
            offset += length
        first += amount
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
        return [pool_memory(model, torch.cat(chunks).float()) for chunks in values]


def packed_chunks(model, parts, cfg, device):
    ids = torch.zeros(len(parts), cfg.chunk_tokens, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    lengths = []
    for row, part in enumerate(parts):
        ids[row, :len(part)] = torch.from_numpy(np.asarray(part, np.int64).copy())
        mask[row, :len(part)] = True
        lengths.append(len(part))
    if device.type == 'cuda':
        ids, mask = ids.pin_memory().to(device, non_blocking=True), mask.pin_memory().to(device, non_blocking=True)
    else:
        ids, mask = ids.to(device), mask.to(device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
        return encode_chunk(model, ids, mask, tuple(lengths))


def b_step(model, embedded, state):
    hidden = embedded + model.backbone.target_positions(torch.tensor([state['position']], device=embedded.device))[None]
    hidden = model.backbone.dropout(hidden)
    values = []
    mask = torch.ones(1, 1, 1, state['cross'][0][0].shape[2], device=hidden.device, dtype=torch.bool)
    for index, layer in enumerate(model.backbone.decoder):
        hidden, current = layer.step(hidden, state['self'][index], state['cross'][index], mask)
        values.append(current)
    logits = F.linear(model.backbone.decoder_norm(hidden[:, -1]).float(), model.embedding().weight.float())
    return logits, {'self': values, 'cross': state['cross'], 'position': state['position'] + 1}


''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''def prefill(model, memory, ids):
    """Same native states as serial prefill; memory[B,M,D], IDs[B,L]."""
    batch = len(ids)
    if model.cfg.family == 'b':
        layers = model.backbone.decoder
        # The approved BART-style own B supports parallel causal prefill. Other
        # compatible own B variants retain the exact batched native step path.
        supported = all(hasattr(layer, 'norm1') and hasattr(layer, 'self_attention')
                        and hasattr(layer.self_attention, 'project_kv') for layer in layers)
        if supported:
            hidden = model.embedding()(ids)
            if model.cfg.method == 'prompt':
                hidden = torch.cat((model.backbone.soft_decoder_prompt[None].expand(batch, -1, -1), hidden), 1)
            hidden = model.backbone.dropout(hidden + model.backbone.target_positions(torch.arange(hidden.shape[1], device=hidden.device))[None])
            causal = torch.ones(hidden.shape[1], hidden.shape[1], dtype=torch.bool, device=hidden.device).tril()[None, None]
            cross_mask = torch.ones(1, 1, 1, memory.shape[1], dtype=torch.bool, device=hidden.device)
            self_values, cross_values = [], []
            for layer in layers:
                self_values.append(layer.self_attention.project_kv(layer.norm1(hidden)))
                cross_values.append(layer.cross_attention.project_kv(memory))
                hidden = layer(hidden, memory, causal, cross_mask)
            logits = F.linear(model.backbone.decoder_norm(hidden[:, -1]).float(), model.embedding().weight.float())
            return logits, {'self': self_values, 'cross': cross_values, 'position': hidden.shape[1]}
        cross = [layer.cross_attention.project_kv(memory) for layer in layers]
        state = {'self': [None] * len(cross), 'cross': cross, 'position': 0}
        if model.cfg.method == 'prompt':
            for embedding in model.backbone.soft_decoder_prompt:
                _, state = b_step(model, embedding[None, None].expand(batch, -1, -1), state)
        for position in range(ids.shape[1]):
            logits, state = b_step(model, model.embedding()(ids[:, position:position + 1]), state)
        return logits, state
    hidden = torch.cat((memory.to(model.embedding().weight.dtype), model.embedding()(ids)), 1)
    positions = torch.arange(hidden.shape[1], device=hidden.device)
    values = []
    for layer in (model.backbone.stages if model.cfg.family == 'c' else model.backbone.layers):
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''        if model.cfg.family == 'c':
            hidden, kv, recurrent = layer.cached(hidden, positions, None, None)
            values.append((kv, recurrent))
        else:
            hidden, kv = layer.cached(hidden, positions, None)
            values.append(kv)
    logits = F.linear(model.backbone.final_norm(hidden[:, -1]).float(), model.embedding().weight.float())
    return logits, {'values': values, 'position': hidden.shape[1]}


def step(model, tokens, state):
    hidden = model.embedding()(tokens[:, None])
    if model.cfg.family == 'b':
        return b_step(model, hidden, state)
    positions = torch.tensor([state['position']], device=hidden.device)
    values = []
    for index, layer in enumerate(model.backbone.stages if model.cfg.family == 'c' else model.backbone.layers):
        if model.cfg.family == 'c':
            kv, recurrent = state['values'][index]
            hidden, kv, recurrent = layer.cached(hidden, positions, kv, recurrent)
            values.append((kv, recurrent))
        else:
            hidden, kv = layer.cached(hidden, positions, state['values'][index])
            values.append(kv)
    logits = F.linear(model.backbone.final_norm(hidden[:, -1]).float(), model.embedding().weight.float())
    return logits, {'values': values, 'position': state['position'] + 1}


@torch.inference_mode()
def greedy_many(model, memories, documents, codec, cfg, device, commit, stop=None, heartbeat=None):
    """Commit each finished case immediately. No reference text or length is read."""
    if cfg.beam_size != 1 or len({value.shape[1] for value in memories}) != 1:
        raise ValueError('Batched greedy requires equal memory lengths and the saved beam size of one')
    memory = torch.cat(memories)
    count = len(memory)
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''    tokens, ended, resets = [[] for _ in memories], [False] * count, [0] * count
    capacity = model.base_cfg.max_decoder_tokens if cfg.family == 'b' else model.base_cfg.max_seq_tokens
    prefix = cfg.prompt_tokens if cfg.family == 'b' and cfg.method == 'prompt' else 0 if cfg.family == 'b' else memory.shape[1]
    context = min(cfg.target_tokens + cfg.target_context, capacity - prefix - 1)
    if context < 2:
        raise ValueError('No generation space remains after the conditioning prefix')
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
        logits, state = prefill(model, memory, torch.ones(count, 1, dtype=torch.long, device=device))
        for position in range(cfg.generation_max_tokens):
            if stop is not None and stop.is_set():
                return False
            if heartbeat is not None and position % 16 == 0:
                heartbeat(position, sum(map(len, tokens)), sum(not value for value in ended))
            controlled = penalize_many(logits, tokens,
                min(cfg.generation_min_tokens, cfg.generation_max_tokens - 1),
                cfg.repetition_penalty, cfg.no_repeat_ngram)
            selected = torch.topk(torch.log_softmax(controlled, -1), 1, dim=-1).indices[:, 0].tolist()
            for row, token in enumerate(selected):
                if ended[row]:
                    continue
                tokens[row].append(token)
                if token == 2 or position + 1 == cfg.generation_max_tokens:
                    text = codec.decode([value for value in tokens[row] if value != 2], skip_special_tokens=True).strip()
                    if not text:
                        raise ValueError('Model generated an empty summary; no canned replacement is supplied')
                    commit(row, text, {'generated_tokens': len(tokens[row]), 'ended_with_eos': token == 2,
                        'hit_generation_limit': token != 2, 'memory_tokens': memory.shape[1],
                        'context_replays_across_beams': resets[row], 'source_tokens': len(documents[row]),
                        'all_source_tokens_encoded': True})
                    ended[row] = True
            if all(ended):
                return True
            if state['position'] - prefix >= context:
                retained_count = max(0, min(cfg.target_context, context - 2))
                # Active streams have identical decode length. Finished streams
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('decoding.py', []).append(r'''                # only receive ignored filler tokens; they cannot affect peers.
                history = []
                for row in range(count):
                    retained = tokens[row][-retained_count:] if retained_count else []
                    padded = ([2] * retained_count + retained)[-retained_count:] if retained_count else []
                    history.append([1, *padded])
                    resets[row] += int(not ended[row])
                logits, state = prefill(model, memory, torch.tensor(history, device=device))
            else:
                logits, state = step(model, torch.tensor(selected, device=device), state)
    return True
''')


[Markdown Cell]

`p6_t4/inference.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''"""Both-GPU packed greedy inference, length-balanced ownership and durable progress."""
from dataclasses import replace
from collections import defaultdict
from pathlib import Path
import gc
import json
import os
import signal
import time
import torch
import torch.multiprocessing as mp
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from summ.config import SummConfig
from summ.data import SummaryWindows
from summ.generation import generate
from summ.io_utils import atomic_json, identity, read_json
from summ.predict import partial_rows, generation_choices
from summ.evaluation import read_predictions
from summ.state import Checkpoints, seed_everything
from .decoding import encode_many, greedy_many, memory_length


def work_plan(directory, split, cases, signature):
    """Retain all committed legacy/new rank rows; balance only the remaining cases."""
    path = directory / (split + '_gpu_plan.json')
    saved = [partial_rows(directory / f'{split}_rank{rank}.jsonl') for rank in range(2)]
    expected = {row['id'] for row in cases}
    if (set(saved[0]) & set(saved[1])) or not ((set(saved[0]) | set(saved[1])) <= expected):
        raise ValueError('Generation journals have duplicate or unknown document IDs')
    if path.is_file():
        plan = read_json(path)
        if plan['signature'] != signature:
            raise ValueError('Saved generation GPU plan belongs to different source data/model settings')
        if sorted(plan['owners'][0] + plan['owners'][1]) != list(range(len(cases))):
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''            raise ValueError('Saved GPU ownership no longer covers every document exactly once')
        for rank in range(2):
            if not set(saved[rank]) <= {cases[index]['id'] for index in plan['owners'][rank]}:
                raise ValueError('Committed generation journal conflicts with the saved GPU ownership')
        return plan
    by_id = {row['id']: index for index, row in enumerate(cases)}
    owners = [[by_id[key] for key in values] for values in saved]
    committed = set(saved[0]) | set(saved[1])
    remaining = sorted([i for i, row in enumerate(cases) if row['id'] not in committed],
                       key=lambda i: (-cases[i]['source_length'], i))
    loads = [0, 0]
    for index in remaining:
        rank = min(range(2), key=lambda i: (loads[i], i))
        owners[rank].append(index)
        loads[rank] += cases[index]['source_length']
    if sorted(owners[0] + owners[1]) != list(range(len(cases))):
        raise ValueError('GPU work plan did not cover every document exactly once')
    plan = {'signature': signature, 'owners': owners, 'remaining_source_token_loads': loads,
            'previous_completed_rows_preserved': len(committed)}
    atomic_json(path, plan)
    return plan


def inference_worker(rank, job_path, splits, choices, variant, selection, event):
    from .control import profile
    from summ.predict import load_best
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    job = read_json(job_path)
    root = Path(job['config']['run_dir']) / 'generation' / variant
    atomic_json(root / f'speed_gpu{rank}.json', {'stage': 'loading_model', 'split': splits[0], 'completed': 0})
    model, job, cfg, _ = load_best(job_path, device, selection)
    cfg = replace(cfg, **choices, inference_checkpoint=selection)
    model.cfg = cfg
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''    model.eval()
    seed_everything(cfg.seed + rank)
    codec = Tokenizer.from_file(job['tokenizer_path'])
    settings = profile()['options']
    for split in splits:
        dataset = SummaryWindows(job['cache'], cfg, split)
        cases = job['cache']['cases'][split]
        plan = read_json(root / (split + '_gpu_plan.json'))
        path = root / f'{split}_rank{rank}.jsonl'
        done = partial_rows(path)
        groups = defaultdict(list)
        for index in plan['owners'][rank]:
            if cases[index]['id'] not in done:
                groups[memory_length(cases[index]['source_length'], cfg)].append(index)
        last, totals = [0.], [0]
        def progress(step=0, current_tokens=0, active=0, force=False):
            if force or time.monotonic() - last[0] >= 2:
                atomic_json(root / f'speed_gpu{rank}.json', {'stage': 'generating', 'split': split,
                    'completed': len(done), 'decode_step': step, 'active_batch': active,
                    'committed_generated_tokens': totals[0],
                    'GPU': torch.cuda.get_device_name(rank), 'peak_bytes': torch.cuda.max_memory_allocated(device)})
                last[0] = time.monotonic()
        with path.open('a', encoding='utf-8', newline='\n') as stream:
            def commit(index, text, details):
                key = cases[index]['id']
                if key in done:
                    raise ValueError('A completed generated document would be duplicated')
                row = {'id': key, 'summary': text, 'details': details}
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
                stream.flush()
                os.fsync(stream.fileno())
                done[key] = row
                totals[0] += details['generated_tokens']
                progress(force=True)
            for _, pending in sorted(groups.items(), reverse=True):
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''                batch_size = settings['generation_batch'] if cfg.beam_size == 1 else 1
                while pending:
                    if event.is_set():
                        progress(force=True)
                        return
                    indices = pending[:batch_size]
                    documents = [dataset.document(index) for index in indices]
                    memories, failed = None, False
                    try:
                        if cfg.beam_size == 1:
                            memories = encode_many(model, documents, cfg, device, settings['inference_chunks'], event)
                            if memories is None:
                                return
                            complete = greedy_many(model, memories, documents, codec, cfg, device,
                                lambda row, text, details: commit(indices[row], text, details), event, progress)
                            if not complete:
                                return
                        else:
                            # A saved beam-search rule is never silently changed to greedy.
                            text, details = generate(model, documents[0], codec, cfg, device)
                            commit(indices[0], text, details)
                    except torch.cuda.OutOfMemoryError:
                        if len(indices) == 1:
                            raise
                        batch_size = max(1, len(indices) // 2)
                        failed = True
                    del memories, documents
                    pending[:] = [index for index in pending if cases[index]['id'] not in done]
                    if failed:
                        gc.collect()
                        torch.cuda.empty_cache()
                progress(force=True)


def parallel_generate(job_path, splits, overrides=None):
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''    """Same prediction schema/decoding controls; aggregate tqdm reports both GPUs."""
    job = read_json(job_path)
    cfg = replace(SummConfig(**job['config']), **(overrides or {}))
    cfg.validate()
    manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    chosen = manager.best() if cfg.inference_checkpoint == 'best' else manager.latest()
    if chosen is None:
        raise FileNotFoundError('No trained checkpoint exists for generation')
    variant = identity({'checkpoint': chosen[1]['sha256'], 'data': job['prepared_identity'], 'decode': generation_choices(cfg)})[:20]
    directory = Path(cfg.run_dir) / 'generation' / variant
    directory.mkdir(parents=True, exist_ok=True)
    signature = {'checkpoint_sha256': chosen[1]['sha256'], 'prepared_identity': job['prepared_identity'], 'generation': generation_choices(cfg)}
    file = directory / 'generation_identity.json'
    if file.is_file() and read_json(file) != signature:
        raise ValueError('Saved generation identity changed')
    atomic_json(file, signature)
    total, initial = 0, 0
    for split in splits:
        cases = job['cache']['cases'][split]
        plan_signature = identity({'generation': signature, 'cache': job['cache']['identity'],
                                   'cases': [(row['id'], row['source_length']) for row in cases]})
        work_plan(directory, split, cases, plan_signature)
        total += len(cases)
        initial += sum(len(partial_rows(directory / f'{split}_rank{rank}.jsonl')) for rank in range(2))
    context = mp.get_context('spawn')
    event = context.Event()
    handlers = {}
    progress = tqdm(total=total, initial=initial, desc='Generate summaries — BOTH T4 GPUs', unit='documents', mininterval=1.)
    # Incremental newline counts: never rescan large journals on every refresh,
    # and never truncate/read-validate a journal while its worker is writing.
    offsets = {}
    for split in splits:
        for rank in range(2):
            journal = directory / f'{split}_rank{rank}.jsonl'
            offsets[journal] = journal.stat().st_size if journal.is_file() else 0
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''    completed = initial
    if initial < total:
        try:
            for signum in (signal.SIGINT, signal.SIGTERM):
                handlers[signum] = signal.signal(signum, lambda *_: event.set())
            workers = mp.spawn(inference_worker,
                args=(str(job_path), tuple(splits), generation_choices(cfg), variant, cfg.inference_checkpoint, event),
                nprocs=2, join=False)
            finished = False
            while not finished:
                finished = workers.join(timeout=1.)
                # Count durable rows, not speculative batches, across ALL splits.
                states = []
                for rank in range(2):
                    state = directory / f'speed_gpu{rank}.json'
                    states.append(read_json(state) if state.is_file() else {})
                    for split in splits:
                        journal = directory / f'{split}_rank{rank}.jsonl'
                        if journal.is_file():
                            with journal.open('rb') as stream:
                                stream.seek(offsets[journal])
                                fresh = stream.read()
                            boundary = fresh.rfind(b'\n') + 1
                            completed += fresh[:boundary].count(b'\n')
                            offsets[journal] += boundary
                progress.update(max(0, completed - progress.n))
                progress.set_postfix(gpu0=states[0].get('stage', 'starting'), gpu1=states[1].get('stage', 'starting'),
                    step0=states[0].get('decode_step', 0), step1=states[1].get('decode_step', 0))
        finally:
            for signum, handler in handlers.items():
                signal.signal(signum, handler)
            progress.close()
        if event.is_set():
            atomic_json(directory / 'speed_generation_pause.json', {'status': 'paused_recoverable', 'completed': progress.n, 'total': total})
            raise InterruptedError('Generation paused; all committed document rows are retained')
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('inference.py', []).append(r'''    else:
        progress.close()
    result = {}
    for split in splits:
        values = {}
        for rank in range(2):
            rows = partial_rows(directory / f'{split}_rank{rank}.jsonl')
            if set(values) & set(rows):
                raise ValueError('Both GPUs generated the same case ID')
            values.update(rows)
        cases = job['cache']['cases'][split]
        if set(values) != {row['id'] for row in cases}:
            raise ValueError('Incomplete inference; no final teacher/submission is claimed')
        path = directory / (split + '_prediction.json')
        atomic_json(path, [{'id': row['id'], 'summary': values[row['id']]['summary']} for row in cases])
        read_predictions(path)
        result[split] = path
    return directory, result
''')


[Markdown Cell]

`p6_t4/kernels.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''"""Larger measured source batches and batched complete gold/pseudo target windows."""
import time
import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from p6_fast.kernels import encode_chunk
from summ.state import capture_rng, restore_rng


def collate(row, chunk_tokens, device=None):
    """CPU-known source lengths avoid a GPU-to-CPU synchronization per document."""
    from p6_fast.data import collate as original
    result = original(row, chunk_tokens, device)
    length = len(row['source'])
    result['source_lengths'] = tuple(min(chunk_tokens, length - start) for start in range(0, length, chunk_tokens))
    return result


def pool_memory(model, memory):
    position = torch.linspace(0, 1, len(memory), device=memory.device)
    positions = torch.stack((position, 1 - position, torch.ones_like(position)), dim=-1)
    memory = model.memory_norm(model.memory_projection(memory) + model.memory_positions(positions))
    importance = F.softplus(model.memory_salience(memory)).clamp_min(1e-4)
    slots = min(len(memory), model.cfg.memory_slots)
    numerator = F.adaptive_avg_pool1d((memory * importance).T[None], slots)
    denominator = F.adaptive_avg_pool1d(importance.T[None], slots)
    return (numerator / denominator).transpose(1, 2) * torch.sigmoid(model.memory_scale)


def encode_document(self, source_ids, source_mask, source_lengths=None):
    lengths = list(source_lengths) if source_lengths is not None else source_mask.sum(-1).cpu().tolist()
    if len(lengths) != len(source_ids) or any(value < 1 for value in lengths):
        raise ValueError('Invalid source chunk coverage')
    settings = getattr(self, '_p6_t4', {})
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''    batch = settings.get('source_batch', 8)
    values = []
    first = 0
    while first < len(lengths):
        args = (source_ids[first:first + batch], source_mask[first:first + batch], tuple(lengths[first:first + batch]))
        function = lambda ids, mask, actual: encode_chunk(self, ids, mask, actual)
        try:
            values.append(checkpoint(function, *args, use_reentrant=False)
                          if self.training and self.cfg.checkpoint_encoder else function(*args))
        except torch.cuda.OutOfMemoryError:
            if self.training or len(args[2]) == 1:
                raise
            batch = max(1, len(args[2]) // 2)
            torch.cuda.empty_cache()
            continue
        first += len(args[2])
    return pool_memory(self, torch.cat(values))


def decoder_hidden(model, decoder_ids, memory):
    backbone, batch = model.backbone, len(decoder_ids)
    if model.cfg.family == 'b':
        hidden = model.embedding()(decoder_ids)
        if model.cfg.method == 'prompt':
            hidden = torch.cat((backbone.soft_decoder_prompt[None].expand(batch, -1, -1), hidden), dim=1)
        hidden = backbone.dropout(hidden + backbone.target_positions(torch.arange(hidden.shape[1], device=hidden.device))[None])
        causal = torch.ones(hidden.shape[1], hidden.shape[1], dtype=torch.bool, device=hidden.device).tril()[None, None]
        cross = torch.ones(1, 1, 1, memory.shape[1], dtype=torch.bool, device=hidden.device)
        for layer in backbone.decoder:
            hidden = layer(hidden, memory, causal, cross)
        return backbone.decoder_norm(hidden)[:, -decoder_ids.shape[1]:]
    hidden = torch.cat((memory.expand(batch, -1, -1).to(model.embedding().weight.dtype), model.embedding()(decoder_ids)), dim=1)
    for layer in (backbone.stages if model.cfg.family == 'c' else backbone.layers):
        hidden = layer(hidden, None)
    return backbone.final_norm(hidden)[:, -decoder_ids.shape[1]:]
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''

def grouped_loss(model, ids, labels, scales, memory):
    hidden = decoder_hidden(model, ids, memory)
    selected = labels != -100
    values, targets = hidden[selected], labels[selected]
    token_scales = scales[:, None].expand_as(labels)[selected]
    loss = hidden.sum() * 0.
    for first in range(0, len(targets), 256):
        logits = F.linear(values[first:first + 256].float(), model.embedding().weight.float())
        losses = F.cross_entropy(logits.float(), targets[first:first + 256], reduction='none',
            label_smoothing=model.cfg.label_smoothing if model.training else 0.)
        loss = loss + (losses * token_scales[first:first + 256]).sum()
    return loss


def forward(self, source_ids, source_mask, decoder_ids, labels, loss_weights=None, source_lengths=None):
    """Same weighted token objective, grouping windows within the SAME document."""
    memory = encode_document(self, source_ids, source_mask, source_lengths)
    decoders = decoder_ids if isinstance(decoder_ids, (tuple, list)) else [decoder_ids]
    targets = labels if isinstance(labels, (tuple, list)) else [labels]
    scales = loss_weights if loss_weights is not None else [1.] * len(decoders)
    if not len(decoders) == len(targets) == len(scales) or not decoders:
        raise ValueError('Mismatched complete target-window lists')
    batch = getattr(self, '_p6_t4', {}).get('window_batch', 2)
    loss = memory.sum() * 0.
    for first in range(0, len(decoders), batch):
        parts, actual, weights = decoders[first:first + batch], targets[first:first + batch], scales[first:first + batch]
        length = max(value.shape[-1] for value in parts)
        ids = torch.cat([F.pad(value, (0, length - value.shape[-1]), value=0) for value in parts])
        padded = torch.cat([F.pad(value, (0, length - value.shape[-1]), value=-100) for value in actual])
        factors = torch.tensor(weights, device=ids.device, dtype=torch.float32)
        function = lambda ids, labels, factors, memory: grouped_loss(self, ids, labels, factors, memory)
        loss = loss + (checkpoint(function, ids, padded, factors, memory, use_reentrant=False)
            if self.training and self.cfg.checkpoint_encoder else function(ids, padded, factors, memory))
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''    return loss


def tune(model, options):
    """Probe before DDP hooks exist; preserve weights/RNG and do no optimizer step."""
    cfg, device = model.cfg, next(model.parameters()).device
    if device.type != 'cuda':
        model._p6_t4 = {'source_batch': cfg.chunk_micro_batch, 'window_batch': min(2, options['window_batch'])}
        return {'status': 'CPU_fixture', 'selected': model._p6_t4}
    saved, mode = capture_rng(device), model.training
    started, rows = time.monotonic(), []
    base = torch.cuda.memory_allocated(device)
    active = sum(value.numel() * value.element_size() for value in model.parameters() if value.requires_grad)
    total = torch.cuda.get_device_properties(device).total_memory
    free, _ = torch.cuda.mem_get_info(device)
    reserve = 3 * active + (1 << 30)
    budget = min(total * .65, base + max(0, free - reserve))
    selected = {'source_batch': min(4, options['source_batch']), 'window_batch': 1}
    try:
        model.train()
        for kind, choices in (('source', (1, 2, 4, 8, 16, 32, 64)), ('window', (1, 2, 4, 8))):
            limit = options['source_batch'] if kind == 'source' else options['window_batch']
            valid = []
            for batch in choices:
                if batch > limit or (valid and time.monotonic() - started >= options['tuning_seconds']):
                    break
                ids = mask = memory = labels = scales = None
                try:
                    model.zero_grad(set_to_none=True)
                    torch.cuda.reset_peak_memory_stats(device)
                    begin = time.perf_counter()
                    with torch.autocast('cuda', dtype=torch.float16):
                        if kind == 'source':
                            ids = torch.full((batch, cfg.chunk_tokens), 7, dtype=torch.long, device=device)
                            mask = torch.ones_like(ids, dtype=torch.bool)
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''                            loss = encode_chunk(model, ids, mask, (cfg.chunk_tokens,) * batch).float().square().mean()
                        else:
                            length = cfg.target_tokens + cfg.target_context
                            ids = torch.full((batch, length), 7, dtype=torch.long, device=device)
                            labels = ids.clone()
                            memory = torch.zeros(1, cfg.memory_slots, model.base_cfg.d_model, device=device, requires_grad=True)
                            scales = torch.ones(batch, device=device) / (batch * length)
                            loss = grouped_loss(model, ids, labels, scales, memory)
                    loss.backward()
                    torch.cuda.synchronize(device)
                    seconds = time.perf_counter() - begin
                    peak = torch.cuda.max_memory_allocated(device)
                    row = {'kind': kind, 'batch': batch, 'seconds': seconds, 'peak_bytes': peak, 'fits_budget': peak <= budget}
                    rows.append(row)
                    if peak <= budget:
                        valid.append(row)
                    else:
                        break
                except torch.cuda.OutOfMemoryError:
                    rows.append({'kind': kind, 'batch': batch, 'OOM': True})
                    break
                finally:
                    model.zero_grad(set_to_none=True)
                    if 'loss' in locals():
                        del loss
                    del ids, mask, memory, labels, scales
                    torch.cuda.empty_cache()
            if valid:
                winner = max(valid, key=lambda value: value['batch'] / value['seconds'])
                selected['source_batch' if kind == 'source' else 'window_batch'] = winner['batch']
            else:
                raise RuntimeError('Even the smallest ' + kind + ' batch exceeded the reserved memory budget; inspect t4_probe reports/settings.')
        model._p6_t4 = selected
        return {'status': 'measured', 'selected': selected, 'trials': rows, 'budget_bytes': int(budget),
            'GPU': torch.cuda.get_device_name(device), 'weights_updated': False, 'RNG_restored': True}
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('kernels.py', []).append(r'''    finally:
        model.zero_grad(set_to_none=True)
        model.train(mode)
        restore_rng(saved, device)
        torch.cuda.empty_cache()
''')


[Markdown Cell]

`p6_t4/launch.py`

In [ ]:
# [Code Cell]
T4_PARTS.setdefault('launch.py', []).append(r'''"""Identical execution installation in parent and NCCL/inference spawn children."""
import argparse
import json
import os
import sys
from summ.io_utils import read_json
from .control import bind, install, redirect_children


def inherited_install():
    fast = os.environ.get('CS781_P6_FAST_PROFILE')
    if fast:
        from p6_fast.control import install as install_fast
        install_fast(read_json(fast))
    if os.environ.get('CS781_P6_T4_PROFILE') and (fast or os.environ.get('CS781_P6_T4_ROLE') == 'teacher'):
        install()


# spawn reimports this module before unpickling the worker. The supervisor must
# not install student training kernels until its original p6_fast layout is bound.
if os.environ.get('CS781_P6_T4_PROFILE'):
    inherited_install()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=('workflow', 'student', 'teacher', 'train'))
    parser.add_argument('--speed-options')
    parser.add_argument('--optimization')
    args, remainder = parser.parse_known_args()
    if args.action == 'workflow':
        if not args.speed_options or '--plan' not in remainder or not args.optimization:
            parser.error('workflow needs --speed-options, --optimization and --plan')
        bind(remainder[remainder.index('--plan') + 1], args.speed_options)
        redirect_children()
''')


In [ ]:
# [Code Cell]
T4_PARTS.setdefault('launch.py', []).append(r'''        from p6_auto_teacher.launch import main as automatic
        sys.argv = ['p6_t4.auto_teacher', 'workflow', '--optimization', args.optimization, *remainder]
        automatic()
    elif args.action == 'student':
        if not args.optimization or '--plan' not in remainder:
            parser.error('student needs --optimization and --plan')
        # This child must install p6_fast FIRST. Its import-time execution only
        # installs T4 when no fast profile is inherited; defer it in that case.
        from p6_fast.control import bind as bind_fast, install as install_fast
        fast = bind_fast(remainder[remainder.index('--plan') + 1], args.optimization)
        install_fast(fast)
        install()
        from summ.workflow import main as workflow
        sys.argv = ['p6_t4.student', *remainder]
        workflow()
    elif args.action == 'teacher':
        install()
        from summ.jobs import hardware
        from summ.predict import export
        inner = argparse.ArgumentParser()
        inner.add_argument('--job', required=True)
        job = inner.parse_args(remainder).job
        hardware()
        print(json.dumps(export(job, teacher_only=True), indent=2), flush=True)
    else:
        if not os.environ.get('CS781_P6_FAST_PROFILE'):
            parser.error('Train must be started by the checked workflow')
        from summ.train import main as train
        sys.argv = ['p6_t4.train', *remainder]
        train()


if __name__ == '__main__':
    main()
''')


In [ ]:
# [Code Cell]
T4_SOURCE_SHA256 = {'__init__.py': '6ee0b6766df685b2593ea705ddb86e763ff7b73c743021d579ac1b53b488af93', 'control.py': 'd9d00efd9184975fa948b1868bcdecff89177b02b8047792454d66c7c5957a81', 'decoding.py': '2873361173c8b8a8f6a82d0bab84fcd57ac6ecf949c0deaf2d82e10f1742b4ae', 'inference.py': 'a57a0e5b51792fcce67e30743935d8aacf7546f08714e36d7c0808ab66f3b29d', 'kernels.py': '8e46bd8ec2b823baa08529848ce851d0b597aff0eebd3b43df15975f567a739b', 'launch.py': '8f2dc7a95dcd8e6ab3fd91f46a9c8b6752dbe4d6c71b69e6a38a28ccfd9af40e'}
t4_directory = code_directory / 'p6_t4'
t4_directory.mkdir(exist_ok=True)
for name, pieces in T4_PARTS.items():
    source = ''.join(pieces)
    assert hashlib.sha256(source.encode()).hexdigest() == T4_SOURCE_SHA256[name], name
    compile(source, name, 'exec')
    (t4_directory / name).write_bytes(source.encode())
speed_options_path = Path(OUTPUT_ROOT) / 't4_speed_options.json'
speed_options_path.write_text(json.dumps({
    'source_batch': T4_SOURCE_BATCH_CAP,
    'window_batch': T4_TARGET_WINDOW_BATCH_CAP,
    'generation_batch': T4_GENERATION_BATCH,
    'inference_chunks': T4_INFERENCE_CHUNKS,
    'tuning_seconds': T4_TUNING_SECONDS,
}, indent=2), encoding='utf-8')
print('T4 ×2 batched execution modules verified. Global training batch and all data retained.')


[Markdown Cell]

## 3. Validate launch settings and inherit exact recovery choices

In [ ]:
# [Code Cell]
PARENT_CATALOG = {'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}


In [ ]:
# [Code Cell]
from dataclasses import asdict
from summ.config import SummConfig
from summ.io_utils import atomic_json, read_json

requested = dict(experiment_id=EXPERIMENT_ID, family=FAMILY, method=METHOD, run_dir=OUTPUT_ROOT,
    input_root=INPUT_ROOT, prepared_dir=PREPARED_DIR, parent_run_dir=PARENT_RUN_DIR,
    tokenizer_path=TOKENIZER_PATH, metadata_path=METADATA_PATH, resume_run_dir=RESUME_RUN_DIR,
    teacher_run_dir=TEACHER_RUN_DIR, max_epochs=MAX_EPOCHS, target_total_epochs=TARGET_TOTAL_EPOCHS,
    max_run_seconds=int(TRAINING_MINUTES * 60), chunk_tokens=CHUNK_TOKENS,
    chunk_micro_batch=CHUNK_MICRO_BATCH, memory_slots=MEMORY_SLOTS, target_tokens=TARGET_TOKENS,
    target_context=TARGET_CONTEXT, backbone_lr=BACKBONE_LR, memory_lr=MEMORY_LR,
    generation_max_tokens=GENERATION_MAX_TOKENS, generation_min_tokens=GENERATION_MIN_TOKENS,
    beam_size=BEAM_SIZE)
requested['accumulation_steps'] = GRADIENT_ACCUMULATION_STEPS
local_job = Path(OUTPUT_ROOT) / 'job.json'
paths = [local_job] if local_job.is_file() else [Path(RESUME_RUN_DIR) / 'job.json'] if RESUME_RUN_DIR else list(Path(INPUT_ROOT).rglob('job.json'))
prior = [(path, read_json(path)) for path in paths if path.is_file()]
prior = [(path, value) for path, value in prior if value.get('version') == 'cs781-phase6-summ-v1'
         and value.get('config', {}).get('experiment_id') == EXPERIMENT_ID]
if prior:
    if len({value['fingerprint'] for _, value in prior}) != 1:
        raise ValueError('Conflicting saved versions. Set RESUME_RUN_DIR explicitly.')
    settings = prior[0][1]['config'].copy()
    transport = {'run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path',
        'metadata_path', 'resume_run_dir', 'teacher_run_dir', 'target_total_epochs', 'max_run_seconds'}
    settings.update({key: value for key, value in requested.items() if key in transport and value is not None})
    print('Inherited original learning choices and epoch plan for exact recovery.')
else:
    if RESUME_RUN_DIR or REQUIRE_EXISTING_CHECKPOINT:
        raise ValueError('This SAME saved experiment was not found; no fresh restart was allowed.')
    settings = asdict(SummConfig(**requested))
if MODE == 'teacher':
    screened_path = Path(RESUME_RUN_DIR) / 'screen_report.json'
    if screened_path.is_file():
        screened = read_json(screened_path)
        if screened.get('status') == 'submission_verified':
            settings.update(screened['generation'])
            settings['inference_checkpoint'] = screened.get('checkpoint_selection', 'best')
            print('Teacher export uses its frozen tune-selected checkpoint and decoding rule.')
config = SummConfig(**settings)
config.validate()
plan_path = Path(OUTPUT_ROOT) / 'pipeline_plan.json'
catalog_path = code_directory / 'parent_catalog.json'
atomic_json(plan_path, {'mode': MODE, 'config': asdict(config)})
atomic_json(catalog_path, PARENT_CATALOG)
print(json.dumps(asdict(config), indent=2))


[Markdown Cell]

## 4. Stream progress and execute

In [ ]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = os.pathsep.join((str(compat_directory), str(code_directory), environment.get('PYTHONPATH', '')))
    environment['CS781_NUMPY_RESUME_REQUIRED'] = '1'
    process = subprocess.Popen(command, cwd=code_directory, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            try:
                ready, _, _ = select.select([process.stdout], [], [], 1.)
                if ready:
                    block = os.read(process.stdout.fileno(), 8192)
                    if block:
                        print(decoder.decode(block), end='', flush=True)
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                print('Requesting a saved stop. Keep this cell running until the child exits.', flush=True)
                process.send_signal(signal.SIGINT)
        result = process.wait()
        print(decoder.decode(b'', final=True), end='', flush=True)
        if result:
            raise subprocess.CalledProcessError(result, command)
    finally:
        process.stdout.close()


In [ ]:
# [Code Cell]
run_streamed([sys.executable, '-u', '-m', 'p6_t4.launch', 'workflow', '--speed-options', str(speed_options_path), '--auto-teacher', str(auto_teacher_options_path), '--optimization', str(optimization_path), '--plan', str(plan_path), '--catalog', str(catalog_path)])


[Markdown Cell]

## 5. Verify status and keep the actual outputs

In [ ]:
# [Code Cell]
from IPython.display import FileLink, display
report = read_json(Path(OUTPUT_ROOT) / 'final_report.json')
print(json.dumps(report, indent=2))
if report.get('status') == 'submission_verified':
    display(FileLink(report['zip_path']))
if MODE == 'teacher' and report.get('status') == 'complete':
    display(FileLink(str(Path(OUTPUT_ROOT) / 'train_summaries.jsonl')))
    display(FileLink(str(Path(OUTPUT_ROOT) / 'teacher_export_receipt.json')))
if (Path(OUTPUT_ROOT) / 'observations.zip').is_file():
    display(FileLink(str(Path(OUTPUT_ROOT) / 'observations.zip')))
print('Save Version with COMPLETE outputs for recovery and later phases.')


[Markdown Cell]

Keep checkpoints and their JSON receipts, job/config, code, own foundation source, tokenizer, all
generation logs/identities, prediction files and summary/metric reports. A paused run has no completed
submission. An observations ZIP helps diagnosis and is not a resumable checkpoint. Generated summaries
are model outputs, not copied validation references. The private test BERTScore is unmeasured.

The original epoch schedule is preserved on continuation. Set `TARGET_TOTAL_EPOCHS=E+1` explicitly
to request one more epoch after E completed epochs. Do not change architecture/method/tokenizer or
silently mix a different run's optimizer. A new hypothesis needs a new experiment ID.

[Markdown Cell]

### Keep the optimization identity with your output
Save `execution_optimization.json`, `code/p6_fast/`, `optimization_report.json` and the full original
task output. Continue with this optimized notebook after saving an optimized checkpoint. A future
source/update-plan mismatch stops visibly. `optimization_report.json` reports the actual source
work saved on YOUR dataset; it does not invent a wall-clock speedup or a private test score.

[Markdown Cell]

### Inputs and important automatic-stage outputs
Fresh run: P6_00 output + own B Phase 1 weights/source/tokenizer + complete trained A/C/D Phase 6
teacher output. Keep all required inputs attached until the teacher targets finish.
`auto_teacher/` stores copied real teacher checkpoints/source/tokenizer, token caches and generation
journals; `teacher_export_receipt.json` + `train_summaries.jsonl` fix the KD target snapshot.
`auto_teacher_status.json` shows the current stage. A pause requires saving COMPLETE output and
resuming this same notebook. Before B job.json exists, point TEACHER_RUN_DIR at the saved
auto_teacher/ folder containing its own job.json and leave RESUME_RUN_DIR unset; keep P6_00
and the B foundation attached. Once B job.json/checkpoints exist, set RESUME_RUN_DIR to the
saved B root instead. Once targets are ready, student continuation can reuse that snapshot.
The student's final submission ZIP is still printed by the last result cell. No teacher-training
epoch, fake target, test-reference target, or separate 06 run is silently added.

[Markdown Cell]

### Read the speed reports and resume safely
`t4_probe_gpu0.json` and `t4_probe_gpu1.json` show measured training batches, trial times,
GPU names and peak allocated memory. `t4_speed_profile.json` records the exact executor/settings.
The training bar counts optimizer-plan updates; the generation bar counts completed documents
across BOTH GPUs and shows their decode steps. Short final batches need not fill both GPUs.

Save COMPLETE output, including checkpoints, code, profiles, job.json, tokenizer, caches,
receipts and generation journals. A submission ZIP or observations ZIP alone cannot resume.
If teacher generation pauses before the B job exists, attach the saved output and set
TEACHER_RUN_DIR to its auto_teacher/ job root; leave RESUME_RUN_DIR=None. Once B job.json
and checkpoints exist, set RESUME_RUN_DIR to that saved B root. Keep this notebook and
the SAME speed caps for further continuation. It can import older partial journals, but
new length-balanced journals should be resumed using this optimized notebook.

Do not confuse the teacher-only receipt with a final student submission. Only
final_report.json with status submission_verified identifies a completed student ZIP.
The original last result cell provides its exact download link. There is no score promise;
these changes target execution time while retaining the learning task.